# 01. Configuration and import

In [2]:
import os
import json
import math
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.linear_model import (Ridge, ElasticNet, LinearRegression,
                                  HuberRegressor, QuantileRegressor)
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor,
                              ExtraTreesRegressor, GradientBoostingRegressor,
                              IsolationForest)
from sklearn.neighbors import KNeighborsRegressor, LocalOutlierFactor, NearestNeighbors
from sklearn.neural_network import MLPRegressor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

SEED = 42
np.random.seed(SEED)

# 02. Configuration

In [ ]:
# ---- runtime ----
FAST_MODE = False          # True -> subsample everything for a quick smoke run

DATA_DIR = "./data"
CACHE_DIR = "./cache"
ARTIFACT_DIR = "./artifacts"
for _d in (DATA_DIR, CACHE_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

# ---- cohort admission ----
MIN_SESSIONS = 60          # minimum sessions to admit a patient
MIN_SPAN_DAYS = 120
MAX_PATIENTS = None        # None = every eligible patient (the full dataset)

# ---- feature engineering ----
SIGNALS = ("sbp", "dbp", "idwg")
HORIZONS = (1, 2, 3)       # sessions ahead (~2, 5, 7 days at 3x/week)
LAGS = (1, 2, 3, 5, 7, 14)
WINDOWS = (3, 7, 14, 30)

# ---- splits ----
VAL_FRAC = 0.20            # temporal tail per patient -> validation
TEST_FRAC = 0.20           # temporal tail per patient -> test
HOLDOUT_PATIENT_FRAC = 0.30

# ---- training / search budgets ----
MAX_TRAIN_ROWS = None      # None = train on every row
TUNE_SAMPLE_ROWS = 30_000  # rows used for hyper-parameter search only
TUNE_DRAWS = 12            # random-search draws per family
TUNE_FOLDS = 3             # forward-chaining folds
KERNEL_FIT_ROWS = 20_000   # cap for O(n^2) learners only (kNN, one-class SVM, LOF)
N_BOOT = 200               # bootstrap resamples for confidence intervals

# ---- scope: global (pooled panel) vs local (one model per patient) ----
RUN_LOCAL_SCOPE = True     # register the per-patient architectures in the bake-off
LOCAL_MIN_TRAIN_ROWS = 30  # a patient with fewer own rows falls back to the pooled fit
LOCAL_MAX_FEATURES = 10    # ~36 own training rows cannot support the full feature matrix

# ---- GOVERNANCE: clinician / ops inputs. Never searched, never tuned. ----
POPULATION_THRESHOLD_MMHG = 140.0
EMERGENCY_FLOOR_MMHG = 180.0    # never personalised, asserted everywhere
OFFSET_CAP_LOOSEN = 15.0        # asymmetric on purpose: loosening is the hazardous direction
OFFSET_CAP_TIGHTEN = 25.0
ALERT_BUDGET_PCT = 5.0          # staffing capacity, not a statistic
WARN_WINDOW = 3                 # sessions of lead time the detector must deliver
EVENT_QUANTILE = 0.95           # personal quantile defining a high-BP event

# ---- serving ----
COLD_START_MIN_READINGS = 7
STEADY_STATE_READINGS = 48

GOVERNANCE = {
    "population_threshold_mmHg": POPULATION_THRESHOLD_MMHG,
    "emergency_floor_mmHg": EMERGENCY_FLOOR_MMHG,
    "offset_cap_loosen": OFFSET_CAP_LOOSEN,
    "offset_cap_tighten": OFFSET_CAP_TIGHTEN,
    "alert_budget_pct": ALERT_BUDGET_PCT,
    "warn_window": WARN_WINDOW,
    "event_quantile": EVENT_QUANTILE,
}

if FAST_MODE:
    MAX_PATIENTS = 120
    MAX_TRAIN_ROWS = 15_000
    TUNE_SAMPLE_ROWS = 8_000
    TUNE_DRAWS = 4
    N_BOOT = 60

print("governance parameters (fixed inputs, excluded from every search):")
print(json.dumps(GOVERNANCE, indent=2))
print(f"\nFAST_MODE={FAST_MODE} | max_patients={MAX_PATIENTS} | max_train_rows={MAX_TRAIN_ROWS}")

# 03. Data Ingestion

In [ ]:
FIGSHARE_FILES = {"d1.csv": 15142151, "idp.csv": 15142154, "vip.csv": 15142157}
PUBLISHED_COUNTS = {"bp_recordings": 4_366_298, "sessions": 165_986, "patients": 1_075}

INGEST_RANGES = {"sbp": (60, 260), "dbp": (30, 160), "min_pulse_pressure": 10}


def fetch(name):
    """Download a HEMOBP file once, return the local path."""
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(
            f"https://ndownloader.figshare.com/files/{FIGSHARE_FILES[name]}", path)
    return path


def reduce_vip(chunksize=1_000_000):
    """Stream vip.csv and collapse it to one row per (patient, session)."""
    key = ["pid", "date"]
    dtypes = {"pid": "int64", "measuretime": "int16", "sbp": "int16", "dbp": "int16",
              "uf": "float32", "blood_flow": "float32", "conductivity": "float32"}
    parts, n_raw, n_dropped = [], 0, 0

    for chunk in pd.read_csv(fetch("vip.csv"), chunksize=chunksize,
                             usecols=["pid", "datatime", "measuretime", "sbp", "dbp", "uf",
                                      "blood_flow", "conductivity"],
                             dtype=dtypes):
        n_raw += len(chunk)
        chunk["date"] = pd.to_datetime(chunk.datatime, errors="coerce").dt.normalize()
        chunk = chunk.dropna(subset=["date"])

        lo_s, hi_s = INGEST_RANGES["sbp"]
        lo_d, hi_d = INGEST_RANGES["dbp"]
        keep = (chunk.sbp.between(lo_s, hi_s) & chunk.dbp.between(lo_d, hi_d)
                & ((chunk.sbp - chunk.dbp) >= INGEST_RANGES["min_pulse_pressure"]))
        n_dropped += int((~keep).sum())
        chunk = chunk[keep]
        if chunk.empty:
            continue

        chunk = chunk.sort_values(key + ["measuretime"])
        first = (chunk.drop_duplicates(key, keep="first")[key + ["measuretime", "sbp", "dbp"]]
                 .rename(columns={"measuretime": "mt_first"}))
        last = (chunk.drop_duplicates(key, keep="last")[key + ["measuretime", "sbp"]]
                .rename(columns={"measuretime": "mt_last", "sbp": "sbp_post"}))
        agg = chunk.groupby(key, as_index=False).agg(
            sbp_min=("sbp", "min"), uf_total=("uf", "max"), n_meas=("sbp", "size"),
            blood_flow=("blood_flow", "mean"), conductivity=("conductivity", "mean"))
        parts.append(first.merge(last, on=key).merge(agg, on=key))

    v = pd.concat(parts, ignore_index=True)
    del parts

    # a session can straddle a chunk boundary, so reduce once more with explicit merges
    f = (v.sort_values(key + ["mt_first"]).drop_duplicates(key, keep="first")
         [key + ["sbp", "dbp"]])
    l = (v.sort_values(key + ["mt_last"]).drop_duplicates(key, keep="last")
         [key + ["sbp_post"]])
    a = v.groupby(key, as_index=False).agg(sbp_min=("sbp_min", "min"),
                                           uf_total=("uf_total", "max"),
                                           n_meas=("n_meas", "sum"),
                                           blood_flow=("blood_flow", "mean"),
                                           conductivity=("conductivity", "mean"))
    s = f.merge(l, on=key).merge(a, on=key)
    s["sbp_drop"] = s.sbp - s.sbp_min            # intra-dialytic fall, first reading -> nadir
    INGEST_META["raw_bp_rows"] = n_raw
    INGEST_META["rows_out_of_range"] = n_dropped
    return s


def load_hemobp():
    """Return (sessions, static): one row per session, one row per patient."""
    s = reduce_vip()
    s["pid"] = s.pid.astype(str)

    d1 = pd.read_csv(fetch("d1.csv"))
    d1["date"] = pd.to_datetime(d1.keyindate, errors="coerce").dt.normalize()
    d1 = d1.dropna(subset=["date"])
    d1["pid"] = d1.pid.astype(str)
    s = s.merge(d1[["pid", "date", "weightstart", "weightend", "dryweight", "temperature",
                    "dialysisstart", "dialysisend"]]
                .drop_duplicates(["pid", "date"]), on=["pid", "date"], how="left")
    s["weight"] = s.weightstart
    s["idwg"] = s.weightstart - s.dryweight       # interdialytic weight gain

    # session length, from the two clock strings d1 already carries. Needed because the
    # ultrafiltration RATE (ml/h/kg), not the total, is the quantity the IDH literature
    # keys on - 10 ml/h/kg is the commonly cited risk threshold.
    def _minutes(col):
        t = pd.to_datetime(col, format="%H:%M", errors="coerce")
        return t.dt.hour * 60 + t.dt.minute
    _dur = (_minutes(s.dialysisend) - _minutes(s.dialysisstart)) % (24 * 60)
    s["session_hours"] = (_dur / 60.0).where(_dur.between(60, 480))

    idp = pd.read_csv(fetch("idp.csv"))
    idp["pid"] = idp.pid.astype(str)
    idp["age"] = 2016 - idp.birthday
    idp["first_dialysis_ts"] = pd.to_datetime(idp.first_dialysis, format="%Y-%m",
                                              errors="coerce")
    static = (idp[["pid", "gender", "age", "DM", "first_dialysis_ts"]]
              .rename(columns={"pid": "patient_id"}))

    return s.rename(columns={"pid": "patient_id", "date": "ts"}), static


INGEST_META = {}
_sess_cache = os.path.join(CACHE_DIR, "sessions.pkl")
_stat_cache = os.path.join(CACHE_DIR, "static.pkl")

if os.path.exists(_sess_cache) and os.path.exists(_stat_cache):
    sessions = pd.read_pickle(_sess_cache)
    static = pd.read_pickle(_stat_cache)
    print("loaded from cache")
else:
    sessions, static = load_hemobp()
    sessions.to_pickle(_sess_cache)
    static.to_pickle(_stat_cache)

print(f"sessions : {len(sessions):,} rows x {sessions.shape[1]} cols, "
      f"{sessions.patient_id.nunique():,} patients")
print(f"static   : {len(static):,} rows")
sessions.head(3)

In [5]:
recon = pd.DataFrame([
    dict(quantity="BP recordings (raw rows)", published=PUBLISHED_COUNTS["bp_recordings"],
         derived=INGEST_META.get("raw_bp_rows", np.nan)),
    dict(quantity="Sessions", published=PUBLISHED_COUNTS["sessions"], derived=len(sessions)),
    dict(quantity="Patients", published=PUBLISHED_COUNTS["patients"],
         derived=sessions.patient_id.nunique()),
])
recon["delta_pct"] = ((recon.derived - recon.published) / recon.published * 100).round(2)
display(recon)
if INGEST_META:
    print(f"rows rejected by the physiological filter: "
          f"{INGEST_META.get('rows_out_of_range', 0):,}")

,quantity,published,derived,delta_pct
0,BP recordings (raw rows),4366298,4366298,0.00
1,Sessions,165986,238731,43.83
2,Patients,1075,1072,-0.28


rows rejected by the physiological filter: 2,840


# 04. Data Validation

In [ ]:
VALIDATION_RANGES = {"sbp": (60, 260), "dbp": (30, 160), "weight": (25, 220),
                     "idwg": (-5, 12), "age": (18, 110), "temperature": (33, 41),
                     # columns recovered from d1/idp and derived in build_panel. A bad
                     # dialysisstart/dialysisend parse silently produces absurd UF rates,
                     # so these are checked rather than trusted.
                     "session_hours": (1.0, 8.0), "blood_flow": (0, 600),
                     "conductivity": (10, 18), "uf_rate": (0, 30),
                     "idwg_rel": (0, 12), "vintage_years": (0, 40)}
MAX_OUT_OF_RANGE = 0.005


def validate(sessions, static, panel=None):
    """Schema, range, structural and referential checks. Returns a report frame."""
    checks = []

    def add(name, passed, detail="", critical=False):
        checks.append(dict(check=name,
                           status="PASS" if passed else ("FAIL" if critical else "WARN"),
                           detail=detail))

    # -- schema
    for col, kind in {"patient_id": "object", "ts": "datetime",
                      "sbp": "number", "dbp": "number"}.items():
        here = col in sessions.columns
        add(f"schema: '{col}' present", here, critical=True)
        if not here:
            continue
        s = sessions[col]
        ok = (pd.api.types.is_numeric_dtype(s) if kind == "number"
              else pd.api.types.is_datetime64_any_dtype(s) if kind == "datetime" else True)
        add(f"schema: '{col}' usable as {kind}", ok, f"got {s.dtype}", critical=True)

    optional = ["weight", "idwg", "sbp_drop", "uf_total", "temperature", "dryweight"]
    missing = [c for c in optional if c not in sessions.columns]
    add("schema: optional columns present", not missing, f"missing {missing}")

    # -- ranges
    for col, (lo, hi) in VALIDATION_RANGES.items():
        src = (sessions if col in sessions.columns
               else static if col in static.columns
               else panel if (panel is not None and col in panel.columns) else None)
        if src is None:
            continue
        s = pd.to_numeric(src[col], errors="coerce").dropna()
        if s.empty:
            continue
        bad = float(((s < lo) | (s > hi)).mean())
        add(f"range: {col} in [{lo}, {hi}]", bad < MAX_OUT_OF_RANGE,
            f"{bad:.3%} outside; observed [{s.min():.1f}, {s.max():.1f}]")

    pp_bad = float(((sessions.sbp - sessions.dbp) < INGEST_RANGES["min_pulse_pressure"]).mean())
    add("range: pulse pressure >= 10 mmHg", pp_bad < MAX_OUT_OF_RANGE, f"{pp_bad:.3%} violate")

    # -- structure
    dup = int(sessions.duplicated(["patient_id", "ts"]).sum())
    add("structure: unique (patient, session)", dup == 0, f"{dup} duplicates", critical=True)

    if panel is not None:
        mono = bool(panel.groupby("series_id").ts.is_monotonic_increasing.all())
        add("structure: timestamps increasing within patient", mono, critical=True)

        contiguous = bool(
            (panel.groupby("series_id").step.transform(lambda s: (s.values == np.arange(len(s))).all()))
            .all())
        add("structure: step index contiguous from 0", contiguous, critical=True)

        g = panel.days_since_last
        add("structure: gaps positive and bounded", bool((g >= 1).all() and (g <= 30).all()),
            f"gap range [{g.min():.0f}, {g.max():.0f}] days", critical=True)

        short = int((panel.groupby("series_id").size() < MIN_SESSIONS).sum())
        add(f"structure: every patient has >= {MIN_SESSIONS} sessions", short == 0,
            f"{short} below floor", critical=True)

        # -- referential
        unmatched = int(panel.age.isna().sum())
        add("join: demographics matched", unmatched == 0,
            f"{unmatched} sessions ({unmatched / max(len(panel), 1):.2%}) unmatched")
        add("join: merge did not duplicate rows",
            len(panel) == panel.drop_duplicates(["series_id", "ts"]).shape[0], critical=True)
        const = int(panel.groupby("series_id")[["gender", "age"]].nunique().max().max()) <= 1
        add("join: demographics constant within patient", const)

    return pd.DataFrame(checks)


def enforce(report):
    fails = report[report.status == "FAIL"]
    if len(fails):
        raise AssertionError("data contract violated:\n" +
                             "\n".join(f"  - {r.check}: {r.detail}" for r in fails.itertuples()))


report = validate(sessions, static)
display(report)
enforce(report)

# 05. Panel Constrction and Splits

In [ ]:
def build_panel(sessions, static):
    """Admit eligible patients, index them by session, attach demographics."""
    g = sessions.groupby("patient_id")
    span = (g.ts.max() - g.ts.min()).dt.days
    sizes = g.size()
    eligible = [p for p in sizes[sizes >= MIN_SESSIONS].index if span[p] >= MIN_SPAN_DAYS]
    df = sessions[sessions.patient_id.isin(eligible)]
    print(f"{len(eligible)} patients meet >={MIN_SESSIONS} sessions and >={MIN_SPAN_DAYS} days")

    if MAX_PATIENTS and len(eligible) > MAX_PATIENTS:
        ranked = sorted(eligible, key=lambda p: (-int(sizes[p]), str(p)))[:MAX_PATIENTS]
        df = df[df.patient_id.isin(ranked)]
        print(f"capped to {MAX_PATIENTS} patients (set MAX_PATIENTS=None for all)")

    p = df.sort_values(["patient_id", "ts"]).copy()
    p["series_id"] = p.patient_id
    p["days_since_last"] = p.groupby("series_id").ts.diff().dt.days.fillna(2).clip(1, 30)
    p["step"] = p.groupby("series_id").cumcount()
    p = p.merge(static, on="patient_id", how="left")

    # derived clinical quantities, used by both EDA and features
    p["pulse_pressure"] = p.sbp - p.dbp
    p["map_est"] = p.dbp + (p.sbp - p.dbp) / 3.0
    p["dow"] = p.ts.dt.dayofweek
    p["is_weekend"] = (p.dow >= 5).astype(int)
    p["is_male"] = p.gender.astype(str).str.upper().str.startswith("M").astype(int)
    p["is_dm"] = p.DM.fillna(0).astype(int)
    p["age_band"] = pd.cut(p.age, [0, 50, 65, 75, 200],
                           labels=["<50", "50-64", "65-74", "75+"], right=False)

    # real quantities the earlier version discarded, all clinically load-bearing
    p["vintage_years"] = ((p.ts - p.first_dialysis_ts).dt.days / 365.25).clip(0, 40)
    p["idwg_rel"] = (100 * p.idwg / p.dryweight).clip(0, 12)      # % of dry weight
    p["uf_rate"] = ((p.uf_total * 1000) / (p.session_hours * p.dryweight)).clip(0, 30)
    p["nadir_sbp"] = p.sbp_min
    p["map_drop"] = p.map_est - (p.sbp_min + 2 * p.dbp) / 3.0
    return p.reset_index(drop=True)


def hash_bucket(key, salt="patient-split", buckets=100):
    """Deterministic bucket, stable across runs, machines and row order."""
    import hashlib
    return int(hashlib.md5(f"{salt}:{key}".encode()).hexdigest()[:8], 16) % buckets


def add_splits(df):
    out = df.copy()
    last = out.groupby("series_id")["step"].transform("max")
    test_edge = (last * (1 - TEST_FRAC)).round()
    val_edge = (last * (1 - TEST_FRAC - VAL_FRAC)).round()
    out["split"] = np.where(out.step > test_edge, "test",
                            np.where(out.step > val_edge, "val", "train"))
    cut = int(round(HOLDOUT_PATIENT_FRAC * 100))
    assign = {i: ("holdout" if hash_bucket(str(i)) < cut else "fit")
              for i in out.series_id.unique()}
    out["patient_split"] = out.series_id.map(assign)
    return out


panel = build_panel(sessions, static)
panel = add_splits(panel)

report = validate(sessions, static, panel)
display(report)
print(f"{int((report.status == 'PASS').sum())} pass | "
      f"{int((report.status == 'WARN').sum())} warn | "
      f"{int((report.status == 'FAIL').sum())} fail")
enforce(report)

display(panel.groupby("split").agg(rows=("sbp", "size"), patients=("series_id", "nunique"),
                                   first=("ts", "min"), last=("ts", "max"))
        .reindex(["train", "val", "test"]))
print("patient split:", panel.groupby("patient_split").series_id.nunique().to_dict())

In [ ]:
# =============================================================================
# SYNTHETIC CLINICAL LAYER  -  patient level
# =============================================================================
# The HEMOBP release carries blood pressures, ultrafiltration, weights, dialysate
# parameters, age, sex, diabetes and dialysis vintage. It carries NO heart rate, NO
# medication list, NO adherence record and NO symptom log. Those four are generated
# here, and nothing below ever modifies a real column.
#
# READ THIS BEFORE TRUSTING ANY SYMPTOM METRIC
# A model trained on generated labels recovers the generating process and nothing more.
# A high symptom AUC downstream is a statement about this cell, not about physiology.
# The layer exists so the multi-task architecture, the serving contract and the
# calibration machinery are built and tested now, and real labels can be dropped in by
# replacing SYNTH_* below with a join. Treat every symptom number as a pipeline test.
#
# Design: a structural causal model, not a lookup rule.
#     real session data  ->  latent patient traits  ->  conditions  ->  medications
#                                      |                     |             |
#                                      v                     v             v
#                              adherence  ---->  heart rate  ---->  symptoms
# Every arrow is stochastic. Marginal rates are calibrated to published cohorts
# (references in SYNTH_LIT), and the next cell audits the result against them.

SYNTH_SEED = 20260730

SYNTH_LIT = {
    # source                                             quantity, value used
    "IDH_session_rate":        ("KDOQI / Flythe USRDS n=39,497", 0.25),
    "symptomatic_session":     ("prospective HD cohort n=124", 0.214),
    "cramps_session":          ("prospective HD cohort n=124", 0.088),
    "dizziness_session":       ("prospective HD cohort n=124", 0.049),
    "nausea_session":          ("prospective HD cohort n=124", 0.026),
    "nausea_vomiting_session": ("dialysis complications review", 0.10),
    "ihd_prevalence":          ("HEMO study baseline", 0.393),
    "chf_prevalence":          ("cardiac disease in maintenance HD", 0.40),
    "arrhythmia_prevalence":   ("cardiac disease in maintenance HD", 0.31),
    "af_prevalence":           ("VIVALDI cross-sectional n=626", 0.265),
    "af_per_year_on_hd":       ("VIVALDI, OR per HD year", 1.08),
    "mean_medications":        ("USRDS 2017 n=176,133", 6.8),
    "acei_cough":              ("pooled 125 trials, n~200,000 (enalapril)", 0.115),
    "acei_angioedema":         ("EHR n=134,945 over 5y", 0.007),
    "acei_dizziness_syncope":  ("label/review range 2-5%", 0.035),
    "af_antithrombotic":       ("VIVALDI, AF patients on antithrombotics", 0.844),
    "polypharmacy_rate":       ("USRDS 2017", 0.738),
    "antihypertensive_nonadh": ("systematic review, pooled mean", 0.382),
    "polypharm_low_adh_OR":    ("HD cross-sectional, Brazil", 3.19),
}

CONDITIONS = ["hypertension", "ihd", "chf", "arrhythmia", "atrial_fib",
              "tachycardia", "bradycardia"]

# Classes only - no molecule names. Deliberately split ACEi from ARB: the cough and
# angioedema signal belongs to ACE inhibitors and is what separates the two clinically,
# so collapsing them would erase the mechanism the symptom layer needs.
MED_CLASSES = ["ace_inhibitor", "arb", "beta_blocker", "ccb_nondhp", "water_pill",
               "blood_thinner", "cholesterol", "heart_rhythm", "sglt2", "nsaid"]

MED_LABELS = {
    "ace_inhibitor": "ACE inhibitors", "arb": "ARBs", "beta_blocker": "Beta-blockers",
    "ccb_nondhp": "Calcium channel blockers (non-DHP)", "water_pill": "Water pills",
    "blood_thinner": "Blood thinners", "cholesterol": "Cholesterol",
    "heart_rhythm": "Heart rhythm", "sglt2": "SGLT2", "nsaid": "Pain relievers (NSAIDs)",
}


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _sig(x):
    return 1.0 / (1.0 + np.exp(-x))


def _z(v):
    v = pd.Series(v).astype(float)
    s = v.std(ddof=0)
    return ((v - v.mean()) / (s if s and np.isfinite(s) else 1.0)).fillna(0.0).values


def synth_patient_layer(panel, seed=SYNTH_SEED):
    """One row per patient: latent traits, diagnosed conditions, medication list.

    Conditions are sampled with literature prevalence as the TARGET MARGINAL, but the
    per-patient probability is tilted by that patient's own real data - a patient whose
    observed SBP runs high is more likely to carry a hypertension label, one with large
    real intradialytic drops is more likely to carry CHF. That tilt is what stops the
    labels being independent noise bolted onto the panel.
    """
    rng = np.random.default_rng(seed)

    q90 = lambda x: x.quantile(0.90)
    P = panel.groupby("series_id").agg(
        age=("age", "first"), is_male=("is_male", "first"), is_dm=("is_dm", "first"),
        vintage=("vintage_years", "median"), n_sessions=("sbp", "size"),
        sbp_med=("sbp", "median"), sbp_sd=("sbp", "std"), pp_med=("pulse_pressure", "median"),
        drop_med=("sbp_drop", "median"), drop_p90=("sbp_drop", q90),
        nadir_med=("nadir_sbp", "median"), idwg_rel=("idwg_rel", "median"),
        ufr=("uf_rate", "median"), weight=("weight", "median"),
    ).reset_index()
    P["vintage"] = P.vintage.fillna(P.vintage.median() if P.vintage.notna().any() else 5.0)
    P["ufr"] = P.ufr.fillna(P.ufr.median() if P.ufr.notna().any() else 9.0)
    n = len(P)

    # ---- observed IDH burden: a REAL per-patient quantity, the anchor for everything --
    idh = (panel.assign(idh=(panel.sbp_drop >= 20).astype(float))
                .groupby("series_id").idh.mean().reindex(P.series_id).values)
    P["idh_rate_obs"] = np.nan_to_num(idh, nan=0.25)

    # ---- latent traits -----------------------------------------------------------
    P["frailty"] = _sig(0.9 * _z(P.age) + 0.5 * P.is_dm + 0.4 * _z(P.vintage)
                        + 0.8 * _z(P.idh_rate_obs) + rng.normal(0, 0.8, n))
    P["autonomic_reserve"] = _sig(-0.7 * _z(P.age) - 0.8 * P.is_dm
                                  - 0.9 * _z(P.idh_rate_obs) + rng.normal(0, 0.9, n))
    # how readily this patient reports anything at all. The literature is emphatic that
    # this varies enormously between patients and is largely independent of BP, so it
    # gets its own axis rather than being folded into frailty.
    P["report_propensity"] = rng.normal(0, 1.0, n)

    # ---- diagnosed conditions ----------------------------------------------------
    def _draw(target, tilt):
        """Bernoulli whose population mean is `target`, tilted per patient by `tilt`."""
        t = np.asarray(tilt, float)
        t = t - t.mean()
        lo, hi = -12.0, 12.0
        for _ in range(60):                      # bisect the intercept onto the target
            mid = (lo + hi) / 2
            if _sig(mid + t).mean() > target:
                hi = mid
            else:
                lo = mid
        return (rng.random(n) < _sig((lo + hi) / 2 + t)).astype(int)

    P["hypertension"] = _draw(0.90, 1.2 * _z(P.sbp_med) + 0.4 * _z(P.pp_med))
    P["ihd"] = _draw(SYNTH_LIT["ihd_prevalence"][1],
                     0.8 * _z(P.age) + 0.6 * P.is_dm + 0.5 * _z(P.vintage))
    P["chf"] = _draw(SYNTH_LIT["chf_prevalence"][1],
                     0.7 * _z(P.drop_p90) + 0.6 * _z(P.idwg_rel) + 0.5 * _z(P.age)
                     + 0.8 * P["ihd"])
    P["arrhythmia"] = _draw(SYNTH_LIT["arrhythmia_prevalence"][1],
                            0.6 * _z(P.age) + 0.5 * P["chf"] + 0.4 * _z(P.vintage))
    # VIVALDI: AF independently associated with age, male sex, CHF and years on HD
    P["atrial_fib"] = _draw(SYNTH_LIT["af_prevalence"][1],
                            0.9 * _z(P.age) + 0.5 * P.is_male + 0.6 * P["chf"]
                            + np.log(SYNTH_LIT["af_per_year_on_hd"][1]) * P.vintage
                            + 1.4 * P["arrhythmia"])
    P["tachycardia"] = _draw(0.12, 0.8 * P["atrial_fib"] + 0.5 * P["chf"]
                             - 0.4 * _z(P.autonomic_reserve))
    P["bradycardia"] = _draw(0.08, 0.6 * _z(P.age) - 0.5 * P["tachycardia"])

    # ---- medication list ---------------------------------------------------------
    # Prescribing follows the conditions, so the list is not independent of the panel.
    med_spec = {
        "ace_inhibitor": (0.18, lambda d: 0.8 * d.chf + 0.5 * d.hypertension
                          + 0.3 * _z(d.sbp_med)),
        "arb":           (0.15, lambda d: 0.7 * d.chf + 0.5 * d.hypertension),
        "beta_blocker":  (0.45, lambda d: 1.1 * d.chf + 0.9 * d.ihd + 1.2 * d.arrhythmia),
        # non-DHP only (verapamil/diltiazem). Far less used in HD than the DHPs, and
        # rate-limiting rather than vasodilating - which is why it lowers HR below.
        "ccb_nondhp":    (0.15, lambda d: 0.6 * d.hypertension + 0.9 * d.arrhythmia
                          + 0.5 * d.atrial_fib),
        # residual renal function only; vintage erodes the indication
        "water_pill":    (0.15, lambda d: 0.6 * d.chf - 0.7 * _z(d.vintage)),
        "blood_thinner": (0.40, lambda d: 1.6 * d.atrial_fib + 1.0 * d.ihd
                          + 0.4 * _z(d.age)),
        "cholesterol":   (0.35, lambda d: 1.2 * d.ihd + 0.5 * d.is_dm),
        "heart_rhythm":  (0.08, lambda d: 2.0 * d.arrhythmia + 1.6 * d.atrial_fib),
        # SGLT2 inhibitors act on the nephron and are of little use once a patient is
        # anuric on dialysis. Kept at a low rate because they are prescribed in practice
        # (residual function, cardiac indication), not because they are standard here.
        "sglt2":         (0.03, lambda d: 0.5 * d.chf),
        # NSAIDs are generally avoided in kidney failure but still appear. They retain
        # fluid and blunt antihypertensives, which drives symptoms further down.
        "nsaid":         (0.12, lambda d: 0.3 * _z(d.age)),
    }
    for m, (rate, tilt) in med_spec.items():
        P["med_" + m] = _draw(rate, tilt(P))
    P["med_sglt2"] = P.med_sglt2 * P.is_dm            # real DM flag gates this
    # a patient on an ACE inhibitor is not also on an ARB - dual RAAS blockade is not
    # standard practice, and letting both fire would double-count the cough mechanism
    _both = (P.med_ace_inhibitor == 1) & (P.med_arb == 1)
    P.loc[_both, "med_arb"] = 0
    P["med_raas"] = ((P.med_ace_inhibitor + P.med_arb) > 0).astype(int)
    P["n_antihypertensive"] = P[["med_ace_inhibitor", "med_arb", "med_beta_blocker",
                                 "med_ccb_nondhp", "med_water_pill"]].sum(axis=1)

    P["n_medications"] = P[["med_" + m for m in MED_CLASSES]].sum(axis=1)
    # USRDS counts every prescription, not every class; scale up to that mean
    # gamma-Poisson, not plain Poisson: a fixed-rate Poisson leaves the count too
    # narrow and pushes the polypharmacy share well above the USRDS 73.8%
    _lam = rng.gamma(4.0, max(SYNTH_LIT["mean_medications"][1]
                              - P.n_medications.mean(), 0.1) / 4.0, n)
    _extra = rng.poisson(_lam)
    P["n_medications"] = P.n_medications + _extra
    P["polypharmacy"] = (P.n_medications >= 5).astype(int)

    # ---- adherence trait ---------------------------------------------------------
    base_adh = 1 - SYNTH_LIT["antihypertensive_nonadh"][1]
    tilt = (-np.log(SYNTH_LIT["polypharm_low_adh_OR"][1]) * P.polypharmacy
            + 0.3 * _z(P.age) - 0.25 * _z(P.n_medications) + rng.normal(0, 0.7, n))
    P["adherence_trait"] = _sig(_logit(base_adh) + (tilt - tilt.mean()))

    return P


PATIENTS = synth_patient_layer(panel)
print(f"synthetic patient layer: {len(PATIENTS)} patients")
display(PATIENTS[CONDITIONS].mean().round(3).rename("prevalence").to_frame().T)
display(PATIENTS[["med_" + m for m in MED_CLASSES]].mean().round(3)
        .rename(index=lambda c: MED_LABELS[c[4:]]).rename("prescribed").to_frame().T)
print(f"medications per patient: mean {PATIENTS.n_medications.mean():.1f} "
      f"(literature {SYNTH_LIT['mean_medications'][1]}), "
      f"polypharmacy {PATIENTS.polypharmacy.mean():.1%} "
      f"(literature {SYNTH_LIT['polypharmacy_rate'][1]:.1%})")
print(f"mean adherence trait: {PATIENTS.adherence_trait.mean():.1%} "
      f"(literature {1 - SYNTH_LIT['antihypertensive_nonadh'][1]:.1%})")

In [ ]:
# =============================================================================
# SYNTHETIC CLINICAL LAYER  -  session level
# =============================================================================
# Adherence, heart rate and the symptom set, one row per real session. Three rules
# govern this cell and are what keep the generated columns usable as model inputs:
#
# 1. NOTHING IS GENERATED FROM A FORECAST TARGET. Every driver is measured at or before
#    the session it describes - sbp, sbp_drop, nadir_sbp, uf_rate, idwg_rel are all REAL
#    columns of that session. Heart rate at t never reads SBP at t+h, so using hr_lag1
#    to predict tomorrow's BP is not circular.
# 2. NOTHING IS DETERMINISTIC. Each outcome carries a patient random effect and a
#    session random effect on top of its drivers. The cohort evidence is that most
#    intradialytic hypotension is asymptomatic and that roughly half of symptomatic
#    sessions meet no BP threshold at all; a clean rule reproduces neither.
# 3. MARGINALS ARE CALIBRATED, NOT GUESSED, and the audit cell then checks the JOINT
#    structure, which was never fitted.
#
# The symptom set spans FOUR mechanisms, and separating them is the whole point - a
# single "BP badness" score would collapse them and learn nothing useful:
#
#   HYPERTENSIVE  driven by HIGH pre-dialysis SBP and by missed antihypertensives.
#                 severe headache, visual changes, altered mental status, focal
#                 neurological deficit, severe epigastric pain, chest pain.
#   HYPOTENSIVE   driven by the REAL intradialytic drop, nadir and UF rate.
#                 dizziness, syncope, palpitations, fatigue.
#   VOLUME        driven by relative interdialytic weight gain, CHF, NSAID retention.
#                 leg swelling, shortness of breath.
#   DRUG EFFECT   driven almost entirely by drug class, barely by BP at all.
#                 dry cough and angioedema (face swelling, throat tightness) on ACE
#                 inhibitors. This is the cluster an ARB patient should NOT get, and it
#                 is why the medication layer splits ACEi from ARB.

SYMPTOMS = [
    "severe_headache", "visual_changes", "altered_mental_status", "chest_pain",
    "shortness_of_breath", "focal_neuro_deficit", "severe_epigastric_pain",
    "dizziness", "syncope", "palpitations", "leg_swelling",
    "fatigue", "dry_cough", "face_swelling", "throat_tightness",
]

SYMPTOM_MECHANISM = {
    "severe_headache": "hypertensive", "visual_changes": "hypertensive",
    "altered_mental_status": "hypertensive", "focal_neuro_deficit": "hypertensive",
    "severe_epigastric_pain": "hypertensive", "chest_pain": "hypertensive",
    "dizziness": "hypotensive", "syncope": "hypotensive",
    "palpitations": "hypotensive", "fatigue": "hypotensive",
    "leg_swelling": "volume", "shortness_of_breath": "volume",
    "dry_cough": "drug", "face_swelling": "drug", "throat_tightness": "drug",
}

# Red flags: an alerting system must not average these against fatigue.
SYMPTOM_RED_FLAG = ["altered_mental_status", "focal_neuro_deficit", "visual_changes",
                    "chest_pain", "throat_tightness", "syncope"]

# rate = target per-session probability. Remaining keys are log-odds weights on the
# drivers assembled in synth_session_layer.
SYMPTOM_SPEC = {
    "severe_headache":        dict(rate=0.035, hi_sbp=0.90, missed=0.60, idwg=0.25),
    "visual_changes":         dict(rate=0.012, hi_sbp=0.85, missed=0.45, dm=0.55),
    "altered_mental_status":  dict(rate=0.008, hi_sbp=0.70, missed=0.35, drop=0.45,
                                   frail=0.60),
    "chest_pain":             dict(rate=0.010, hi_sbp=0.50, missed=0.25, drop=0.40,
                                   ufr=0.20, frail=0.35),
    "focal_neuro_deficit":    dict(rate=0.003, hi_sbp=0.80, missed=0.35, frail=0.50),
    "severe_epigastric_pain": dict(rate=0.006, hi_sbp=0.45, missed=0.20),
    "dizziness":              dict(rate=0.049, drop=0.85, low=0.90, ufr=0.30,
                                   frail=0.45, adher=0.25),
    "syncope":                dict(rate=0.005, drop=1.10, low=1.40, ufr=0.35,
                                   frail=0.55, adher=0.30),
    "palpitations":           dict(rate=0.015, drop=0.30, low=0.25, ufr=0.20, frail=0.30),
    "fatigue":                dict(rate=0.120, drop=0.35, low=0.25, ufr=0.30, idwg=0.15,
                                   frail=0.70),
    "leg_swelling":           dict(rate=0.045, idwg=0.95, hi_sbp=0.25, frail=0.35),
    "shortness_of_breath":    dict(rate=0.020, idwg=0.75, hi_sbp=0.20, drop=0.15,
                                   frail=0.50, missed=0.25),
    "dry_cough":              dict(rate=0.025),
    "face_swelling":          dict(rate=0.0015),
    "throat_tightness":       dict(rate=0.0010),
}

SYMPTOM_CONDITION_FX = {
    "severe_headache": {"hypertension": 0.35},
    "visual_changes": {"hypertension": 0.30},
    "altered_mental_status": {"chf": 0.30},
    "chest_pain": {"ihd": 1.30, "chf": 0.30},
    "focal_neuro_deficit": {"atrial_fib": 0.90, "hypertension": 0.30},
    "dizziness": {"bradycardia": 0.40, "is_dm": 0.30},
    "syncope": {"bradycardia": 0.90, "arrhythmia": 0.70, "atrial_fib": 0.50},
    "palpitations": {"arrhythmia": 1.20, "atrial_fib": 1.50, "tachycardia": 0.90},
    "leg_swelling": {"chf": 1.30},
    "shortness_of_breath": {"chf": 1.20, "ihd": 0.30},
    "fatigue": {"chf": 0.60, "ihd": 0.20},
}

# Drug-class effects. The ACE-inhibitor block is the load-bearing part: cough in roughly
# 11% of ACEi patients (pooled trial data, far above the label figure) and angioedema in
# 0.1-0.7%. ARBs keep a small residual angioedema signal and essentially no cough.
SYMPTOM_MED_FX = {
    "dry_cough":        {"med_ace_inhibitor": 4.20, "med_arb": 0.25},
    "face_swelling":    {"med_ace_inhibitor": 3.40, "med_arb": 0.90},
    "throat_tightness": {"med_ace_inhibitor": 3.60, "med_arb": 0.90},
    "dizziness":        {"med_ace_inhibitor": 0.35, "med_arb": 0.25,
                         "med_water_pill": 0.30, "med_beta_blocker": 0.20},
    "syncope":          {"med_beta_blocker": 0.35, "med_ccb_nondhp": 0.30,
                         "med_heart_rhythm": 0.30, "med_water_pill": 0.25},
    "palpitations":     {"med_beta_blocker": -0.60, "med_heart_rhythm": -0.50,
                         "med_ccb_nondhp": -0.35},
    "fatigue":          {"med_beta_blocker": 0.35, "med_ccb_nondhp": 0.20},
    # NSAIDs retain sodium and water and blunt antihypertensive effect
    "leg_swelling":     {"med_nsaid": 1.10, "med_ccb_nondhp": 0.30,
                         "med_water_pill": -0.45},
    "shortness_of_breath": {"med_nsaid": 0.40, "med_water_pill": -0.30},
    "severe_headache":  {"med_nsaid": 0.30},
    "severe_epigastric_pain": {"med_nsaid": 1.10, "med_blood_thinner": 0.45},
    "focal_neuro_deficit": {"med_blood_thinner": 0.30},
}


def _calibrate(base_logit, target):
    """Solve the intercept that puts the realised mean rate on `target`."""
    lo, hi = -18.0, 10.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if _sig(base_logit + mid).mean() > target:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


def synth_session_layer(panel, patients, seed=SYNTH_SEED + 1):
    """Adherence, heart rate and symptom labels for every real session."""
    rng = np.random.default_rng(seed)
    S = panel.merge(patients, on="series_id", how="left", suffixes=("", "_pt"))
    S = S.sort_values(["series_id", "step"]).reset_index(drop=True)
    n = len(S)
    pat_idx = pd.factorize(S.series_id)[0]
    n_pat = int(pat_idx.max()) + 1

    # ---- daily adherence: AR(1), because missed doses come in streaks ---------------
    rho = 0.6
    eps = rng.normal(0, 1.0, n)
    z = np.zeros(n)
    for sid, idx in S.groupby("series_id").indices.items():
        idx = idx[np.argsort(S.step.values[idx])]
        acc = rng.normal(0, 1.0)
        for j in idx:
            acc = rho * acc + np.sqrt(1 - rho ** 2) * eps[j]
            z[j] = acc
    p_adh = _sig(_logit(S.adherence_trait.values) + 0.8 * z)
    S["took_all_meds"] = (rng.random(n) < p_adh).astype(int)
    S["missed_antihypertensive"] = ((1 - S.took_all_meds)
                                    * (S.n_antihypertensive > 0)).astype(int)

    # ---- heart rate ----------------------------------------------------------------
    # As ultrafiltration outpaces vascular refilling, cardiac output and MAP fall and
    # heart rate rises. Three classes here lower or blunt it: beta-blockers, non-DHP
    # calcium channel blockers (rate-limiting, unlike the dihydropyridines) and
    # antiarrhythmics. Drivers are same-session REAL columns only.
    zu = _z(S.uf_rate.fillna(S.uf_rate.median()))
    zi = _z(S.idwg_rel.fillna(S.idwg_rel.median()))
    zd = _z(S.sbp_drop.fillna(0))
    rate_limiting = (9.0 * S.med_beta_blocker.values
                     + 5.0 * S.med_ccb_nondhp.values
                     + 4.0 * S.med_heart_rhythm.values)
    blunting = np.clip(0.55 * (rate_limiting / 9.0), 0, 0.8)

    hr_base = (74.0
               + 9.0 * rng.normal(0, 1, n_pat)[pat_idx]      # patient random intercept
               - rate_limiting
               + 12.0 * S.tachycardia.values
               - 11.0 * S.bradycardia.values
               + 3.0 * S.chf.values
               - 0.10 * (S.age.values - 65))
    response = (1.0 - blunting) * (4.5 * zu + 2.0 * zi + 3.5 * zd)
    af_noise = rng.normal(0, 1, n) * (4.0 + 9.0 * S.atrial_fib.values)
    S["heart_rate"] = np.clip(hr_base + response + af_noise, 38, 165).round(0)
    S["hr_tachy"] = (S.heart_rate > 100).astype(int)
    S["hr_brady"] = (S.heart_rate < 60).astype(int)

    # ---- symptoms ------------------------------------------------------------------
    # A shared per-session latent makes symptoms co-occur: a bad session tends to be bad
    # in several ways at once, which independent Bernoullis would never produce.
    session_fx = rng.normal(0, 0.7, n)
    drivers = {
        "hi_sbp": _z(S.sbp),                                   # REAL pre-dialysis SBP
        "drop":   zd,                                          # REAL intradialytic fall
        "low":    (S.nadir_sbp.fillna(999).values < 90).astype(float),
        "ufr":    zu,
        "idwg":   zi,
        "missed": S.missed_antihypertensive.values.astype(float),
        "adher":  S.took_all_meds.values.astype(float) - S.took_all_meds.mean(),
        "frail":  _z(S.frailty),
        "dm":     S.is_dm.values.astype(float),
    }

    SYMPTOM_RATES, SYMPTOM_INTERCEPTS = {}, {}
    for sym, w in SYMPTOM_SPEC.items():
        base = (session_fx
                + w.get("report", 0.55) * S.report_propensity.values
                + rng.normal(0, 0.9, n_pat)[pat_idx])     # symptom-specific pt effect
        for d, arr in drivers.items():
            if w.get(d):
                base = base + w[d] * arr
        for col, beta in SYMPTOM_CONDITION_FX.get(sym, {}).items():
            base = base + beta * S[col].values
        for col, beta in SYMPTOM_MED_FX.get(sym, {}).items():
            base = base + beta * S[col].values
        icept = _calibrate(base, w["rate"])
        S["sym_" + sym] = (rng.random(n) < _sig(base + icept)).astype(int)
        SYMPTOM_RATES[sym] = float(S["sym_" + sym].mean())
        SYMPTOM_INTERCEPTS[sym] = float(icept)

    sym_cols = ["sym_" + s for s in SYMPTOMS]
    S["sym_any"] = (S[sym_cols].sum(axis=1) > 0).astype(int)
    S["sym_count"] = S[sym_cols].sum(axis=1)
    S["sym_red_flag"] = (S[["sym_" + s for s in SYMPTOM_RED_FLAG]]
                         .sum(axis=1) > 0).astype(int)
    for mech in sorted(set(SYMPTOM_MECHANISM.values())):
        cols = ["sym_" + s for s, m in SYMPTOM_MECHANISM.items() if m == mech]
        S["sym_mech_" + mech] = (S[cols].sum(axis=1) > 0).astype(int)
    S["idh_kdoqi"] = ((S.sbp_drop >= 20) & S.sym_any.astype(bool)).astype(int)
    return S, SYMPTOM_RATES


SESSIONS_SYNTH, SYMPTOM_RATES = synth_session_layer(panel, PATIENTS)

SYMPTOM_GROUPS = (["any", "red_flag"]
                  + ["mech_" + m for m in sorted(set(SYMPTOM_MECHANISM.values()))])
SYNTH_COLS = (["heart_rate", "hr_tachy", "hr_brady", "took_all_meds",
               "missed_antihypertensive", "sym_count", "idh_kdoqi"]
              + ["sym_" + s for s in SYMPTOMS]
              + ["sym_" + g for g in SYMPTOM_GROUPS]
              + CONDITIONS + ["med_" + m for m in MED_CLASSES]
              + ["med_raas", "n_antihypertensive", "n_medications", "polypharmacy",
                 "adherence_trait", "frailty"])
panel = panel.merge(SESSIONS_SYNTH[["series_id", "step"] + SYNTH_COLS],
                    on=["series_id", "step"], how="left")

print(f"panel now {panel.shape[1]} columns ({len(SYNTH_COLS)} synthetic)")
display(pd.DataFrame({
    "symptom": SYMPTOMS,
    "mechanism": [SYMPTOM_MECHANISM[s] for s in SYMPTOMS],
    "red_flag": [s in SYMPTOM_RED_FLAG for s in SYMPTOMS],
    "target_rate": [SYMPTOM_SPEC[s]["rate"] for s in SYMPTOMS],
    "realised_rate": [round(SYMPTOM_RATES[s], 4) for s in SYMPTOMS],
    "n_positive": [int(panel["sym_" + s].sum()) for s in SYMPTOMS],
}).style.hide(axis="index").set_caption("synthetic symptom set by mechanism"))
print(f"sessions with >=1 symptom : {panel.sym_any.mean():.1%}")
print(f"sessions with a red flag  : {panel.sym_red_flag.mean():.2%}")
print(f"heart rate                : {panel.heart_rate.mean():.0f} "
      f"+/- {panel.heart_rate.std():.0f} bpm")
print(f"sessions with all meds    : {panel.took_all_meds.mean():.1%}")

In [ ]:
# =============================================================================
# SYNTHESIS AUDIT  -  does the generated layer look like the published cohorts?
# =============================================================================
# Marginal rates were calibrated, so matching those proves nothing. The rows that carry
# weight are the JOINT ones - the asymptomatic-IDH fraction, the mechanism separation,
# the drug-class odds ratios. None were targeted, and all are where a naive
# "BP badness score" rule would fail visibly.

audit = []


def chk(name, got, want, tol, source):
    audit.append(dict(check=name, generated=round(float(got), 4),
                      literature=round(float(want), 4) if want is not None else np.nan,
                      within_tol=bool(want is None or abs(got - want) <= tol),
                      source=source))


def odds_ratio(y, x):
    t = pd.crosstab(pd.Series(x).astype(int), pd.Series(y).astype(int)) + 0.5
    if t.shape != (2, 2):
        return np.nan
    return float((t.loc[1, 1] * t.loc[0, 0]) / (t.loc[1, 0] * t.loc[0, 1]))


for s in SYMPTOMS:
    chk(f"rate: {s}", panel["sym_" + s].mean(), SYMPTOM_SPEC[s]["rate"],
        max(0.25 * SYMPTOM_SPEC[s]["rate"], 0.002), "calibrated (not evidence)")

# --- joint structure: none of these were calibrated --------------------------------
idh_bp = panel.sbp_drop >= 20
# NB: this audits the REAL sbp_drop column, not anything generated here.
chk("IDH by BP criterion (REAL data)", idh_bp.mean(), SYNTH_LIT["IDH_session_rate"][1],
    0.15, SYNTH_LIT["IDH_session_rate"][0])
chk("IDH sessions that are ASYMPTOMATIC", 1 - panel.loc[idh_bp, "sym_any"].mean(),
    0.70, 0.25, "IDH only occasionally symptomatic (cohort n=458)")
chk("symptomatic sessions NOT meeting IDH", 1 - idh_bp[panel.sym_any == 1].mean(),
    0.46, 0.30, "11 of 24 symptom events unrelated to BP")

# --- MECHANISM SEPARATION: the property a single BP score cannot have ---------------
hi_sbp = panel.sbp >= panel.sbp.quantile(0.85)
chk("OR hypertensive cluster | top-15% SBP", odds_ratio(panel.sym_mech_hypertensive,
                                                        hi_sbp), None, 0,
    "should be >1: this cluster tracks HIGH pressure")
chk("OR hypertensive cluster | SBP drop>=20", odds_ratio(panel.sym_mech_hypertensive,
                                                         idh_bp), None, 0,
    "should be near 1: NOT a hypotensive cluster")
chk("OR hypotensive cluster | SBP drop>=20", odds_ratio(panel.sym_mech_hypotensive,
                                                        idh_bp), None, 0,
    "should be >1: tracks the intradialytic fall")
chk("OR hypotensive cluster | top-15% SBP", odds_ratio(panel.sym_mech_hypotensive,
                                                       hi_sbp), None, 0,
    "should be near 1 or below")
chk("OR volume cluster | top-quartile IDWG",
    odds_ratio(panel.sym_mech_volume, panel.idwg_rel >= panel.idwg_rel.quantile(0.75)),
    None, 0, "should be >1: tracks fluid gain")

# --- DRUG-CLASS SPECIFICITY: ACEi vs ARB is the sharpest test ----------------------
ace, arb = panel.med_ace_inhibitor == 1, panel.med_arb == 1
chk("dry cough rate on ACE inhibitor", panel.loc[ace, "sym_dry_cough"].mean(),
    SYNTH_LIT["acei_cough"][1], 0.06, SYNTH_LIT["acei_cough"][0])
chk("dry cough rate on ARB", panel.loc[arb, "sym_dry_cough"].mean(), 0.01, 0.02,
    "ARBs do not cause the bradykinin cough")
chk("OR dry cough | ACE inhibitor", odds_ratio(panel.sym_dry_cough, ace), None, 0,
    "should be large; this is the class signature")
chk("angioedema rate on ACE inhibitor",
    panel.loc[ace, ["sym_face_swelling", "sym_throat_tightness"]].max(axis=1).mean(),
    SYNTH_LIT["acei_angioedema"][1], 0.012, SYNTH_LIT["acei_angioedema"][0])
chk("OR epigastric pain | NSAID", odds_ratio(panel.sym_severe_epigastric_pain,
                                             panel.med_nsaid == 1), None, 0,
    "NSAID gastric toxicity")
chk("OR leg swelling | NSAID", odds_ratio(panel.sym_leg_swelling,
                                          panel.med_nsaid == 1), None, 0,
    "NSAID sodium and water retention")
chk("OR palpitations | atrial fibrillation dx",
    odds_ratio(panel.sym_palpitations, panel.atrial_fib == 1), None, 0,
    "condition-symptom coupling")
chk("OR chest pain | IHD dx", odds_ratio(panel.sym_chest_pain, panel.ihd == 1),
    None, 0, "condition-symptom coupling")
chk("OR any hypertensive symptom | missed antihypertensive",
    odds_ratio(panel.sym_mech_hypertensive, panel.missed_antihypertensive == 1),
    None, 0, "the adherence-to-symptom link the model has to learn")

# --- heart rate --------------------------------------------------------------------
for cls, want in (("beta_blocker", -9.0), ("ccb_nondhp", -5.0), ("heart_rhythm", -4.0)):
    on = panel["med_" + cls] == 1
    chk(f"HR: on {cls} minus off (bpm)",
        panel.loc[on, "heart_rate"].mean() - panel.loc[~on, "heart_rate"].mean(),
        want, 7.0, "negative chronotropy (marginal, so class overlap confounds it)")
chk("HR vs UF rate (within-patient r)",
    panel.assign(h=panel.heart_rate - panel.groupby("series_id").heart_rate.transform("mean"),
                 u=panel.uf_rate - panel.groupby("series_id").uf_rate.transform("mean"))
         .dropna(subset=["h", "u"]).pipe(lambda d: d.h.corr(d.u)),
    None, 0, "HR rises as UF outpaces refilling")
chk("HR mean (bpm)", panel.heart_rate.mean(), 74.0, 10.0, "resting HR in HD cohorts")

# --- adherence and prescribing ------------------------------------------------------
chk("sessions fully adherent", panel.took_all_meds.mean(),
    1 - SYNTH_LIT["antihypertensive_nonadh"][1], 0.12,
    SYNTH_LIT["antihypertensive_nonadh"][0])
_lag = panel.groupby("series_id").took_all_meds.shift(1)
chk("adherence autocorrelation (lag 1)", panel.took_all_meds.corr(_lag), None, 0,
    "missed doses come in streaks, not iid")
chk("patients on both ACEi and ARB", float((PATIENTS.med_ace_inhibitor
                                            & PATIENTS.med_arb).mean()), 0.0, 0.0,
    "dual RAAS blockade is not standard practice")
chk("blood thinner use in AF patients",
    PATIENTS.loc[PATIENTS.atrial_fib == 1, "med_blood_thinner"].mean(),
    SYNTH_LIT["af_antithrombotic"][1], 0.25, SYNTH_LIT["af_antithrombotic"][0])
chk("medications per patient", PATIENTS.n_medications.mean(),
    SYNTH_LIT["mean_medications"][1], 0.8, SYNTH_LIT["mean_medications"][0])
chk("polypharmacy rate", PATIENTS.polypharmacy.mean(),
    SYNTH_LIT["polypharmacy_rate"][1], 0.10, SYNTH_LIT["polypharmacy_rate"][0])

# --- heterogeneity ------------------------------------------------------------------
_pt = panel.groupby("series_id").sym_any.mean()
chk("between-patient SD of symptom rate", _pt.std(), None, 0,
    "reporting propensity varies widely")
chk("10th pct of per-patient symptom rate", float(_pt.quantile(0.10)), None, 0,
    "low-reporting tail must exist")
chk("90th pct of per-patient symptom rate", float(_pt.quantile(0.90)), None, 0,
    "high-reporting tail must exist")

AUDIT = pd.DataFrame(audit)
display(AUDIT.style.apply(
    lambda c: ["background-color: #e8f5e9" if v else
               ("background-color: #ffebee" if not pd.isna(AUDIT.literature.iloc[i]) else "")
               for i, v in enumerate(c)], subset=["within_tol"])
    .hide(axis="index").set_caption("synthesis audit vs published hemodialysis cohorts"))

_checked = AUDIT[AUDIT.literature.notna()]
print(f"{int(_checked.within_tol.sum())}/{len(_checked)} quantitative checks within tolerance")
_failed = _checked[~_checked.within_tol]
if len(_failed):
    print("outside tolerance:", ", ".join(_failed.check))

fig, ax = plt.subplots(1, 4, figsize=(21, 4.4))
r = pd.Series({s: panel["sym_" + s].mean() for s in SYMPTOMS}).sort_values()
cmap = {"hypertensive": "indianred", "hypotensive": "steelblue",
        "volume": "seagreen", "drug": "darkorange"}
ax[0].barh(r.index, r.values, color=[cmap[SYMPTOM_MECHANISM[s]] for s in r.index])
ax[0].set(title="Per-session rate by mechanism", xlabel="fraction of sessions")
ax[0].set_xscale("log")

sbp_bin = pd.qcut(panel.sbp, 8, duplicates="drop")
drop_bin = pd.cut(panel.sbp_drop, [-100, 0, 10, 20, 30, 40, 500],
                  labels=["<0", "0-10", "10-20", "20-30", "30-40", "40+"])
g1 = panel.groupby(sbp_bin, observed=True).sym_mech_hypertensive.mean()
ax[1].plot(range(len(g1)), g1.values, marker="o", color="indianred", label="hypertensive")
g2 = panel.groupby(sbp_bin, observed=True).sym_mech_hypotensive.mean()
ax[1].plot(range(len(g2)), g2.values, marker="s", color="steelblue", label="hypotensive")
ax[1].set(title="Mechanism clusters vs pre-dialysis SBP octile",
          xlabel="SBP octile (low to high)", ylabel="rate")
ax[1].legend(fontsize=8)

g3 = panel.groupby(drop_bin, observed=True)[["sym_mech_hypotensive",
                                             "sym_mech_hypertensive"]].mean()
g3.plot(marker="o", ax=ax[2], color=["steelblue", "indianred"])
ax[2].set(title="Mechanism clusters vs REAL intradialytic drop",
          xlabel="SBP drop (mmHg)", ylabel="rate")
ax[2].legend(fontsize=8)

cough = pd.DataFrame({
    "ACE inhibitor": [panel.loc[ace, "sym_dry_cough"].mean(),
                      panel.loc[ace, ["sym_face_swelling",
                                      "sym_throat_tightness"]].max(axis=1).mean()],
    "ARB": [panel.loc[arb, "sym_dry_cough"].mean(),
            panel.loc[arb, ["sym_face_swelling",
                            "sym_throat_tightness"]].max(axis=1).mean()],
    "neither": [panel.loc[~ace & ~arb, "sym_dry_cough"].mean(),
                panel.loc[~ace & ~arb, ["sym_face_swelling",
                                        "sym_throat_tightness"]].max(axis=1).mean()],
}, index=["dry cough", "angioedema"])
cough.T.plot(kind="bar", ax=ax[3], color=["darkorange", "firebrick"], rot=0)
ax[3].set(title="Drug-class specificity", ylabel="rate")
ax[3].set_yscale("log")
plt.tight_layout()
plt.show()

# 06. Exploratory Data Analysis - Distribtion and completeness

In [ ]:
num_cols = ["sbp", "dbp", "idwg", "weight", "pulse_pressure", "map_est", "age", "sbp_drop",
            "heart_rate", "uf_rate", "idwg_rel", "vintage_years", "sym_count"]
num_cols = [c for c in num_cols if c in panel.columns]
desc = panel[num_cols].describe().T
desc["skew"] = panel[num_cols].skew()
desc["kurtosis"] = panel[num_cols].kurtosis()
desc["missing_pct"] = panel[num_cols].isna().mean() * 100
display(desc.round(2))

fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for a, c in zip(ax.ravel(), ["sbp", "dbp", "idwg", "pulse_pressure", "map_est", "weight"]):
    d = panel[c].dropna()
    sns.histplot(d, bins=60, ax=a, color="steelblue")
    a.axvline(d.mean(), color="crimson", ls="--", lw=1.2, label=f"mean {d.mean():.1f}")
    a.axvline(d.median(), color="darkorange", ls=":", lw=1.2, label=f"median {d.median():.1f}")
    a.set_title(c)
    a.legend(fontsize=7)
plt.tight_layout()
plt.show()

# normality is not assumed anywhere downstream, but the degree of departure is worth knowing
for c in ("sbp", "dbp", "idwg"):
    d = panel[c].dropna().sample(min(5000, panel[c].notna().sum()), random_state=SEED)
    w, p = stats.shapiro(d)
    print(f"{c:5s} Shapiro-Wilk W={w:.4f} p={p:.3g} -> "
          f"{'not normal' if p < 0.05 else 'consistent with normal'}")

## Between patient Vs. Within patient variance

In [ ]:
pm = panel.groupby("series_id").sbp.agg(["mean", "std", "count"])
between = float(pm["mean"].std())
within = float(pm["std"].mean())
icc = between ** 2 / (between ** 2 + within ** 2)

print(f"between-patient SD of mean SBP : {between:.1f} mmHg")
print(f"mean within-patient SD of SBP  : {within:.1f} mmHg")
print(f"between-patient share of variance (ICC-like): {icc:.1%}")

# one-way ANOVA across patients: is patient identity a real factor?
groups = [g.sbp.dropna().values for _, g in panel.groupby("series_id") if g.sbp.notna().sum() > 10]
F_stat, p_anova = stats.f_oneway(*groups[:300])
print(f"one-way ANOVA over patients: F={F_stat:.1f}, p={p_anova:.3g}")
print("-> personalisation is justified" if icc > 0.25 else
      "-> personalisation is marginal on this cohort")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
sns.histplot(pm["mean"], bins=40, ax=ax[0], color="steelblue")
ax[0].axvline(POPULATION_THRESHOLD_MMHG, c="crimson", ls="--", label="population threshold")
ax[0].set(title="Q2 - distribution of patient mean SBP", xlabel="mean SBP (mmHg)")
ax[0].legend(fontsize=8)
sns.scatterplot(x=pm["mean"], y=pm["std"], s=18, alpha=.5, ax=ax[1])
ax[1].set(title="Q2 - patient mean vs patient SD", xlabel="mean SBP", ylabel="within-patient SD")
sample_ids = pm.sort_values("count", ascending=False).index[:40]
sns.boxplot(data=panel[panel.series_id.isin(sample_ids)], x="series_id", y="sbp",
            ax=ax[2], fliersize=0)
ax[2].set(title="Q2 - 40 longest patients", xlabel="")
ax[2].set_xticklabels([])
plt.tight_layout()
plt.show()

# How mch memory does the series have?

In [ ]:
def mean_acf(df, col="sbp", max_lag=20, min_len=40):
    """Average within-patient autocorrelation up to max_lag."""
    acc = np.zeros(max_lag + 1)
    n = 0
    for _, g in df.groupby("series_id", sort=False):
        y = g.sort_values("step")[col].values.astype(float)
        y = y[np.isfinite(y)]
        if len(y) < min_len:
            continue
        y = y - y.mean()
        denom = np.dot(y, y)
        if denom <= 0:
            continue
        acc += [np.dot(y[:len(y) - k], y[k:]) / denom for k in range(max_lag + 1)]
        n += 1
    return acc / max(n, 1), n


acf_sbp, n_used = mean_acf(panel, "sbp")
acf_idwg, _ = mean_acf(panel, "idwg")
conf = 1.96 / np.sqrt(panel.groupby("series_id").size().median())

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
ax[0].bar(range(len(acf_sbp)), acf_sbp, color="steelblue")
ax[0].axhline(conf, ls="--", c="crimson", lw=1)
ax[0].axhline(-conf, ls="--", c="crimson", lw=1)
ax[0].set(title=f"Q3 - mean within-patient ACF of SBP (n={n_used})", xlabel="lag (sessions)")
ax[1].bar(range(len(acf_idwg)), acf_idwg, color="seagreen")
ax[1].axhline(conf, ls="--", c="crimson", lw=1)
ax[1].set(title="Q3 - mean within-patient ACF of IDWG", xlabel="lag (sessions)")

# how much does one session tell you about the next? persistence R^2 per patient
pers = []
for _, g in panel.groupby("series_id", sort=False):
    y = g.sort_values("step").sbp.values.astype(float)
    if len(y) > 30 and np.isfinite(y).all():
        pers.append(np.corrcoef(y[:-1], y[1:])[0, 1] ** 2)
sns.histplot(pers, bins=30, ax=ax[2], color="darkorange")
ax[2].set(title="Q3 - per-patient lag-1 R² (persistence ceiling)", xlabel="R²")
plt.tight_layout()
plt.show()

print(f"significant lags (|acf| > {conf:.3f}): "
      f"{[k for k in range(1, len(acf_sbp)) if abs(acf_sbp[k]) > conf][:12]}")
print(f"median per-patient lag-1 R²: {np.median(pers):.3f} "
      f"-> persistence is a strong baseline; the learned model must beat it")

## The cadence artefact

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
sns.boxplot(data=panel, x="dow", y="sbp", ax=ax[0], fliersize=0)
ax[0].set(title="Q4 - SBP by day of week (0=Mon)", xlabel="day of week")
gap = panel.days_since_last.clip(1, 4)
sns.boxplot(x=gap, y=panel.sbp, ax=ax[1], fliersize=0)
ax[1].set(title="Q4 - SBP by gap since last session", xlabel="days since last")
sns.histplot(panel.days_since_last, bins=30, ax=ax[2], color="slateblue")
ax[2].set(title="Q4 - session gap distribution", xlabel="days")
plt.tight_layout()
plt.show()

dow_groups = [g.sbp.dropna().values for _, g in panel.groupby("dow") if g.sbp.notna().sum() > 30]
F_dow, p_dow = stats.f_oneway(*dow_groups)
gap_groups = [g.sbp.dropna().values for _, g in panel.groupby(gap) if g.sbp.notna().sum() > 30]
F_gap, p_gap = stats.f_oneway(*gap_groups)
print(f"day-of-week ANOVA : F={F_dow:.1f}, p={p_dow:.3g}")
print(f"gap-length ANOVA  : F={F_gap:.1f}, p={p_gap:.3g}")
print(f"SBP after a 1-day gap vs a 3-day gap: "
      f"{panel[gap == 1].sbp.mean():.1f} vs {panel[gap == 3].sbp.mean():.1f} mmHg")

## Missingness

In [ ]:
miss_cols = ["sbp", "dbp", "idwg", "weight", "sbp_drop", "uf_total", "temperature", "age", "DM",
             "session_hours", "uf_rate", "vintage_years", "blood_flow", "heart_rate"]
miss_cols = [c for c in miss_cols if c in panel.columns]
miss = panel[miss_cols].isna().mean().sort_values()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
sns.barplot(x=miss.values * 100, y=miss.index, ax=ax[0], color="steelblue")
ax[0].set(title="Q5 - missing rate by column", xlabel="% missing")
per_pt_missing = panel.groupby("series_id")[["idwg", "weight", "temperature"]].apply(
    lambda g: g.isna().mean().mean())
sns.histplot(per_pt_missing * 100, bins=30, ax=ax[1], color="indianred")
ax[1].set(title="Q5 - per-patient mean missingness", xlabel="% missing")
plt.tight_layout()
plt.show()

print("columns dropped from modelling if missing > 60%:",
      [c for c in miss_cols if miss[c] > 0.6] or "none")

## How common is a high BP event

In [ ]:
event_rows = []
for sid, g in panel.groupby("series_id", sort=False):
    y = g.sort_values("step").sbp.values.astype(float)
    if len(y) < 30:
        continue
    thr = np.nanquantile(y, EVENT_QUANTILE)
    fut = np.column_stack([np.roll(y, -k) for k in range(1, WARN_WINDOW + 1)]).astype(float)
    fut[len(y) - WARN_WINDOW:, :] = np.nan
    event_rows.append(dict(series_id=sid, threshold=thr,
                           base_rate=float(np.nanmean(np.nanmax(fut, axis=1) >= thr))))
EV = pd.DataFrame(event_rows)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
sns.histplot(EV.base_rate * 100, bins=30, ax=ax[0], color="seagreen")
ax[0].axvline(EV.base_rate.mean() * 100, c="crimson", ls="--",
              label=f"mean {EV.base_rate.mean():.1%}")
ax[0].set(title=f"Q6 - per-patient event base rate (q{EVENT_QUANTILE}, {WARN_WINDOW} sessions)",
          xlabel="% of sessions followed by an event")
ax[0].legend(fontsize=8)
sns.histplot(EV.threshold, bins=40, ax=ax[1], color="steelblue")
ax[1].axvline(POPULATION_THRESHOLD_MMHG, c="crimson", ls="--", label="population 140")
ax[1].axvline(EMERGENCY_FLOOR_MMHG, c="black", ls="-", label="emergency floor 180")
ax[1].set(title="Q6 - personal event thresholds", xlabel="patient q95 SBP (mmHg)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"mean event base rate: {EV.base_rate.mean():.2%} "
      f"(range {EV.base_rate.min():.2%} - {EV.base_rate.max():.2%})")
print(f"patients whose personal q95 sits ABOVE the population threshold: "
      f"{(EV.threshold > POPULATION_THRESHOLD_MMHG).mean():.1%}")
print(f"patients whose personal q95 sits above the emergency floor: "
      f"{(EV.threshold > EMERGENCY_FLOOR_MMHG).mean():.1%} "
      f"-> the floor is never personalised regardless")

In [ ]:
# ---- covariance and correlation structure of the raw signals ----
# Pooled correlation over stacked patients mostly re-measures the between-patient
# spread seen in the ICC cell, exactly as a pooled ACF would. The honest view
# subtracts each patient's own mean first. Any pair collinear WITHIN patient is one
# piece of information, and every feature built on both members is one feature twice.
CORR_COLS = [c for c in ["sbp", "dbp", "idwg", "weight", "pulse_pressure", "map_est",
                         "heart_rate", "uf_rate", "idwg_rel", "vintage_years", "sym_count",
                       "uf_total", "sbp_drop", "temperature", "age", "days_since_last"]
           if c in panel.columns and panel[c].notna().mean() > .2]

P = panel[["series_id"] + CORR_COLS]

# Pooled: mixes between- and within-patient variation. Given Q2's ICC this view is
# mostly a statement about which patients are big/old/sick, not about dynamics.
pooled_r = P[CORR_COLS].corr()
pooled_cov = P[CORR_COLS].cov()

# Within-patient: subtract each patient's own mean first. This is the covariance
# structure a per-patient forecaster actually gets to exploit.
W = P.copy()
W[CORR_COLS] = W[CORR_COLS] - W.groupby("series_id")[CORR_COLS].transform("mean")
within_r = W[CORR_COLS].corr()


def grad(df, caption, fmt="{:.2f}", cmap="RdBu_r", lo=-1, hi=1):
    return (df.style.background_gradient(cmap=cmap, vmin=lo, vmax=hi)
              .format(fmt, na_rep="-").set_caption(caption))


def pair_frame(m, name="r"):
    """Unique off-diagonal pairs of a symmetric matrix, strongest first."""
    a = m.where(np.triu(np.ones(m.shape, bool), 1)).stack()
    return (a.rename(name).reset_index()
             .rename(columns={"level_0": "a", "level_1": "b"})
             .reindex(a.abs().sort_values(ascending=False).reset_index(drop=True).index))


display(grad(pooled_r, "pooled Pearson r (between + within patient)"))
display(grad(within_r, "WITHIN-patient Pearson r (what the model can actually use)"))
display(grad(pooled_cov, "pooled covariance (mixed units: read the signs, not the magnitudes)",
             fmt="{:.1f}", cmap="viridis", lo=None, hi=None))

# how much of each pooled correlation is a between-patient artefact
gap = pair_frame(pooled_r, "pooled").merge(pair_frame(within_r, "within"), on=["a", "b"])
gap["between_artefact"] = (gap.pooled - gap.within).abs()
display(gap.sort_values("between_artefact", ascending=False).head(10)
           .style.format({"pooled": "{:+.2f}", "within": "{:+.2f}",
                          "between_artefact": "{:.2f}"})
           .set_caption("pairs whose pooled correlation is mostly between-patient spread")
           .hide(axis="index"))

# cross-lag: does a signal at t-l still say anything about SBP at t, within a patient?
rows = []
for c in [x for x in ("sbp", "dbp", "idwg", "weight", "sbp_drop", "uf_total") if x in CORR_COLS]:
    for l in LAGS:
        rows.append(dict(signal=c, lag=l, r=W.sbp.corr(W.groupby("series_id")[c].shift(l))))
XLAG = pd.DataFrame(rows).pivot(index="signal", columns="lag", values="r")
display(grad(XLAG, "within-patient corr(signal at t-lag, SBP at t)"))

fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
sns.heatmap(pooled_r, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax[0], cbar=False, annot_kws={"size": 7})
ax[0].set_title("pooled r")
sns.heatmap(within_r, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax[1], cbar=False, annot_kws={"size": 7})
ax[1].set_title("within-patient r")
sns.heatmap(XLAG, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[2],
            cbar=False, annot_kws={"size": 7})
ax[2].set_title("cross-lag r vs SBP")
plt.tight_layout()
plt.show()

REDUNDANT_RAW = gap[gap.within.abs() > .9][["a", "b", "within"]]
print(f"raw pairs collinear within patient (|r| > 0.9): "
      f"{[f'{r.a}~{r.b} ({r.within:+.2f})' for r in REDUNDANT_RAW.itertuples()] or 'none'}")
print("-> any engineered feature built on both members of such a pair is one feature, not two.")

# 07. Featre Engineering

In [ ]:
def rolling_slope(v, w):
    """Vectorised rolling OLS slope over a window of w points.

    x = 0..w-1 is fixed, so slope = [mean(x*y) - mean(x)*mean(y)] / var(x).
    A .rolling().apply(np.polyfit) equivalent is ~200x slower and dominates runtime.
    """
    y = v.astype(float)
    x_mean, x_var = (w - 1) / 2.0, (w * w - 1) / 12.0
    yv = y.values
    dot = np.full(len(yv), np.nan)
    if len(yv) >= w:
        dot[w - 1:] = np.convolve(np.nan_to_num(yv), np.arange(w, dtype=float)[::-1], mode="valid")
    return (pd.Series(dot, index=y.index) / w - x_mean * y.rolling(w, min_periods=w).mean()) / x_var


def build_features_one(g):
    """All causal features + forecast targets for one patient."""
    g = g.sort_values("step").reset_index(drop=True).copy()

    for s in SIGNALS:
        v = g[s]
        for l in LAGS:
            g[f"{s}_lag{l}"] = v.shift(l)
        for w in WINDOWS:
            r = v.shift(1).rolling(w, min_periods=max(2, w // 3))
            g[f"{s}_mean{w}"] = r.mean()
            g[f"{s}_std{w}"] = r.std()
            g[f"{s}_min{w}"] = r.min()
            g[f"{s}_max{w}"] = r.max()
            g[f"{s}_range{w}"] = g[f"{s}_max{w}"] - g[f"{s}_min{w}"]
            g[f"{s}_slope{w}"] = rolling_slope(v.shift(1), w)
        for a in (0.1, 0.3):
            g[f"{s}_ewm{a}"] = v.shift(1).ewm(alpha=a, adjust=False).mean()
        exp = v.shift(1).expanding(min_periods=5)
        g[f"{s}_base_mean"] = exp.mean()
        g[f"{s}_base_std"] = exp.std()
        g[f"{s}_z"] = (v.shift(1) - g[f"{s}_base_mean"]) / g[f"{s}_base_std"].replace(0, np.nan)
        g[f"{s}_d1"] = v.shift(1) - v.shift(2)

    if g.weight.notna().sum() > 3:
        g["weight_lag1"] = g.weight.shift(1)
        g["weight_slope24"] = rolling_slope(g.weight.shift(1), 24)
        g["weight_delta_base"] = (g.weight.shift(1)
                                  - g.weight.shift(1).expanding(min_periods=5).median())
    else:
        for c in ("weight_lag1", "weight_slope24", "weight_delta_base"):
            g[c] = np.nan

    g["gap_mean24"] = g.days_since_last.shift(1).rolling(24, min_periods=4).mean()
    g["sbp_drop_lag1"] = g.sbp_drop.shift(1)
    g["uf_lag1"] = g.uf_total.shift(1)

    # ---- correlation-driven: contrasts, ratios and residuals, not duplicate levels ----
    # The correlation cell above shows sbp/dbp/map_est and the rolling means of one
    # signal are near-collinear. A difference of two collinear levels is not collinear
    # with either, so these carry what the level features cannot. All read .shift(1)+.
    pp = (g.sbp - g.dbp).shift(1)
    g["pp_lag1"] = pp
    g["pp_mean7"] = pp.rolling(7, min_periods=3).mean()
    _pe = pp.expanding(min_periods=5)
    g["pp_z"] = (pp - _pe.mean()) / _pe.std().replace(0, np.nan)

    for s in SIGNALS:
        g[f"{s}_mom_3_14"] = g[f"{s}_mean3"] - g[f"{s}_mean14"]        # short vs long trend
        g[f"{s}_excess_base"] = g[f"{s}_mean7"] - g[f"{s}_base_mean"]  # vs personal norm
        g[f"{s}_vol_ratio"] = g[f"{s}_std3"] / g[f"{s}_std14"].replace(0, np.nan)
        g[f"{s}_ewm_resid"] = g[s].shift(1) - g[f"{s}_ewm0.3"]         # level removed

    # the pooled idwg~weight correlation is a body-size artefact; normalise it away
    _w1 = g.weight.shift(1).replace(0, np.nan)
    g["idwg_per_kg"] = g.idwg.shift(1) / _w1
    g["uf_per_kg"] = g.uf_total.shift(1) / _w1
    g["sbp_dbp_ratio"] = g.sbp.shift(1) / g.dbp.shift(1).replace(0, np.nan)
    g["idwg_x_sbp_slope7"] = g["idwg_lag1"] * g["sbp_slope7"]

    # ---- synthetic clinical layer: causal history features + symptom targets -------
    # Guarded on presence so inference_features() still works for a cold-start patient
    # whose history has not yet accumulated these columns.
    for c in ("heart_rate", "took_all_meds", "missed_antihypertensive",
              "uf_rate", "idwg_rel", "sym_any", "sym_count"):
        if c in g:
            g[f"{c}_lag1"] = g[c].shift(1)
            g[f"{c}_mean7"] = g[c].shift(1).rolling(7, min_periods=2).mean()
            g[f"{c}_mean30"] = g[c].shift(1).rolling(30, min_periods=5).mean()

    # Same-day adherence is KNOWN when the forecast is made: the dose is taken in the
    # morning, the session follows, and the prediction is for the NEXT session. This is
    # the single deliberate exception to the "features read <= t-1" rule, and it is a
    # real one - a missed antihypertensive is a primary driver of the hypertensive
    # symptom cluster, and discarding today's value throws that away for no benefit.
    for c in ("took_all_meds", "missed_antihypertensive"):
        if c in g:
            g["adh_" + c + "_today"] = g[c]

    if "heart_rate" in g:
        _hb = g.heart_rate.shift(1).expanding(min_periods=5)
        g["hr_z"] = (g.heart_rate.shift(1) - _hb.mean()) / _hb.std().replace(0, np.nan)
        # shock index (HR/SBP) rises when the circulation is compensating for volume
        # loss, and moves before either component does on its own
        g["shock_index_lag1"] = g.heart_rate.shift(1) / g.sbp.shift(1).replace(0, np.nan)
        g["hr_d1"] = g.heart_rate.shift(1) - g.heart_rate.shift(2)

    for _s in SYMPTOMS + SYMPTOM_GROUPS:
        c = f"sym_{_s}"
        if c in g:
            g[f"{c}_lag1"] = g[c].shift(1)
            g[f"{c}_rate30"] = g[c].shift(1).rolling(30, min_periods=5).mean()

    for h in HORIZONS:
        for _s in SYMPTOMS + SYMPTOM_GROUPS:
            c = f"sym_{_s}"
            if c in g:
                g[f"y_{c}_h{h}"] = g[c].shift(-h)

    for h in HORIZONS:
        for s in SIGNALS:
            g[f"y_{s}_h{h}"] = g[s].shift(-h)
    return g


DROP_FROM_FEATURES = {
    "ts", "series_id", "patient_id", "gender", "step", "age_band", "split", "patient_split",
    "sbp", "dbp", "idwg", "weight", "sbp_post", "sbp_min", "sbp_drop", "uf_total",
    "temperature", "weightstart", "weightend", "dryweight", "n_meas", "pulse_pressure",
    "map_est", "DM", "keyindate", "datatime", "dow",
    # real session measurements that are contemporaneous with the row: usable only via
    # their lagged forms above, never raw
    "session_hours", "blood_flow", "conductivity", "first_dialysis_ts",
    "nadir_sbp", "map_drop", "uf_rate", "idwg_rel",
    # synthetic session measurements, same rule
    "heart_rate", "hr_tachy", "hr_brady", "took_all_meds", "missed_antihypertensive",
    "sym_count", "idh_kdoqi",
    # LATENT generator variables. These are not observable in any real deployment, so
    # letting a model see them would inflate every symptom metric with pure leakage.
    "adherence_trait", "frailty",
} | {f"sym_{s}" for s in SYMPTOMS} | {f"sym_{g}" for g in SYMPTOM_GROUPS}


def build_features(panel):
    out = pd.concat([build_features_one(g) for _, g in panel.groupby("series_id", sort=False)],
                    ignore_index=True)
    for c in out.select_dtypes("float64").columns:
        out[c] = out[c].astype("float32")   # halves memory; mmHg/kg lose no usable precision
    names = [c for c in out.columns
             if c not in DROP_FROM_FEATURES and not c.startswith("y_")
             and pd.api.types.is_numeric_dtype(out[c])]
    return out, names


F, FEATURES = build_features(panel)
print(f"{len(FEATURES)} features x {len(F):,} rows across {F.series_id.nunique()} patients")
print(f"memory: {F.memory_usage(deep=True).sum() / 1e6:.0f} MB")

## Featre Dictionary

In [ ]:
def feature_group(name):
    if name.startswith(("sbp_", "dbp_", "idwg_")):
        base = name.split("_", 1)[1]
        if base.startswith("lag"):
            return "lag"
        if base.startswith(("mean", "std", "min", "max", "range")):
            return "rolling moment"
        if base.startswith("slope"):
            return "rolling slope"
        if base.startswith("ewm_resid"):
            return "ewma residual"
        if base.startswith("ewm"):
            return "exponential mean"
        if base.startswith("base"):
            return "expanding baseline"
        if base.startswith("mom_"):
            return "momentum contrast"
        if base.startswith("excess_base"):
            return "baseline excess"
        if base.startswith("vol_ratio"):
            return "volatility ratio"
        if base.startswith("per_kg"):
            return "size-normalised"
        if base.startswith("x_"):
            return "interaction"
        if base == "z":
            return "personal z-score"
        if base == "d1":
            return "first difference"
    if name.startswith("pp_") or name == "sbp_dbp_ratio":
        return "pulse-pressure dynamics"
    if name in ("uf_per_kg", "uf_lag1"):
        return "size-normalised"
    if name.startswith("weight"):
        return "weight dynamics"
    if name in ("gap_mean24", "days_since_last", "is_weekend"):
        return "session cadence"
    if name in ("age", "is_male", "is_dm", "vintage_years"):
        return "static demographic"
    # ---- clinical layer. Without these branches every one of the ~87 clinical features
    # lands in "other" and the importance-by-group plot goes blind to exactly the inputs
    # that were added for the symptom head.
    if name.startswith(("heart_rate", "hr_", "shock_index")):
        return "heart rate"
    if name.startswith("sym_"):
        return "symptom history"
    if name.startswith("med_") or name in ("n_medications", "polypharmacy",
                                           "n_antihypertensive"):
        return "medication class"
    if name.startswith(("took_all_meds", "missed_antihypertensive", "adh_")):
        return "adherence"
    if name in CONDITIONS:
        return "diagnosed condition"
    if name.startswith(("uf_rate", "idwg_rel", "uf_lag", "uf_per_kg")):
        return "ultrafiltration"
    return "other"


FEATURE_DICT = pd.DataFrame({"feature": FEATURES})
FEATURE_DICT["group"] = FEATURE_DICT.feature.map(feature_group)
FEATURE_DICT["signal"] = FEATURE_DICT.feature.str.split("_").str[0]
FEATURE_DICT["missing_pct"] = (F[FEATURES].isna().mean() * 100).values.round(1)

display(FEATURE_DICT.groupby("group").agg(n=("feature", "size"),
                                          mean_missing_pct=("missing_pct", "mean")).round(1))
display(FEATURE_DICT.head(12))

## Leakage Adit

In [ ]:
def leakage_audit(F, features):
    rows = []

    def add(name, ok, detail=""):
        rows.append(dict(probe=name, status="PASS" if ok else "FAIL", detail=detail))

    # 1. no feature is a perfect copy of the contemporaneous reading
    corr = F[features].corrwith(F.sbp).abs().sort_values(ascending=False)
    add("no feature reproduces the current reading", bool(corr.max() < 0.999),
        f"max |corr| with sbp = {corr.max():.4f} ({corr.index[0]})")

    # 2. shifting a patient's own future away must not change their feature row
    sid = F.series_id.value_counts().index[0]
    g = panel[panel.series_id == sid].copy()
    mid = len(g) // 2
    g2 = g.copy()
    g2.loc[g2.step > mid, "sbp"] = g2.loc[g2.step > mid, "sbp"] + 50   # corrupt the future
    a = build_features_one(g)[features].iloc[:mid]
    b = build_features_one(g2)[features].iloc[:mid]
    same = np.allclose(a.fillna(-999).values, b.fillna(-999).values, atol=1e-4)
    add("corrupting the future leaves past features unchanged", same,
        f"patient {sid}, first {mid} rows")

    # 3. targets must genuinely sit in the future
    chk = F[["series_id", "step", "sbp", f"y_sbp_h{HORIZONS[0]}"]].dropna()
    merged = chk.merge(F[["series_id", "step", "sbp"]],
                       left_on=["series_id", chk.step + HORIZONS[0]],
                       right_on=["series_id", "step"], suffixes=("", "_future"))
    aligned = np.allclose(merged[f"y_sbp_h{HORIZONS[0]}"].values,
                          merged.sbp_future.values, atol=1e-3)
    add("target y_h equals the reading h steps later", aligned, f"{len(merged):,} rows checked")

    # 4. a model trained on shuffled targets must not beat persistence
    d = F[F[f"y_sbp_h{HORIZONS[0]}"].notna()]
    d = d.sample(min(4000, len(d)), random_state=SEED)
    y_shuf = d[f"y_sbp_h{HORIZONS[0]}"].sample(frac=1, random_state=SEED).values
    m = make_pipeline_ridge().fit(d[features], y_shuf)
    mae_shuf = mean_absolute_error(d[f"y_sbp_h{HORIZONS[0]}"], m.predict(d[features]))
    mae_pers = mean_absolute_error(d[f"y_sbp_h{HORIZONS[0]}"], d.sbp_lag1.fillna(d.sbp.mean()))
    add("shuffled-target model does not beat persistence", bool(mae_shuf > mae_pers),
        f"shuffled MAE {mae_shuf:.2f} vs persistence {mae_pers:.2f}")

    # 5. symptom targets must sit in the future too. The four probes above all test the
    #    BP head; without this one a shift bug in the symptom targets would pass silently.
    sym_t = [c for c in F.columns if c.startswith("y_sym_")]
    if sym_t:
        base = sym_t[0].replace(f"_h{HORIZONS[0]}", "").replace("y_", "")
        chk = F[["series_id", "step", base, sym_t[0]]].dropna()
        mg = chk.merge(F[["series_id", "step", base]],
                       left_on=["series_id", chk.step + HORIZONS[0]],
                       right_on=["series_id", "step"], suffixes=("", "_future"))
        ok = np.allclose(mg[sym_t[0]].values, mg[base + "_future"].values)
        add("symptom target equals the label h steps later", bool(ok),
            f"{base}, {len(mg):,} rows checked")

    # 6. no symptom FEATURE may reproduce its own contemporaneous label - the history
    #    features are shifted, so a perfect correlation means the shift was lost.
    if sym_t:
        raw_labels = ({f"sym_{s}" for s in SYMPTOMS} | {f"sym_{g}" for g in SYMPTOM_GROUPS}
                      | {"sym_count"})
        worst, worst_c = 0.0, "-"
        for c in [c for c in features if c.startswith("sym_")]:
            lab = c
            for suf in ("_lag1", "_rate30", "_mean7", "_mean30"):
                if lab.endswith(suf):
                    lab = lab[: -len(suf)]
                    break
            if lab not in raw_labels or lab not in F.columns:
                continue
            r = abs(float(F[c].corr(F[lab])))
            if np.isfinite(r) and r > worst:
                worst, worst_c = r, c
        add("no symptom feature reproduces its own label", bool(worst < 0.95),
            f"max |corr| = {worst:.3f} ({worst_c})")

    # 7. the same-day adherence exception must be exactly that - today, never tomorrow
    for c in [c for c in features if c.endswith("_today")]:
        raw = c.replace("adh_", "").replace("_today", "")
        if raw in F.columns:
            same = bool(np.allclose(F[c].fillna(-9).values, F[raw].fillna(-9).values))
            fwd = F.groupby("series_id")[raw].shift(-1)
            ahead = float(pd.Series(F[c]).corr(fwd))
            add(f"{c} equals today, not tomorrow", same and abs(ahead) < 0.95,
                f"matches same-day={same}, |corr| with next session={ahead:.3f}")

    return pd.DataFrame(rows)


def make_pipeline_ridge(alpha=10.0):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", Ridge(alpha=alpha))])


AUDIT = leakage_audit(F, FEATURES)
display(AUDIT)
assert (AUDIT.status == "PASS").all(), "leakage audit failed - stop and fix before modelling"
print("all leakage probes pass")

In [ ]:
# ---- correlation-based feature selection ----
# Two screens, both fitted on TRAIN rows only.
#   relevance : max(|Pearson|, |Spearman|) vs each SBP target. Spearman is there so a
#               monotone-but-nonlinear feature the trees can use survives a linear test.
#   redundancy: average-linkage clustering on 1 - |r|, cut at 1 - REDUNDANCY_R. One
#               survivor per cluster: most relevant, tie-broken on completeness.
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

REDUNDANCY_R  = 0.95   # |r| at or above which two features are treated as one feature
MIN_TARGET_R  = 0.02   # max(|pearson|, |spearman|) below which a feature is noise
SELECT_ROWS   = 40_000

# Read by literal name at serving time (detector scores / BPModel) or forced in by
# the cadence finding in EDA. Never dropped, whatever the correlation screen says.
MUST_KEEP = {"sbp_lag1", "sbp_z", "sbp_ewm0.1", "sbp_slope7", "sbp_mean7",
             "sbp_base_mean", "sbp_base_std", "days_since_last", "is_weekend"}

FEATURES_ALL = list(FEATURES)          # pre-selection list, kept for the audit trail

# TRAIN ROWS ONLY. Screening on val/test is selection leakage and would silently
# invalidate every confidence interval computed downstream.
_tr = F[F.split == "train"]
if len(_tr) > SELECT_ROWS:
    _tr = _tr.sample(SELECT_ROWS, random_state=SEED)
Xtr = _tr[FEATURES_ALL].astype("float64")

# 1. degenerate columns -------------------------------------------------------
nun = Xtr.nunique(dropna=True)
dead = set(nun[nun < 2].index) | set(Xtr.columns[Xtr.notna().mean() < .05])

pool = [f for f in FEATURES_ALL if f not in dead]

# 2. relevance: linear AND monotone, so a tree-only feature is not thrown away -
# Relevance is scored against EVERY head, not just the BP one. Scoring on y_sbp alone
# would prune the symptom history and adherence features - they are weak predictors of
# tomorrow's mmHg and strong predictors of tomorrow's symptoms - and the symptom head
# further down would then never see them.
SELECTION_TARGETS = [f"y_sbp_h{h}" for h in HORIZONS]
SELECTION_TARGETS += [c for c in (f"y_sym_any_h{HORIZONS[0]}",
                                  f"y_sym_cramps_h{HORIZONS[0]}",
                                  f"y_sym_dizziness_h{HORIZONS[0]}") if c in F.columns]

rel = pd.DataFrame(index=pool)
for t in SELECTION_TARGETS:
    m = _tr[t].notna()
    if m.sum() < 200:
        continue
    rel[f"p_{t}"] = Xtr.loc[m, pool].corrwith(_tr.loc[m, t]).abs()
    rel[f"s_{t}"] = Xtr.loc[m, pool].corrwith(_tr.loc[m, t], method="spearman").abs()
rel["relevance"] = rel.max(axis=1).fillna(0.0)
weak = set(rel.index[rel.relevance < MIN_TARGET_R]) - MUST_KEEP

# 3. redundancy: cluster on 1 - |r| and keep one representative per cluster ----
Rv = Xtr[pool].corr().abs().fillna(0.0).to_numpy(copy=True)
np.fill_diagonal(Rv, 1.0)
link = hierarchy.linkage(squareform(np.clip(1.0 - Rv, 0, None), checks=False),
                         method="average")
CLUSTERS = pd.DataFrame({
    "feature": pool,
    "cluster": hierarchy.fcluster(link, t=1.0 - REDUNDANCY_R, criterion="distance"),
})
CLUSTERS["relevance"] = CLUSTERS.feature.map(rel.relevance)
CLUSTERS["missing_pct"] = CLUSTERS.feature.map(F[pool].isna().mean() * 100)
CLUSTERS["group"] = CLUSTERS.feature.map(feature_group)

reps = []
for _, blk in CLUSTERS.groupby("cluster"):
    forced = [f for f in blk.feature if f in MUST_KEEP]
    reps.extend(forced or [blk.sort_values(["relevance", "missing_pct"],
                                           ascending=[False, True]).feature.iloc[0]])
rep_set = set(reps)

FEATURES_SELECTED = [f for f in FEATURES_ALL
                     if f in MUST_KEEP or (f in rep_set and f not in weak)]

# 3b. enforce the invariant. Clustering bounds distance WITHIN a cluster, not between
# the representatives of two different clusters, so a surviving pair can still sit
# above the threshold. One greedy pass in relevance order closes that gap.
_Rsel = Xtr[FEATURES_SELECTED].corr().abs().fillna(0.0)
_order = sorted(FEATURES_SELECTED,
                key=lambda f: (f not in MUST_KEEP, -float(rel.relevance.get(f, 0.0))))
kept = []
for f in _order:
    if f in MUST_KEEP or all(_Rsel.loc[f, k] < REDUNDANCY_R for k in kept):
        kept.append(f)
FEATURES_SELECTED = [f for f in FEATURES_ALL if f in set(kept)]

# 4. report -------------------------------------------------------------------
SELECTION = CLUSTERS.assign(status=np.where(
    CLUSTERS.feature.isin(FEATURES_SELECTED), "keep",
    np.where(CLUSTERS.feature.isin(weak), "drop: weak", "drop: redundant")))

summary = (SELECTION.pivot_table(index="group", columns="status", values="feature",
                                 aggfunc="size", fill_value=0)
                    .assign(kept_pct=lambda d: 100 * d.get("keep", 0) / d.sum(axis=1)))
display(summary.round(1).style.background_gradient(cmap="Greens", subset=["kept_pct"])
        .set_caption("feature selection - outcome by group"))

biggest = SELECTION.cluster.value_counts().head(6).index
display(SELECTION[SELECTION.cluster.isin(biggest)]
        .sort_values(["cluster", "relevance"], ascending=[True, False])
        .style.format({"relevance": "{:.3f}", "missing_pct": "{:.1f}"})
        .background_gradient(cmap="Blues", subset=["relevance"])
        .set_caption("the six largest redundancy clusters (one survivor each)")
        .hide(axis="index"))

before = Rv.copy()
after = Xtr[FEATURES_SELECTED].corr().abs().fillna(0.0).to_numpy(copy=True)
np.fill_diagonal(before, 0.0)
np.fill_diagonal(after, 0.0)
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
sns.heatmap(before, cmap="magma", vmin=0, vmax=1, ax=ax[0], cbar=False,
            xticklabels=False, yticklabels=False)
ax[0].set_title(f"|r| before - {len(pool)} features (max off-diag {before.max():.3f})")
sns.heatmap(after, cmap="magma", vmin=0, vmax=1, ax=ax[1], cbar=False,
            xticklabels=False, yticklabels=False)
ax[1].set_title(f"|r| after - {len(FEATURES_SELECTED)} features (max off-diag {after.max():.3f})")
top = rel.loc[FEATURES_SELECTED].relevance.sort_values(ascending=False).head(18)
sns.barplot(x=top.values, y=top.index, ax=ax[2], color="steelblue")
ax[2].set(title="Kept features by |corr| with y_sbp (train only)", xlabel="max |r|")
plt.tight_layout()
plt.show()

FEATURES = FEATURES_SELECTED          # everything downstream picks this up unchanged
assert len(FEATURES) >= 20, "selection too aggressive - raise REDUNDANCY_R"
assert MUST_KEEP <= set(FEATURES), "a serving-critical feature was dropped"
print(f"{len(FEATURES_ALL)} -> {len(FEATURES)} features "
      f"({len(weak)} weak, {len(set(pool)) - len(rep_set)} redundant, {len(dead)} degenerate)")
_pairs = Xtr[FEATURES].corr().abs().fillna(0.0)
_free_mask = ~pd.Series(FEATURES).isin(MUST_KEEP).to_numpy()
_forced_only = np.outer(~_free_mask, ~_free_mask)      # forced-vs-forced pairs
_chk = _pairs.to_numpy(copy=True)
np.fill_diagonal(_chk, 0.0)
_chk[_forced_only] = 0.0
print(f"max |r| over every pair with at least one freely-selected member: {_chk.max():.4f} "
      f"(invariant: < {REDUNDANCY_R})")
assert _chk.max() < REDUNDANCY_R, "redundancy invariant violated"
_ff = _pairs.to_numpy(copy=True)
np.fill_diagonal(_ff, 0.0)
_ff[~_forced_only] = 0.0
print(f"max |r| among the {len(MUST_KEEP)} forced serving features: {_ff.max():.3f} "
      f"- these are allowed to stay collinear on purpose")

# 08. Metrics, baselines and the selection rle

In [ ]:
def bootstrap_ci(fn, y, p, n_boot=None, seed=SEED):
    """Point estimate plus a percentile bootstrap CI."""
    n_boot = n_boot or N_BOOT
    rng = np.random.default_rng(seed)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if len(y) < 10:
        return np.nan, (np.nan, np.nan)
    idx = rng.integers(0, len(y), size=(n_boot, len(y)))
    vals = np.array([fn(y[i], p[i]) for i in idx])
    return float(fn(y, p)), (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


def paired_delta(err_a, err_b, n_boot=None, seed=SEED):
    """Mean of (|err_a| - |err_b|) with a paired bootstrap CI. Negative favours a.

    Two models scored on the same rows fail on the same hard patients. Comparing their
    independent CIs discards that correlation and is far less powerful than bootstrapping
    the per-row differences directly.
    """
    n_boot = n_boot or N_BOOT
    m = np.isfinite(err_a) & np.isfinite(err_b)
    d = np.abs(err_a[m]) - np.abs(err_b[m])
    if len(d) < 30:
        return np.nan, (np.nan, np.nan), np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(d.mean()), (float(lo), float(hi)), float((boot >= 0).mean())


def evaluate(y, p, last=None, **meta):
    """Point-forecast metrics with a CI, bias and direction accuracy."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if len(y) < 10:
        return {}
    mae, (lo, hi) = bootstrap_ci(mean_absolute_error, y, p)
    out = dict(meta, n=len(y), MAE=round(mae, 3), MAE_lo=round(lo, 3), MAE_hi=round(hi, 3),
               RMSE=round(float(np.sqrt(mean_squared_error(y, p))), 3),
               R2=round(float(r2_score(y, p)), 3),
               bias=round(float(np.mean(p - y)), 2),
               target_sd=round(float(np.std(y)), 2))
    if last is not None:
        b = np.asarray(last, float)[m]
        ok = np.isfinite(b)
        dt, dp = y[ok] - b[ok], p[ok] - b[ok]
        nz = np.abs(dt) > 1e-9
        out["DirAcc"] = (round(float(np.mean(np.sign(dt[nz]) == np.sign(dp[nz]))), 3)
                         if nz.sum() and np.abs(dp).max() > 1e-9 else np.nan)
    return out


def forecast_baselines(df, signal, h):
    """Five non-learned references, all computable at serving time."""
    last = df[f"{signal}_lag1"].values
    return {
        "baseline:persistence": last,
        "baseline:seasonal_naive_7": df.get(f"{signal}_lag7", pd.Series(np.nan, index=df.index)).values,
        "baseline:drift": last + h * (last - df[f"{signal}_lag2"].values),
        "baseline:personal_mean": df[f"{signal}_base_mean"].values,
        "baseline:ewma": df[f"{signal}_ewm0.3"].values,
    }


print("selection rule registered:")
print("  a learned model is selected iff (best baseline MAE - learned MAE) > paired CI width")
print("  and, inside a statistical tie, the CHEAPEST candidate wins")

# 09. Model 01 - The Forecaster

In [ ]:
def make_model(kind, **kw):
    """Factory so training, tuning, evaluation and serving construct models identically."""
    if kind == "ridge":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", Ridge(alpha=kw.get("alpha", 10.0)))])
    if kind == "elasticnet":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", ElasticNet(alpha=kw.get("alpha", .05),
                                              l1_ratio=kw.get("l1_ratio", .5),
                                              random_state=SEED))])
    if kind == "huber":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", HuberRegressor(epsilon=kw.get("epsilon", 1.35), max_iter=300))])
    if kind == "hgb":
        return HistGradientBoostingRegressor(
            loss="absolute_error", random_state=SEED,
            max_iter=kw.get("max_iter", 250), learning_rate=kw.get("learning_rate", .07),
            max_leaf_nodes=kw.get("max_leaf_nodes", 31),
            min_samples_leaf=kw.get("min_samples_leaf", 20),
            l2_regularization=kw.get("l2_regularization", 0.0),
            max_features=kw.get("max_features", 1.0))
    if kind == "hgb_mse":
        return HistGradientBoostingRegressor(
            loss="squared_error", random_state=SEED,
            max_iter=kw.get("max_iter", 250), learning_rate=kw.get("learning_rate", .07))
    if kind == "random_forest":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("model", RandomForestRegressor(
                             n_estimators=kw.get("n_estimators", 200),
                             min_samples_leaf=kw.get("min_samples_leaf", 20),
                             max_features=kw.get("max_features", 1.0),
                             n_jobs=-1, random_state=SEED))])
    if kind == "extra_trees":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("model", ExtraTreesRegressor(
                             n_estimators=kw.get("n_estimators", 200),
                             min_samples_leaf=kw.get("min_samples_leaf", 20),
                             n_jobs=-1, random_state=SEED))])
    if kind == "knn":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", KNeighborsRegressor(n_neighbors=kw.get("n_neighbors", 25),
                                                       weights="distance", n_jobs=-1))])
    if kind == "mlp":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", MLPRegressor(
                             hidden_layer_sizes=kw.get("hidden_layer_sizes", (64, 64)),
                             alpha=kw.get("alpha", 1e-4), max_iter=200,
                             early_stopping=True, random_state=SEED))])
    raise ValueError(f"unknown model kind: {kind}")


# kinds that must be fitted on a bounded sample because they scale badly with n
KERNEL_KINDS = {"knn"}

FORECAST_MODELS = {}


def register(name, family, cost="low", note="", scope="global"):
    """Add one architecture to the bake-off. fn(train, eval, signal, h) -> predictions.

    scope="local" declares that the model fits per patient. The bake-off reads this to
    decide which training frame the model gets: a per-patient model may legitimately use
    the target patient's own earlier sessions even when that patient is in the holdout
    cohort, because that is their own past, not another patient's future.
    """
    def deco(fn):
        FORECAST_MODELS[name] = dict(fn=fn, family=family, cost=cost, note=note, scope=scope)
        return fn
    return deco


def target_col(signal, h):
    return f"y_{signal}_h{h}"


def cap_rows(df, cap):
    if cap and len(df) > cap:
        return df.sample(cap, random_state=SEED)
    return df


def fit_global(kind, tr, ev, signal, h, delta=False, params=None):
    """Shared plumbing for every global tabular model, with an optional Δ-target."""
    y, lag = target_col(signal, h), f"{signal}_lag1"
    est = make_model(kind, **(params or {})) if isinstance(kind, str) else clone(kind)
    tr = cap_rows(tr, KERNEL_FIT_ROWS if kind in KERNEL_KINDS else MAX_TRAIN_ROWS)
    if delta:
        base = tr[lag].values
        ok = np.isfinite(base) & np.isfinite(tr[y].values)
        est.fit(tr.loc[ok, FEATURES], tr.loc[ok, y].values - base[ok])
        return est.predict(ev[FEATURES]) + ev[lag].values
    est.fit(tr[FEATURES], tr[y].values)
    return est.predict(ev[FEATURES])


# ── A. global tabular models on the engineered feature matrix ────────────────
@register("ridge", "linear", "low", "L2 on the full engineered feature matrix")
def _a1(tr, ev, s, h):
    return fit_global("ridge", tr, ev, s, h, params=TUNED.get("ridge"))


@register("elasticnet", "linear", "low", "L1/L2 mix; sparsifies the feature set")
def _a2(tr, ev, s, h):
    return fit_global("elasticnet", tr, ev, s, h, params=TUNED.get("elasticnet"))


@register("huber", "linear", "low", "robust loss; tests whether tails drive the fit")
def _a3(tr, ev, s, h):
    return fit_global("huber", tr, ev, s, h)


@register("hgb_mae", "trees", "medium", "gradient boosting, absolute-error loss")
def _a4(tr, ev, s, h):
    return fit_global("hgb", tr, ev, s, h, params=TUNED.get("hgb"))


@register("hgb_mse", "trees", "medium", "same model, squared loss - a pure loss ablation")
def _a5(tr, ev, s, h):
    return fit_global("hgb_mse", tr, ev, s, h)


@register("random_forest", "trees", "high", "bagged trees; variance-reduction contrast to boosting")
def _a6(tr, ev, s, h):
    return fit_global("random_forest", tr, ev, s, h, params=TUNED.get("random_forest"))


@register("extra_trees", "trees", "high", "extremely randomised trees")
def _a7(tr, ev, s, h):
    return fit_global("extra_trees", tr, ev, s, h)


@register("knn", "kernel", "high", "instance-based; tests local structure in feature space")
def _a8(tr, ev, s, h):
    return fit_global("knn", tr, ev, s, h)


@register("mlp", "neural", "medium", "2x64 dense net on the engineered features")
def _a9(tr, ev, s, h):
    return fit_global("mlp", tr, ev, s, h)


# ── B. target reparameterisation ─────────────────────────────────────────────
# Training on the level means the model spends most of its capacity re-learning the
# patient's own mean, which lag1 already encodes. Training on y - lag1 hands it that for
# free and asks only for the change.
@register("ridge_delta", "reparam", "low", "same features, target = y - lag1")
def _b1(tr, ev, s, h):
    return fit_global("ridge", tr, ev, s, h, delta=True, params=TUNED.get("ridge"))


@register("hgb_delta", "reparam", "medium", "boosting on the residual-from-persistence target")
def _b2(tr, ev, s, h):
    return fit_global("hgb", tr, ev, s, h, delta=True, params=TUNED.get("hgb"))


# ── C. window-only representation (DLinear-style) ────────────────────────────
def window_cols(signal):
    return [f"{signal}_lag{l}" for l in LAGS if f"{signal}_lag{l}" in F.columns]


@register("window_linear", "representation", "low", "linear map from the raw lookback window")
def _c1(tr, ev, s, h):
    cols, y = window_cols(s), target_col(s, h)
    est = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
                    ("m", Ridge(alpha=1.0))])
    trn = cap_rows(tr.dropna(subset=[y]), MAX_TRAIN_ROWS)
    est.fit(trn[cols], trn[y].values)
    return est.predict(ev[cols])


@register("window_delta", "representation", "low", "window-only, Δ-target")
def _c2(tr, ev, s, h):
    cols, y, lag = window_cols(s), target_col(s, h), f"{s}_lag1"
    est = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
                    ("m", Ridge(alpha=1.0))])
    trn = cap_rows(tr.dropna(subset=[y, lag]), MAX_TRAIN_ROWS)
    est.fit(trn[cols], trn[y].values - trn[lag].values)
    return est.predict(ev[cols]) + ev[lag].values


# ── D. scope: per-patient vs shared ──────────────────────────────────────────
@register("local_ar", "scope", "medium",
          "one autoregression per patient, on their own history", scope="local")
def _d1(tr, ev, s, h):
    cols, y = window_cols(s), target_col(s, h)
    out = np.full(len(ev), np.nan)
    gmean = float(tr[y].mean())
    for sid, idx in ev.groupby("series_id").indices.items():
        trn = tr[tr.series_id == sid].dropna(subset=[y] + cols)
        sub = ev.iloc[idx]
        if len(trn) < 40:                      # too little history -> fall back to persistence
            out[idx] = sub[f"{s}_lag1"].fillna(gmean).values
            continue
        X = np.column_stack([np.ones(len(trn)), trn[cols].values])
        beta, *_ = np.linalg.lstsq(X, trn[y].values, rcond=None)
        Xe = np.column_stack([np.ones(len(sub)), sub[cols].fillna(trn[cols].median()).values])
        out[idx] = Xe @ beta
    return out


@register("global_plus_intercept", "scope", "low", "global model + each patient's mean residual")
def _d2(tr, ev, s, h):
    y = target_col(s, h)
    trn = cap_rows(tr, MAX_TRAIN_ROWS)
    est = make_model("ridge", **(TUNED.get("ridge") or {})).fit(trn[FEATURES], trn[y].values)
    resid = pd.Series(trn[y].values - est.predict(trn[FEATURES]),
                      index=trn.series_id.values).groupby(level=0).mean()
    return est.predict(ev[FEATURES]) + ev.series_id.map(resid).fillna(0.0).values


# ── E. classical smoothers, one per series ───────────────────────────────────
def holt_damped(y, h, alpha, beta, phi):
    """h-step-ahead forecast made at each t using only observations <= t."""
    out = np.full(len(y), np.nan)
    level, trend = (y[0] if np.isfinite(y[0]) else np.nanmean(y)), 0.0
    damp = sum(phi ** i for i in range(1, h + 1))
    for t in range(len(y)):
        if np.isfinite(y[t]):
            prev = level
            level = alpha * y[t] + (1 - alpha) * (level + phi * trend)
            trend = beta * (level - prev) + (1 - beta) * phi * trend
        out[t] = level + damp * trend
    return out


def theta_method(y, h, alpha=0.3):
    """Theta method: exponentially-smoothed level plus half the fitted linear drift."""
    out = np.full(len(y), np.nan)
    level, k, csum = (y[0] if np.isfinite(y[0]) else np.nanmean(y)), 0, 0.0
    y0 = level
    for t in range(len(y)):
        if np.isfinite(y[t]):
            level = alpha * y[t] + (1 - alpha) * level
            k += 1
            csum += y[t] - y0
        drift = (csum / max(k, 1)) / max(k, 1) if k > 1 else 0.0
        out[t] = level + 0.5 * h * drift
    return out


def series_forecast(df, s, h, fn, **kw):
    """Apply a per-series recursion, aligned to the frame it is given."""
    out = pd.Series(np.nan, index=df.index)
    for _, g in df.groupby("series_id", sort=False):
        g = g.sort_values("step")
        out.loc[g.index] = fn(g[s].values.astype(float), h, **kw)
    return out.values


@register("holt_damped", "classical", "low", "damped-trend smoother, α/β/φ fitted on train")
def _e1(tr, ev, s, h):
    y = target_col(s, h)
    trn = tr.dropna(subset=[y])
    trn = cap_rows(trn, 20_000)
    best, params = np.inf, (0.3, 0.05, 0.9)
    for a in (0.1, 0.3, 0.5):
        for b in (0.0, 0.05, 0.2):
            for p in (0.8, 0.98):
                pred = series_forecast(trn, s, h, holt_damped, alpha=a, beta=b, phi=p)
                m = np.isfinite(pred) & np.isfinite(trn[y].values)
                if m.sum() < 50:
                    continue
                e = mean_absolute_error(trn[y].values[m], pred[m])
                if e < best:
                    best, params = e, (a, b, p)
    HOLT_PARAMS[(s, h)] = params
    a, b, p = params
    return series_forecast(ev, s, h, holt_damped, alpha=a, beta=b, phi=p)


@register("theta", "classical", "low", "SES plus half the drift; a standard benchmark")
def _e2(tr, ev, s, h):
    return series_forecast(ev, s, h, theta_method, alpha=0.3)


# ── F. per-patient (local) scope ─────────────────────────────────────────────
# One model per patient, fitted only on that patient's own earlier sessions. The
# constraint that makes this hard is p >> n: at MIN_SESSIONS=60 a patient contributes
# roughly 0.6*60 = 36 training rows, which cannot support 80+ features. Handing the
# local arm the full FEATURES matrix would be a strawman, so it gets a compact set
# that ~36 rows can actually estimate.
def local_feature_set(signal):
    cand = window_cols(signal) + [f"{signal}_ewm0.3", f"{signal}_mean7", f"{signal}_slope7",
                                  f"{signal}_base_mean", "days_since_last"]
    return [c for c in dict.fromkeys(cand) if c in F.columns][:LOCAL_MAX_FEATURES]


def fit_local(kind, tr, ev, signal, h, delta=False, params=None):
    """Fit one estimator per series_id; fall back to the pooled fit when history is thin.

    The fallback matters: without it a patient with 20 sessions would be scored as a
    failure of the local *architecture* rather than of the data available to it.
    """
    y, lag = target_col(signal, h), f"{signal}_lag1"
    cols = local_feature_set(signal)
    out = np.full(len(ev), np.nan)

    def _fit(frame):
        yy = frame[y].values - frame[lag].values if delta else frame[y].values
        ok = np.isfinite(yy)
        if ok.sum() < LOCAL_MIN_TRAIN_ROWS:
            return None
        est = make_model(kind, **(params or {}))
        est.fit(frame[cols][ok], yy[ok])
        return est

    pooled = _fit(cap_rows(tr.dropna(subset=[y]), MAX_TRAIN_ROWS or 60_000))
    tr_by = {sid: g for sid, g in tr.dropna(subset=[y]).groupby("series_id", sort=False)}

    for sid, idx in ev.groupby("series_id", sort=False).indices.items():
        sub = ev.iloc[idx]
        est = (_fit(tr_by[sid]) if sid in tr_by else None) or pooled
        if est is None:
            continue
        p = est.predict(sub[cols])
        out[idx] = p + sub[lag].values if delta else p
    return out


if RUN_LOCAL_SCOPE:
    @register("local_ridge", "scope", "medium",
              "one ridge per patient, compact per-patient feature set", scope="local")
    def _f1l(tr, ev, s, h):
        return fit_local("ridge", tr, ev, s, h, params={"alpha": 1.0})

    @register("local_ridge_delta", "scope", "medium",
              "per-patient ridge on the Δ-target", scope="local")
    def _f2l(tr, ev, s, h):
        return fit_local("ridge", tr, ev, s, h, delta=True, params={"alpha": 1.0})

    @register("local_hgb", "scope", "high",
              "per-patient boosting - expected to overfit, included as the honest test",
              scope="local")
    def _f3l(tr, ev, s, h):
        return fit_local("hgb", tr, ev, s, h,
                         params={"max_iter": 80, "max_leaf_nodes": 8, "min_samples_leaf": 8})


HOLT_PARAMS = {}
TUNED = {}      # filled by the hyper-parameter search in §7.2

print(f"{len(FORECAST_MODELS)} single architectures registered")
display(pd.Series([m["family"] for m in FORECAST_MODELS.values()]).value_counts().rename("n").to_frame())

## Ensembling

In [ ]:
ENS_BASE = ("ridge", "hgb")


def ens_base_preds(tr, ev, s, h):
    """Predictions from each base leg on the SAME rows. Returns {name: array}."""
    out = {}
    for kind in ENS_BASE:
        try:
            out[kind] = np.asarray(fit_global(kind, tr, ev, s, h, params=TUNED.get(kind)), float)
        except Exception:
            pass
    try:
        a, b, p = HOLT_PARAMS.get((s, h), (0.4, 0.1, 0.9))
        out["holt"] = np.asarray(series_forecast(ev, s, h, holt_damped, alpha=a, beta=b, phi=p),
                                 float)
    except Exception:
        out.pop("holt", None)

    if len(out) < 2:
        out["ridge_delta"] = np.asarray(fit_global("ridge", tr, ev, s, h, delta=True), float)

    # a leg that does not align with `ev` is dropped rather than allowed to break the stack
    bad = [k for k, v in out.items() if v is None or len(np.atleast_1d(v)) != len(ev)]
    for k in bad:
        out.pop(k)
    if not out:
        out["ridge"] = np.asarray(fit_global("ridge", tr, ev, s, h), float)
    return out


def inner_split(tr):
    """Time-ordered inner split of the TRAINING rows only - never touches eval."""
    q = tr.groupby("series_id").step.transform(lambda x: x.rank(pct=True))
    return tr[q <= .75], tr[q > .75]


@register("ens_mean", "ensemble", "medium", "unweighted mean across families")
def _f1(tr, ev, s, h):
    P = ens_base_preds(tr, ev, s, h)
    return np.nanmean(np.column_stack(list(P.values())), axis=1)


@register("ens_median", "ensemble", "medium", "median across families - robust to one bad leg")
def _f2(tr, ev, s, h):
    P = ens_base_preds(tr, ev, s, h)
    return np.nanmedian(np.column_stack(list(P.values())), axis=1)


@register("ens_inv_mae", "ensemble", "medium", "weights inverse to inner-validation MAE")
def _f3(tr, ev, s, h):
    y = target_col(s, h)
    tr_in, tr_val = inner_split(tr)
    if len(tr_val) < 100 or len(tr_in) < 200:
        return _f1(tr, ev, s, h)
    Pv = ens_base_preds(tr_in, tr_val, s, h)      # weights learned on held-out TRAIN rows
    Pe = ens_base_preds(tr, ev, s, h)
    names = [k for k in Pv if k in Pe]
    w = np.array([1.0 / max(float(np.nanmean(np.abs(Pv[k] - tr_val[y].values))), 1e-6)
                  for k in names])
    w = w / w.sum()
    ENS_WEIGHTS[("ens_inv_mae", s, h)] = dict(zip(names, w))
    return np.nansum(np.column_stack([Pe[k] for k in names]) * w, axis=1)


@register("ens_stack_ridge", "ensemble", "high", "out-of-fold stack, non-negative meta-learner")
def _f4(tr, ev, s, h):
    y = target_col(s, h)
    oof, truth = {}, []
    folds = list(forward_chain_folds(tr, 3))
    for tr_m, va_m in folds:
        if tr_m.sum() < 200 or va_m.sum() < 60:
            continue
        P = ens_base_preds(tr[tr_m], tr[va_m], s, h)
        for k, v in P.items():
            oof.setdefault(k, []).append(np.asarray(v, float))
        truth.append(tr[va_m][y].values)
    if not truth:
        return _f1(tr, ev, s, h)
    names = [k for k in oof if len(oof[k]) == len(truth)]
    if len(names) < 2:
        return _f1(tr, ev, s, h)

    Xm = np.column_stack([np.concatenate(oof[k]) for k in names])
    ym = np.concatenate(truth)
    ok = np.isfinite(Xm).all(1) & np.isfinite(ym)
    if ok.sum() < 100:
        return _f1(tr, ev, s, h)
    try:                          # non-negative least squares: no leg gets a negative vote
        from scipy.optimize import nnls
        coef, _ = nnls(Xm[ok], ym[ok])
    except Exception:
        coef = np.clip(np.linalg.lstsq(Xm[ok], ym[ok], rcond=None)[0], 0, None)
    if coef.sum() <= 0:
        return _f1(tr, ev, s, h)
    coef = coef / coef.sum()
    ENS_WEIGHTS[("ens_stack_ridge", s, h)] = dict(zip(names, coef))
    Pe = ens_base_preds(tr, ev, s, h)
    return np.nansum(np.column_stack([Pe[k] for k in names]) * coef, axis=1)


ENS_WEIGHTS = {}

# ── optional gradient-boosting backends, registered only if importable ───────
try:
    import lightgbm as lgb

    @register("lightgbm", "trees", "medium", "leaf-wise boosting, L1 objective")
    def _g1(tr, ev, s, h):
        est = lgb.LGBMRegressor(objective="mae", n_estimators=400, learning_rate=.05,
                                num_leaves=31, n_jobs=-1, random_state=SEED, verbose=-1)
        return fit_global(est, tr, ev, s, h)
except Exception:
    print("LightGBM not installed - skipped")

try:
    import xgboost as xgb

    @register("xgboost", "trees", "medium", "level-wise boosting")
    def _g2(tr, ev, s, h):
        est = xgb.XGBRegressor(n_estimators=400, learning_rate=.05, max_depth=5,
                               subsample=.9, n_jobs=-1, random_state=SEED, verbosity=0)
        return fit_global(est, tr, ev, s, h)
except Exception:
    print("XGBoost not installed - skipped")

print(f"{len(FORECAST_MODELS)} architectures total")
print("ensembles:", [k for k, v in FORECAST_MODELS.items() if v["family"] == "ensemble"])

# 10. Hyperprameter tning

In [ ]:
# ---- hyper-parameter tuning --------------------------------------------------
# TUNER picks the search strategy. All three backends share ONE scorer (cv_score) and
# ONE cross-validation scheme (forward_chain_folds), so switching backends changes how
# the space is explored and nothing else - the reported MAEs stay comparable.
#   "optuna" : TPE over continuous ranges + fold-level median pruning. Default.
#   "grid"   : sklearn GridSearchCV over GRID_SPACE. Exhaustive and reproducible.
#   "random" : the original uniform draw over SEARCH_SPACE. Fallback.
import time

TUNER = "optuna"
TUNE_TRIALS = 10 if FAST_MODE else 40      # optuna only; grid size is set by GRID_SPACE

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    optuna = None
    if TUNER == "optuna":
        TUNER = "random"
        print("optuna not installed (pip install optuna) - falling back to TUNER='random'")

from sklearn.model_selection import GridSearchCV


def forward_chain_folds(d, n_folds):
    """Expanding-window folds inside each patient's own timeline."""
    q = d.groupby("series_id").step.transform(lambda s: s.rank(pct=True))
    edges = np.linspace(0.4, 1.0, n_folds + 1)
    for i in range(n_folds):
        yield (q <= edges[i]).values, ((q > edges[i]) & (q <= edges[i + 1])).values


# Discrete grids: used verbatim by "random", and by "grid" in the reduced GRID_SPACE
# form, because the full SEARCH_SPACE for hgb is 3^6 = 729 configurations.
SEARCH_SPACE = {
    "ridge": {"alpha": [0.1, 1.0, 10.0, 100.0, 300.0]},
    "elasticnet": {"alpha": [0.005, 0.02, 0.05, 0.2], "l1_ratio": [0.1, 0.5, 0.9]},
    "hgb": {"max_iter": [150, 250, 400], "learning_rate": [0.03, 0.07, 0.12],
            "max_leaf_nodes": [15, 31, 63], "min_samples_leaf": [20, 50, 100],
            "l2_regularization": [0.0, 0.5, 2.0], "max_features": [0.6, 0.8, 1.0]},
    "random_forest": {"n_estimators": [200, 400], "min_samples_leaf": [5, 20, 50],
                      "max_features": [0.3, 0.6, 1.0]},
}

GRID_SPACE = {
    "ridge": {"alpha": [1.0, 10.0, 100.0]},
    "elasticnet": {"alpha": [0.005, 0.05, 0.2], "l1_ratio": [0.1, 0.5, 0.9]},
    "hgb": {"max_iter": [250, 400], "learning_rate": [0.03, 0.07],
            "max_leaf_nodes": [31, 63], "min_samples_leaf": [20, 50]},
    "random_forest": {"n_estimators": [200, 400], "min_samples_leaf": [5, 20],
                      "max_features": [0.3, 0.6]},
}


def optuna_params(kind, trial):
    """Continuous ranges rather than a discrete grid - the point of using TPE."""
    if kind == "ridge":
        return {"alpha": trial.suggest_float("alpha", 1e-2, 1e3, log=True)}
    if kind == "elasticnet":
        return {"alpha": trial.suggest_float("alpha", 1e-3, 1.0, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.05, 0.95)}
    if kind == "hgb":
        return {"max_iter": trial.suggest_int("max_iter", 100, 600, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 8, 127, log=True),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 200, log=True),
                "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
                "max_features": trial.suggest_float("max_features", 0.4, 1.0)}
    if kind == "random_forest":
        return {"n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 80, log=True),
                "max_features": trial.suggest_float("max_features", 0.2, 1.0)}
    return {}


_searched = ({k for g in SEARCH_SPACE.values() for k in g}
             | {k for g in GRID_SPACE.values() for k in g})
assert not (set(GOVERNANCE) & _searched), \
    "a governance parameter leaked into a hyper-parameter grid"


def cv_score(kind, params, d, target, n_folds, trial=None):
    """Mean MAE across forward-chaining folds.

    When `trial` is passed the running mean is reported to Optuna after each fold, so a
    configuration that is already losing on fold 1 is killed before it burns folds 2-3.
    """
    errs = []
    for i, (tr_m, va_m) in enumerate(forward_chain_folds(d, n_folds)):
        tr, va = d[tr_m], d[va_m]
        if len(tr) < 200 or len(va) < 50:
            continue
        m = make_model(kind, **params).fit(tr[FEATURES], tr[target].values)
        errs.append(mean_absolute_error(va[target].values, m.predict(va[FEATURES])))
        if trial is not None:
            trial.report(float(np.mean(errs)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return float(np.mean(errs)) if errs else np.nan


class PatientForwardChain:
    """sklearn-compatible splitter wrapping forward_chain_folds.

    GridSearchCV only ever sees positional indices, so the per-patient percentile rank
    that defines a fold has to be resolved HERE, from the frame, and frozen into index
    arrays. This is the reason GridSearchCV cannot be dropped in without a shim: neither
    TimeSeriesSplit (one global timeline) nor GroupKFold (no time order) is correct when
    every patient has their own unaligned session history.
    """

    def __init__(self, d, n_folds):
        self.folds = [(np.flatnonzero(a), np.flatnonzero(b))
                      for a, b in forward_chain_folds(d, n_folds)]
        self.folds = [(a, b) for a, b in self.folds if len(a) >= 200 and len(b) >= 50]

    def split(self, X=None, y=None, groups=None):
        yield from self.folds

    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.folds)


OPTUNA_STUDIES = {}


def tune_optuna(kind, d, target):
    def objective(trial):
        sc = cv_score(kind, optuna_params(kind, trial), d, target, TUNE_FOLDS, trial=trial)
        if not np.isfinite(sc):
            raise optuna.TrialPruned()
        return sc

    study = optuna.create_study(
        direction="minimize", study_name=f"{kind}-{target}",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    study.optimize(objective, n_trials=TUNE_TRIALS, show_progress_bar=False)
    OPTUNA_STUDIES[kind] = study
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not done:
        return {}, np.nan
    return dict(study.best_params), float(study.best_value)


def tune_grid(kind, d, target):
    est = make_model(kind)
    pre = "model__" if hasattr(est, "named_steps") and "model" in est.named_steps else ""
    grid = {f"{pre}{k}": v for k, v in GRID_SPACE[kind].items()}
    gs = GridSearchCV(est, grid, scoring="neg_mean_absolute_error",
                      cv=PatientForwardChain(d, TUNE_FOLDS), n_jobs=1, refit=False)
    gs.fit(d[FEATURES], d[target].values)
    best = {(k[len(pre):] if pre else k): v for k, v in gs.best_params_.items()}
    return best, float(-gs.best_score_)


def tune_random(kind, d, target):
    rng = np.random.default_rng(SEED)
    grid = SEARCH_SPACE[kind]
    best, best_p = np.inf, {}
    for _ in range(TUNE_DRAWS):
        cand = {}
        for k, v in grid.items():
            pick = v[int(rng.integers(len(v)))]
            cand[k] = float(pick) if isinstance(pick, float) else int(pick)
        sc = cv_score(kind, cand, d, target, TUNE_FOLDS)
        if np.isfinite(sc) and sc < best:
            best, best_p = sc, cand
    return best_p, best


def run_tuning(signal="sbp", h=None):
    h = h or HORIZONS[0]
    target = target_col(signal, h)
    d = F[(F[target].notna()) & (F.split != "test")]
    d = (cap_rows(d, TUNE_SAMPLE_ROWS)
         .sort_values(["series_id", "step"]).reset_index(drop=True))
    backend = {"optuna": tune_optuna, "grid": tune_grid, "random": tune_random}[TUNER]

    log = []
    for kind in SEARCH_SPACE:
        base = cv_score(kind, {}, d, target, TUNE_FOLDS)     # defaults: the bar to beat
        t0 = time.time()
        try:
            best_p, best = backend(kind, d, target)
        except Exception as e:
            print(f"  {kind}: {TUNER} search failed ({type(e).__name__}: {e}) - keeping defaults")
            best_p, best = {}, np.nan
        # never ship a configuration that lost to the defaults on the same folds
        if not np.isfinite(best) or best >= base:
            best_p, best = {}, base
        TUNED[kind] = best_p
        n_tr = (len(OPTUNA_STUDIES[kind].trials) if TUNER == "optuna" and kind in OPTUNA_STUDIES
                else (int(np.prod([len(v) for v in GRID_SPACE[kind].values()]))
                      if TUNER == "grid" else TUNE_DRAWS))
        log.append(dict(model=kind, cv_mae_default=round(base, 4), cv_mae_tuned=round(best, 4),
                        gain_pct=round(100 * (base - best) / base, 2) if base else np.nan,
                        n_configs=n_tr, secs=round(time.time() - t0, 1), params=best_p))
    return pd.DataFrame(log)


if TUNER == "grid":
    _n = sum(int(np.prod([len(v) for v in g.values()])) for g in GRID_SPACE.values())
    print(f"tuning: GridSearchCV, {_n} configurations x {TUNE_FOLDS} folds = {_n * TUNE_FOLDS} fits")
elif TUNER == "optuna":
    print(f"tuning: Optuna TPE + MedianPruner, {TUNE_TRIALS} trials x {TUNE_FOLDS} folds per family")
else:
    print(f"tuning: uniform random, {TUNE_DRAWS} draws x {TUNE_FOLDS} folds per family")
print(f"on <= {TUNE_SAMPLE_ROWS:,} rows, {len(FEATURES)} features ...")

TUNE_LOG = run_tuning()
display(TUNE_LOG)
print("\ntuned parameters carried into the bake-off:")
print(json.dumps(TUNED, indent=2, default=str))

# ---- search diagnostics ------------------------------------------------------
if TUNER == "optuna" and OPTUNA_STUDIES:
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.4))
    for kind, st in OPTUNA_STUDIES.items():
        vals = [t.value for t in st.trials
                if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
        if vals:
            ax[0].plot(np.minimum.accumulate(vals), marker="o", ms=3, label=kind)
    ax[0].set(title="Optuna - best CV MAE so far", xlabel="completed trial",
              ylabel="CV MAE (mmHg)")
    ax[0].legend(fontsize=8)

    rows = []
    for kind, st in OPTUNA_STUDIES.items():
        states = pd.Series([t.state.name for t in st.trials]).value_counts()
        rows.append(dict(model=kind, **states.to_dict()))
    PRUNING = pd.DataFrame(rows).fillna(0).set_index("model")
    PRUNING.plot(kind="barh", stacked=True, ax=ax[1], colormap="Set2")
    ax[1].set(title="Trial outcomes (pruned trials are folds not paid for)",
              xlabel="trials")
    plt.tight_layout()
    plt.show()

    for kind, st in OPTUNA_STUDIES.items():
        try:
            imp = optuna.importance.get_param_importances(st)
            top = ", ".join(f"{k} {v:.0%}" for k, v in list(imp.items())[:3])
            print(f"{kind:14s} most influential: {top}")
        except Exception:
            pass

## Bake-off rnner

In [ ]:
BAKEOFF_SIGNAL = "sbp"


def run_bakeoff(signal=BAKEOFF_SIGNAL, horizons=HORIZONS):
    """Evaluate every architecture on three eval sets.

    Two training frames, deliberately different:
      tr_fit : train rows of the FIT cohort only. Global models see nothing from a
               holdout patient, which is what makes `holdout_pt` a real cold-start test.
      tr_own : train rows of every patient. Only models registered scope="local" get
               this, because a per-patient model legitimately reads that patient's own
               earlier sessions at serving time - it is their past, not someone else's.

    Eval sets:
      val / test : later sessions of FIT-cohort patients (seen patient, unseen time)
      holdout_pt : later sessions of HOLDOUT patients (unseen patient AND unseen time)
    """
    rows, per_row = [], {}
    for h in horizons:
        y = target_col(signal, h)
        tr_fit = F[(F.split == "train") & F[y].notna() & (F.patient_split == "fit")]
        tr_own = F[(F.split == "train") & F[y].notna()]

        parts = {
            "val": F[(F.split == "val") & F[y].notna() & (F.patient_split == "fit")],
            "test": F[(F.split == "test") & F[y].notna() & (F.patient_split == "fit")],
            "holdout_pt": F[(F.split == "test") & F[y].notna() & (F.patient_split == "holdout")],
        }

        for split, ev in parts.items():                      # baselines first
            if not len(ev):
                continue
            for name, pred in forecast_baselines(ev, signal, h).items():
                r = evaluate(ev[y].values, pred, ev[f"{signal}_lag1"].values,
                             model=name, family="baseline", cost="low", scope="none",
                             horizon=h, split=split)
                if r:
                    rows.append(r)
                if split == "test":
                    per_row[(name, h)] = (ev[y].values, np.asarray(pred, float),
                                          ev.series_id.values)

        for name, meta in FORECAST_MODELS.items():
            tr = tr_own if meta.get("scope") == "local" else tr_fit
            for split, ev in parts.items():
                if not len(ev):
                    continue
                try:
                    pred = np.asarray(meta["fn"](tr, ev, signal, h), float)
                except Exception as e:
                    rows.append(dict(model=name, family=meta["family"], cost=meta["cost"],
                                     scope=meta.get("scope", "global"), horizon=h, split=split,
                                     n=0, MAE=np.nan, note=f"{type(e).__name__}: {e}"))
                    continue
                r = evaluate(ev[y].values, pred, ev[f"{signal}_lag1"].values,
                             model=name, family=meta["family"], cost=meta["cost"],
                             scope=meta.get("scope", "global"), horizon=h, split=split)
                if r:
                    rows.append(r)
                if split == "test":
                    per_row[(name, h)] = (ev[y].values, pred, ev.series_id.values)
            print(f"  h={h} {name:22s} done")
    return pd.DataFrame(rows), per_row


print(f"running the bake-off: {len(FORECAST_MODELS)} architectures x {len(HORIZONS)} horizons")
print(f"  fit cohort     : {F[F.patient_split == 'fit'].series_id.nunique()} patients "
      f"-> train the global models, evaluated on val/test")
print(f"  holdout cohort : {F[F.patient_split == 'holdout'].series_id.nunique()} patients "
      f"-> never seen by a global model, evaluated on holdout_pt")
BOARD, PER_ROW = run_bakeoff()

val_board = (BOARD[BOARD.split == "val"]
             .groupby(["model", "family", "cost"], as_index=False)
             .agg(MAE=("MAE", "mean"), n=("n", "sum"))
             .sort_values("MAE"))
display(val_board.head(20).round(3))

# ---- global vs per-patient scope, and the price of an unseen patient ---------
SCOPE_BOARD = (BOARD[(BOARD.split.isin(["test", "holdout_pt"])) & BOARD.MAE.notna()]
               .pivot_table(index=["model", "family", "scope"], columns="split",
                            values="MAE", aggfunc="mean")
               .reset_index())
if {"test", "holdout_pt"} <= set(SCOPE_BOARD.columns):
    SCOPE_BOARD["cold_start_penalty"] = SCOPE_BOARD.holdout_pt - SCOPE_BOARD.test
SCOPE_BOARD = SCOPE_BOARD.sort_values("test")
display(SCOPE_BOARD.head(25).round(3)
        .style.background_gradient(cmap="Reds", subset=["cold_start_penalty"])
        .set_caption("test = seen patient / unseen time | holdout_pt = unseen patient")
        .hide(axis="index"))

learned = SCOPE_BOARD[SCOPE_BOARD.scope.isin(["global", "local"])]
if len(learned):
    display(learned.groupby("scope")[["test", "holdout_pt"]].agg(["mean", "min", "size"])
            .round(3))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.6))
sub = SCOPE_BOARD[SCOPE_BOARD.scope != "none"].head(14)
sub.plot(x="model", y=["test", "holdout_pt"], kind="barh", ax=ax[0])
ax[0].set(title="Seen-patient vs unseen-patient MAE", xlabel="MAE (mmHg)", ylabel="")
if "cold_start_penalty" in SCOPE_BOARD:
    sns.boxplot(data=SCOPE_BOARD[SCOPE_BOARD.scope != "none"], x="scope",
                y="cold_start_penalty", ax=ax[1], color="steelblue")
    ax[1].axhline(0, c="k", ls="--", lw=1)
    ax[1].set(title="Cold-start penalty by scope", ylabel="holdout_pt MAE - test MAE")
plt.tight_layout()
plt.show()

failed = BOARD[BOARD.MAE.isna()]
if len(failed):                       # "note" only exists once something has errored
    cols = [c for c in ("model", "horizon", "split", "note") if c in failed.columns]
    print("\narchitectures that errored:")
    display(failed[cols].drop_duplicates("model"))

## Paired comparison and selection

In [ ]:
def pooled_errors(name):
    """Concatenate a model's test predictions across horizons."""
    ys, ps, sids = [], [], []
    for h in HORIZONS:
        if (name, h) not in PER_ROW:
            return None
        y, p, sid = PER_ROW[(name, h)]
        ys.append(y)
        ps.append(p)
        sids.append(sid)
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(sids)


leader = val_board[~val_board.model.str.startswith("baseline:")].iloc[0].model
ref = pooled_errors(leader)
print(f"validation leader: {leader}")

cmp_rows = []
for name in sorted({k[0] for k in PER_ROW}):
    cur = pooled_errors(name)
    if cur is None or ref is None:
        continue
    delta, (lo, hi), p_worse = paired_delta(np.abs(cur[1] - cur[0]), np.abs(ref[1] - ref[0]))
    # per-patient win rate: does it beat the leader for the median patient, or just on average?
    dfw = pd.DataFrame({"sid": cur[2], "e": np.abs(cur[1] - cur[0]),
                        "eref": np.abs(ref[1] - ref[0])}).dropna()
    pp = dfw.groupby("sid")[["e", "eref"]].mean()
    cmp_rows.append(dict(model=name, test_MAE=round(float(np.nanmean(np.abs(cur[1] - cur[0]))), 3),
                         delta_vs_leader=round(delta, 3), lo=round(lo, 3), hi=round(hi, 3),
                         ties_leader=bool(lo <= 0 <= hi),
                         patient_win_rate=round(float((pp.e < pp.eref).mean()), 3)))

CMP = pd.DataFrame(cmp_rows).sort_values("test_MAE")
CMP = CMP.merge(val_board[["model", "family", "cost"]], on="model", how="left")
display(CMP.head(25))

# tie set = everything statistically indistinguishable from the leader, learned models only
TIE_SET = set(CMP[(CMP.ties_leader) & (~CMP.model.str.startswith("baseline:"))].model) | {leader}
COST_RANK = {"low": 0, "medium": 1, "high": 2}
cand = val_board[val_board.model.isin(TIE_SET)].copy()
cand["cost_rank"] = cand.cost.map(COST_RANK).fillna(1)
cand = cand.sort_values(["cost_rank", "MAE"])
WINNER = cand.iloc[0].model

print(f"\ntie set ({len(TIE_SET)}): {sorted(TIE_SET)}")
print(f"cheapest member of the tie set -> WINNER = {WINNER}")

# ship decision against the strongest baseline
base_board = val_board[val_board.model.str.startswith("baseline:")]
best_base = base_board.iloc[0].model
b = pooled_errors(best_base)
w = pooled_errors(WINNER)
gain, (glo, ghi), _ = paired_delta(np.abs(b[1] - b[0]), np.abs(w[1] - w[0]))
print(f"\nstrongest baseline: {best_base}")
print(f"paired gain of {WINNER} over it: {gain:+.3f} mmHg  95% CI [{glo:+.3f}, {ghi:+.3f}]")
DECISION = "learned model" if glo > 0 else "BASELINE"
print(f"selection rule -> {DECISION}")

## Diagnostics

In [ ]:
sb = BOARD[(BOARD.split == "test") & BOARD.MAE.notna()]
top = val_board.head(14).model.tolist()

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
sns.barplot(data=sb[sb.model.isin(top)], y="model", x="MAE", hue="horizon", ax=ax[0])
ax[0].set(title="Test MAE by architecture and horizon", xlabel="MAE (mmHg)")
piv = sb.pivot_table(index="model", columns="horizon", values="MAE").loc[
    [m for m in top if m in sb.model.values]]
sns.heatmap(piv, annot=True, fmt=".2f", cmap="viridis_r", ax=ax[1])
ax[1].set(title="MAE heatmap")
fam = sb.groupby("family").MAE.mean().sort_values()
sns.barplot(x=fam.values, y=fam.index, ax=ax[2], color="steelblue")
ax[2].set(title="Mean test MAE by family", xlabel="MAE (mmHg)")
plt.tight_layout()
plt.show()

yt, yp, sid = pooled_errors(WINNER)
resid = yp - yt
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
ax[0].scatter(yt, yp, s=8, alpha=.25)
lim = [np.nanmin(yt), np.nanmax(yt)]
ax[0].plot(lim, lim, "k--")
ax[0].set(title=f"{WINNER}: predicted vs actual", xlabel="actual SBP", ylabel="predicted")
sns.histplot(resid, bins=60, ax=ax[1], color="indianred")
ax[1].axvline(0, c="k", ls="--")
ax[1].set(title=f"Residuals (bias {np.nanmean(resid):+.2f} mmHg)", xlabel="pred - actual")
per_pt = pd.DataFrame({"sid": sid, "e": np.abs(resid)}).groupby("sid").e.mean().sort_values()
ax[2].plot(np.arange(len(per_pt)), per_pt.values, lw=1.5)
ax[2].axhline(per_pt.mean(), c="crimson", ls="--", label=f"mean {per_pt.mean():.2f}")
ax[2].set(title="Per-patient MAE, sorted", xlabel="patient rank", ylabel="MAE (mmHg)")
ax[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

worst = per_pt.tail(max(1, len(per_pt) // 10))
print(f"worst 10% of patients carry {worst.sum() / per_pt.sum():.1%} of the total error")

# ---- importance -------------------------------------------------------------
# Single-column permutation splits credit between correlated features and makes each
# of them look useless. The selection cell already grouped the survivors into
# redundancy clusters, so permute each cluster as a BLOCK: the score drop is then
# the value of the information, not of one arbitrary encoding of it.
h0 = HORIZONS[0]
y0 = target_col(BAKEOFF_SIGNAL, h0)
d_tr = cap_rows(F[(F.split == "train") & F[y0].notna()], 20_000)
d_te = cap_rows(F[(F.split == "test") & F[y0].notna()], 4_000)
imp_model = make_model("hgb", **(TUNED.get("hgb") or {})).fit(d_tr[FEATURES], d_tr[y0].values)


def block_importance(model, X, y, blocks, n_repeats=3, seed=SEED):
    """MAE increase when a whole correlated block is permuted with one shared index."""
    rng = np.random.default_rng(seed)
    base = mean_absolute_error(y, model.predict(X))
    out = {}
    for name, cols in blocks.items():
        cols = [c for c in cols if c in X.columns]
        if not cols:
            continue
        d = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = Xp[cols].to_numpy()[rng.permutation(len(Xp))]
            d.append(mean_absolute_error(y, model.predict(Xp)) - base)
        out[name] = float(np.mean(d))
    return pd.Series(out, name="importance").sort_values(ascending=False), base


if "CLUSTERS" in dir():
    _c = CLUSTERS[CLUSTERS.feature.isin(FEATURES)]
    BLOCKS = {}
    for cid, blk in _c.groupby("cluster"):
        members = blk.sort_values("relevance", ascending=False).feature.tolist()
        BLOCKS[members[0] if len(members) == 1 else f"{members[0]} (+{len(members)-1})"] = members
    for f in FEATURES:                      # anything the screen never clustered
        if f not in set(_c.feature):
            BLOCKS[f] = [f]
else:                                        # fallback: block by feature family
    BLOCKS = {g: d.tolist() for g, d in
              pd.Series(FEATURES).groupby(pd.Series(FEATURES).map(feature_group))}

blk_imp, base_mae = block_importance(imp_model, d_te[FEATURES], d_te[y0].values, BLOCKS)
IMP = (blk_imp.rename_axis("block").reset_index()
       .assign(group=lambda x: x.block.str.split(" ").str[0].map(feature_group),
               n_features=lambda x: x.block.map(lambda b: len(BLOCKS[b]))))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
sns.barplot(data=IMP.head(18), x="importance", y="block", ax=ax[0], color="steelblue")
ax[0].set(title=f"Block permutation importance - top 18 (base MAE {base_mae:.2f} mmHg)",
          xlabel="MAE increase (mmHg)")
grp = IMP.groupby("group").importance.sum().sort_values()
sns.barplot(x=grp.values, y=grp.index, ax=ax[1], color="seagreen")
ax[1].set(title="Importance summed by feature group", xlabel="MAE increase (mmHg)")
plt.tight_layout()
plt.show()

display(IMP.head(12).style.format({"importance": "{:+.3f}"}).hide(axis="index"))
print(f"blocks worth > 0.10 mmHg: {int((IMP.importance > .10).sum())} of {len(IMP)}")

## Final Forecaster

In [ ]:
class FinalForecaster:
    """Frozen, servable forecaster for one (signal, horizon)."""

    def __init__(self, kind, signal, h, est=None, cols=None, legs=None, weights=None,
                 holt=None, intercepts=None, name=""):
        self.kind, self.signal, self.h, self.name = kind, signal, h, name
        self.est, self.cols = est, cols
        self.legs, self.weights = legs or {}, weights or {}
        self.holt, self.intercepts = holt, intercepts

    def _tabular(self, est, X, lag1, delta):
        p = est.predict(X)
        return p + lag1 if delta else p

    def predict(self, X, lag1=None, hist=None, series_id=None):
        """X: feature frame. lag1: previous reading. hist: raw signal history (classical legs)."""
        if self.kind in ("tabular", "delta"):
            return self._tabular(self.est, X[FEATURES], lag1, self.kind == "delta")
        if self.kind in ("window", "window_delta"):
            return self._tabular(self.est, X[self.cols], lag1, self.kind == "window_delta")
        if self.kind == "global_intercept":
            base = self.est.predict(X[FEATURES])
            adj = (pd.Series(series_id).map(self.intercepts).fillna(0.0).values
                   if series_id is not None else 0.0)
            return base + adj
        if self.kind == "classical":
            fn, kw = self.holt
            return np.array([fn(np.asarray(hist, float), self.h, **kw)[-1]])
        if self.kind == "ensemble":
            parts, w = [], []
            for leg, weight in self.weights.items():
                if leg == "holt":
                    fn, kw = self.holt
                    parts.append(np.full(len(X), fn(np.asarray(hist, float), self.h, **kw)[-1])
                                 if hist is not None else np.full(len(X), np.nan))
                else:
                    delta = leg.endswith("_delta")
                    parts.append(self._tabular(self.legs[leg], X[FEATURES], lag1, delta))
                w.append(weight)
            M = np.column_stack(parts)
            w = np.asarray(w, float)
            ok = np.isfinite(M)
            M = np.where(ok, M, 0.0)
            wsum = (ok * w).sum(axis=1)
            return (M * w).sum(axis=1) / np.where(wsum > 0, wsum, np.nan)
        raise ValueError(self.kind)


# how each architecture is rebuilt as a frozen artifact
TABULAR_SPECS = {
    "ridge": ("tabular", "ridge"), "elasticnet": ("tabular", "elasticnet"),
    "huber": ("tabular", "huber"), "hgb_mae": ("tabular", "hgb"),
    "hgb_mse": ("tabular", "hgb_mse"), "random_forest": ("tabular", "random_forest"),
    "extra_trees": ("tabular", "extra_trees"), "knn": ("tabular", "knn"),
    "mlp": ("tabular", "mlp"), "ridge_delta": ("delta", "ridge"), "hgb_delta": ("delta", "hgb"),
}


def fit_final(name, tr, signal, h):
    """Refit one architecture on train+val and wrap it as a servable object."""
    y = target_col(signal, h)
    tr = tr[tr[y].notna()]

    if name in TABULAR_SPECS:
        kind, mk = TABULAR_SPECS[name]
        est = make_model(mk, **(TUNED.get(mk) or {}))
        fit_rows = cap_rows(tr, KERNEL_FIT_ROWS if mk in KERNEL_KINDS else MAX_TRAIN_ROWS)
        if kind == "delta":
            base = fit_rows[f"{signal}_lag1"].values
            ok = np.isfinite(base) & np.isfinite(fit_rows[y].values)
            est.fit(fit_rows.loc[ok, FEATURES], fit_rows.loc[ok, y].values - base[ok])
        else:
            est.fit(fit_rows[FEATURES], fit_rows[y].values)
        return FinalForecaster(kind, signal, h, est=est, name=name)

    if name in ("window_linear", "window_delta"):
        cols = window_cols(signal)
        est = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
                        ("m", Ridge(alpha=1.0))])
        rows = cap_rows(tr.dropna(subset=[y]), MAX_TRAIN_ROWS)
        if name == "window_delta":
            est.fit(rows[cols], rows[y].values - rows[f"{signal}_lag1"].values)
            return FinalForecaster("window_delta", signal, h, est=est, cols=cols, name=name)
        est.fit(rows[cols], rows[y].values)
        return FinalForecaster("window", signal, h, est=est, cols=cols, name=name)

    if name == "global_plus_intercept":
        rows = cap_rows(tr, MAX_TRAIN_ROWS)
        est = make_model("ridge", **(TUNED.get("ridge") or {})).fit(rows[FEATURES], rows[y].values)
        inter = pd.Series(rows[y].values - est.predict(rows[FEATURES]),
                          index=rows.series_id.values).groupby(level=0).mean().to_dict()
        return FinalForecaster("global_intercept", signal, h, est=est, intercepts=inter, name=name)

    if name in ("holt_damped", "theta"):
        if name == "theta":
            return FinalForecaster("classical", signal, h, holt=(theta_method, {"alpha": 0.3}),
                                   name=name)
        a, b, p = HOLT_PARAMS.get((signal, h), (0.3, 0.05, 0.9))
        return FinalForecaster("classical", signal, h,
                               holt=(holt_damped, {"alpha": a, "beta": b, "phi": p}), name=name)

    if name.startswith("ens_"):
        # refit the legs on train+val, then reuse the weights learned during the bake-off
        weights = ENS_WEIGHTS.get((name, signal, h))
        if not weights:                       # ens_mean / ens_median -> equal weights
            weights = {k: 1.0 / len(ENS_BASE) for k in ENS_BASE}
        legs = {}
        for leg in weights:
            if leg == "holt":
                continue
            mk = leg.replace("_delta", "")
            est = make_model(mk, **(TUNED.get(mk) or {}))
            rows = cap_rows(tr, MAX_TRAIN_ROWS)
            if leg.endswith("_delta"):
                base = rows[f"{signal}_lag1"].values
                ok = np.isfinite(base) & np.isfinite(rows[y].values)
                est.fit(rows.loc[ok, FEATURES], rows.loc[ok, y].values - base[ok])
            else:
                est.fit(rows[FEATURES], rows[y].values)
            legs[leg] = est
        a, b, p = HOLT_PARAMS.get((signal, h), (0.4, 0.1, 0.9))
        return FinalForecaster("ensemble", signal, h, legs=legs, weights=weights,
                               holt=(holt_damped, {"alpha": a, "beta": b, "phi": p}), name=name)

    return None       # not expressible as a frozen artifact (e.g. local_ar)


SERVABLE = WINNER
if fit_final(WINNER, F.head(500), BAKEOFF_SIGNAL, HORIZONS[0]) is None:
    fallback = [m for m in cand.model if fit_final(m, F.head(500), BAKEOFF_SIGNAL,
                                                   HORIZONS[0]) is not None]
    SERVABLE = fallback[0] if fallback else "ridge"
    print(f"{WINNER} cannot be frozen as a standalone artifact "
          f"(it refits per patient at serving time); shipping {SERVABLE} from the same tie set")

# Every patient, fit and holdout cohorts alike. The cohort split exists to keep the
# bake-off honest about unseen patients; the model that actually ships should learn from
# all available history. Nothing below feeds back into a reported evaluation number.
fit_rows = F[F.split.isin(["train", "val"])]
FORECASTERS = {}
for s in SIGNALS:
    for h in HORIZONS:
        FORECASTERS[(s, h)] = fit_final(SERVABLE, fit_rows, s, h)
        print(f"  refit {SERVABLE} for {s} h={h}")

print(f"\nfinal forecaster: {SERVABLE}, {len(FORECASTERS)} frozen models "
      f"({len(SIGNALS)} signals x {len(HORIZONS)} horizons)")

In [ ]:
# =============================================================================
# MODEL 4  -  the symptom head (multi-task with the BP forecaster)
# =============================================================================
# The BP head and this head share one feature matrix and one set of temporal splits, so
# a single feature row yields both the mmHg forecast and the symptom probabilities. Both
# are fitted for EVERY horizon in HORIZONS, and both are evaluated on the same three
# arms as the BP bake-off - including holdout_pt, the unseen-patient cold start.
#
# Metrics are NOT accuracy or ROC-AUC. Most of these symptoms occur in 0.1-12% of
# sessions, where predicting "never" scores >88% accurate and ROC-AUC is dominated by the
# negative class. What matters operationally: does the probability mean what it says
# (Brier skill, calibration), and among the sessions we flag, how many are real
# (precision at a staffing-feasible alert budget). PR-AUC is always reported as a
# multiple of the base rate, because 0.2 is excellent at 2% and useless at 20%.
#
# Every number below measures the synthesis cell, not physiology.

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, roc_auc_score,
                             precision_score, recall_score)
try:                                    # sklearn >= 1.6 replaced cv="prefit"
    from sklearn.frozen import FrozenEstimator
    _calibrate_fitted = lambda est: CalibratedClassifierCV(FrozenEstimator(est),
                                                           method="isotonic")
except ImportError:
    _calibrate_fitted = lambda est: CalibratedClassifierCV(est, method="isotonic",
                                                           cv="prefit")

SYMPTOM_TARGETS = SYMPTOM_GROUPS + SYMPTOMS
MIN_POSITIVES = 60          # below this a per-symptom model is not worth fitting


def symptom_frame(split, cohort="fit"):
    d = F[(F.split == split) & (F.patient_split == cohort)]
    return d[d[f"y_sym_any_h{HORIZONS[0]}"].notna()]


def make_symptom_model(kind="hgb"):
    if kind == "logistic":
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", LogisticRegression(max_iter=2000, C=0.3,
                                                      class_weight="balanced"))])
    return HistGradientBoostingClassifier(
        random_state=SEED, max_iter=250, learning_rate=0.06,
        max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0)


def eval_symptom(y, p, cut=None, budget_pct=ALERT_BUDGET_PCT):
    """Rare-event metrics, each reported against the base rate it has to beat.

    `cut` is the probability threshold fixed on VALIDATION. Passing it is what makes the
    test and holdout numbers honest: choosing the threshold on the split you are scoring
    would quietly optimise precision after the fact.
    """
    y = np.asarray(y, int)
    base = float(y.mean())
    if base <= 0 or base >= 1 or len(y) < 50:
        return {}
    if cut is None:
        cut = float(np.percentile(p, 100 - budget_pct))
    flag = (p >= cut).astype(int)
    prec = float(precision_score(y, flag, zero_division=0))
    return dict(
        n=len(y), base_rate=round(base, 4),
        PR_AUC=round(float(average_precision_score(y, p)), 4),
        PR_lift=round(float(average_precision_score(y, p)) / base, 2),
        ROC_AUC=round(float(roc_auc_score(y, p)), 4),
        Brier_skill=round(1 - brier_score_loss(y, p) / (base * (1 - base)), 4),
        prec_at_budget=round(prec, 4),
        recall_at_budget=round(float(recall_score(y, flag, zero_division=0)), 4),
        lift_at_budget=round(prec / base, 2),
        alert_rate=round(float(flag.mean()), 4),
    )


d_tr, d_va, d_te = symptom_frame("train"), symptom_frame("val"), symptom_frame("test")
d_ho = symptom_frame("test", cohort="holdout")      # unseen PATIENT, unseen time
print(f"symptom head: {len(d_tr):,} train / {len(d_va):,} val / {len(d_te):,} test / "
      f"{len(d_ho):,} holdout-patient rows, {len(HORIZONS)} horizons")

SYMPTOM_MODELS, SYMPTOM_CUTS, SYMPTOM_SKIPPED = {}, {}, []
sym_rows = []
for h in HORIZONS:
    for s in SYMPTOM_TARGETS:
        y = f"y_sym_{s}_h{h}"
        if y not in F.columns:
            continue
        ytr = d_tr[y].dropna().astype(int)
        if ytr.sum() < MIN_POSITIVES:
            if h == HORIZONS[0]:
                # record WHY, and how many sessions would be needed, rather than
                # silently dropping the symptom from the report
                rate = max(float(SYMPTOM_SPEC.get(s, {}).get("rate", np.nan)), 1e-9)
                SYMPTOM_SKIPPED.append(dict(symptom=s, positives=int(ytr.sum()),
                                            rate=SYMPTOM_SPEC.get(s, {}).get("rate"),
                                            train_rows_needed=int(MIN_POSITIVES / rate)))
            continue

        m_tr = d_tr[d_tr[y].notna()]
        m_va = d_va[d_va[y].notna()]
        base_est = make_symptom_model("hgb")
        base_est.fit(m_tr[FEATURES], m_tr[y].values.astype(int))
        # calibrate on VALIDATION; fitting calibration on train relearns the training fit
        est = _calibrate_fitted(base_est)
        est.fit(m_va[FEATURES], m_va[y].values.astype(int))
        SYMPTOM_MODELS[(s, h)] = est

        # threshold fixed on validation, then reused unchanged on test and holdout
        p_va = est.predict_proba(m_va[FEATURES])[:, 1]
        cut = float(np.percentile(p_va, 100 - ALERT_BUDGET_PCT))
        SYMPTOM_CUTS[(s, h)] = cut

        row = dict(symptom=s, horizon=h,
                   mechanism=SYMPTOM_MECHANISM.get(s, "group"),
                   red_flag=s in SYMPTOM_RED_FLAG)
        for arm, frame in (("test", d_te), ("holdout_pt", d_ho)):
            f2 = frame[frame[y].notna()]
            if not len(f2):
                continue
            p = est.predict_proba(f2[FEATURES])[:, 1]
            r = eval_symptom(f2[y].values.astype(int), p, cut=cut)
            row.update({(k if arm == "test" else f"{arm}_{k}"): v for k, v in r.items()})
        # the reference every learned score must beat: this patient's own recent rate
        f2 = d_te[d_te[y].notna()]
        col = f"sym_{s}_rate30"
        p_ref = (f2[col].fillna(ytr.mean()).values if col in f2.columns
                 else np.full(len(f2), ytr.mean()))
        r_ref = eval_symptom(f2[y].values.astype(int), p_ref)
        row["PR_AUC_personal_rate"] = r_ref.get("PR_AUC")
        row["beats_personal_rate"] = bool(row.get("PR_AUC", 0) > r_ref.get("PR_AUC", 1))
        sym_rows.append(row)

SYMPTOM_BOARD = pd.DataFrame(sym_rows)
_h0 = SYMPTOM_BOARD[SYMPTOM_BOARD.horizon == HORIZONS[0]]
_cols = [c for c in ["symptom", "mechanism", "red_flag", "base_rate", "PR_AUC", "PR_lift",
                     "Brier_skill", "lift_at_budget", "holdout_pt_PR_lift",
                     "holdout_pt_lift_at_budget", "beats_personal_rate"]
         if c in _h0.columns]
display(_h0[_cols].style.hide(axis="index")
        .set_caption(f"symptom head, h={HORIZONS[0]} "
                     "(SYNTHETIC LABELS - measures the generator, not physiology)"))

if SYMPTOM_SKIPPED:
    display(pd.DataFrame(SYMPTOM_SKIPPED).style.hide(axis="index")
            .set_caption(f"symptoms below the {MIN_POSITIVES}-positive floor: not modelled "
                         "individually, but still inside sym_red_flag and the mechanism "
                         "rollups"))

# ---- calibration is the only property that transfers to a real deployment ----------
plotted = [s for s in SYMPTOM_GROUPS if (s, HORIZONS[0]) in SYMPTOM_MODELS]
fig, ax = plt.subplots(1, 4, figsize=(21, 4.5))
for s in plotted:
    y = f"y_sym_{s}_h{HORIZONS[0]}"
    f2 = d_te[d_te[y].notna()]
    p = SYMPTOM_MODELS[(s, HORIZONS[0])].predict_proba(f2[FEATURES])[:, 1]
    q = pd.qcut(pd.Series(p), 10, labels=False, duplicates="drop")
    obs = pd.DataFrame({"p": p, "y": f2[y].values}).groupby(q).agg(
        pred=("p", "mean"), actual=("y", "mean"))
    ax[0].plot(obs.pred, obs.actual, marker="o", ms=4, label=s)
lim = ax[0].get_xlim()
ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set(title="Calibration on test (decile)", xlabel="predicted", ylabel="observed")
ax[0].legend(fontsize=7)

sb = _h0.dropna(subset=["PR_AUC"])
ax[1].barh(sb.symptom, sb.PR_lift, color="steelblue")
ax[1].axvline(1, c="crimson", ls="--")
ax[1].set(title="PR-AUC as a multiple of base rate", xlabel="lift")
if "holdout_pt_PR_lift" in sb:
    ax[2].scatter(sb.PR_lift, sb.holdout_pt_PR_lift, s=30)
    m = float(np.nanmax([sb.PR_lift.max(), sb.holdout_pt_PR_lift.max(), 1]))
    ax[2].plot([0, m], [0, m], "k--", lw=1)
    ax[2].set(title="Seen vs unseen patient", xlabel="test PR lift",
              ylabel="holdout-patient PR lift")
hz = SYMPTOM_BOARD.dropna(subset=["PR_lift"]).pivot_table(
    index="horizon", columns="mechanism", values="PR_lift", aggfunc="mean")
hz.plot(marker="o", ax=ax[3])
ax[3].set(title="PR lift decay by horizon", xlabel="sessions ahead", ylabel="mean PR lift")
ax[3].legend(fontsize=7)
plt.tight_layout()
plt.show()

_beat = _h0[_h0.beats_personal_rate == True].symptom.tolist()
print(f"beats the patient's own 30-session rate: {_beat or 'none'}")
print(f"{len(SYMPTOM_MODELS)} symptom models fitted "
      f"({len(SYMPTOM_TARGETS)} targets x {len(HORIZONS)} horizons, "
      f"{len(SYMPTOM_SKIPPED)} below the positive floor)")
print("REMINDER: these labels were generated in the synthesis cell. The numbers above "
      "verify the pipeline end to end; they are not clinical evidence.")

# 11. Model 02 - The personalized offset model

In [ ]:
def cohort_key(age, is_male):
    band = "<50" if age < 50 else "50-64" if age < 65 else "65-74" if age < 75 else "75+"
    return f"{band}|{'M' if is_male == 1 else 'F'}"


def apply_caps(pred):
    """The governance mechanism, applied identically to every candidate."""
    off = np.clip(np.asarray(pred, float) - POPULATION_THRESHOLD_MMHG,
                  -OFFSET_CAP_TIGHTEN, OFFSET_CAP_LOOSEN)
    return POPULATION_THRESHOLD_MMHG + off


class OffsetModel:
    """Capped shrinkage blend of a patient's own band and a demographic cohort prior."""

    def __init__(self, warm=48, k=30.0, q=0.90):
        self.warm, self.k, self.q = warm, k, q
        self.cohort_prior, self.global_prior = {}, POPULATION_THRESHOLD_MMHG

    def fit(self, F, prior_pool="fit"):
        head = F[F.step < self.warm].copy()
        head["ck"] = [cohort_key(a, m) for a, m in zip(head.age.fillna(65), head.is_male)]
        pool = head[head.patient_split == prior_pool] if prior_pool else head
        self.cohort_prior = pool.groupby("ck").sbp.quantile(self.q).to_dict()
        self.global_prior = float(pool.sbp.quantile(self.q))
        return self

    def threshold_for(self, sbp_history, age, is_male):
        n = int(pd.Series(sbp_history).notna().sum())
        ck = cohort_key(age, is_male)
        cohort = float(self.cohort_prior.get(ck, self.global_prior))
        if n >= 5:
            personal = float(pd.Series(sbp_history).head(self.warm).quantile(self.q))
            w = n / (n + self.k)                       # shrinkage: trust grows with history
        else:
            personal, w = cohort, 0.0
        blend = w * personal + (1 - w) * cohort
        thr = float(apply_caps([blend])[0])
        assert thr < EMERGENCY_FLOOR_MMHG, "offset breached the emergency floor"
        return dict(threshold=round(thr, 1), offset=round(thr - POPULATION_THRESHOLD_MMHG, 1),
                    cohort_key=ck, cohort=round(cohort, 1), personal=round(personal, 1),
                    n_warm=n, shrinkage_w=round(w, 3),
                    capped=abs(blend - thr) > 1e-6)

    def transform(self, F):
        rows = []
        for sid, g in F[F.step < self.warm].groupby("series_id", sort=False):
            if int(g.sbp.notna().sum()) < 5:
                continue
            r = self.threshold_for(g.sbp, float(g.age.fillna(65).iloc[0]), int(g.is_male.iloc[0]))
            rows.append(dict(series_id=sid, patient_split=g.patient_split.iloc[0], **r))
        return pd.DataFrame(rows)


def observed_band(F, warm, q=0.90):
    """What each patient's band actually turned out to be, after the warm-up window."""
    return (F[F.step >= warm].groupby("series_id").sbp.quantile(q)
            .rename("actual").reset_index())


# grid over the blend's own hyper-parameters, searched on `fit` patients only
grid_rows = []
for warm in (24, 36, 48, 72):
    for k in (10, 30, 60):
        for q in (.85, .90, .95):
            om = OffsetModel(warm, k, q).fit(F)
            O = om.transform(F).merge(observed_band(F, warm), on="series_id").dropna(subset=["actual"])
            fit_part = O[O.patient_split == "fit"]
            if len(fit_part) < 20:
                continue
            grid_rows.append(dict(warm=warm, k=k, q=q, n=len(fit_part),
                                  mae_fit=mean_absolute_error(fit_part.actual, fit_part.threshold)))

GRID = pd.DataFrame(grid_rows).sort_values("mae_fit")
display(GRID.head(8).round(3))

BLEND = OffsetModel(int(GRID.iloc[0].warm), float(GRID.iloc[0].k), float(GRID.iloc[0].q)).fit(F)
print(f"blend selected on fit patients: warm={BLEND.warm}, k={BLEND.k}, q={BLEND.q}")

OFF = BLEND.transform(F)
assert OFF.threshold.max() < EMERGENCY_FLOOR_MMHG
print(f"{len(OFF)} patients | {int(OFF.capped.sum())} bound by a governance cap")
print(f"max personalised threshold {OFF.threshold.max():.1f} < emergency floor "
      f"{EMERGENCY_FLOOR_MMHG:.0f} mmHg - invariant holds")

## The learned training frame

In [ ]:
OFFSET_TARGET_Q = 0.90
OFFSET_FUTURE = 30
OFFSET_CUTS = (20, 32, 48, 72, 100, 140)


CLIN_KEYS = (CONDITIONS + ["med_" + m for m in MED_CLASSES]
             + ["n_medications", "n_antihypertensive"])


def clinical_context(src=None):
    """Condition / medication / adherence context as a flat, zero-filled dict.

    Model 2 sets a PERSONAL blood-pressure threshold. Whether a patient carries heart
    failure, takes a rate-limiting agent, or routinely misses antihypertensives is
    exactly the information that should move that threshold - and the first version of
    this model ignored all of it, keying only on age, sex and cohort.
    """
    out = {("clin_" + k): 0.0 for k in CLIN_KEYS}
    out["clin_adherence"] = 1.0
    if src is None:
        return out
    for k in CLIN_KEYS:
        if k in src:
            v = pd.Series(src[k]).dropna()
            if len(v):
                out["clin_" + k] = float(v.iloc[-1])
    if "took_all_meds" in src:
        v = pd.Series(src["took_all_meds"]).dropna()
        if len(v):
            out["clin_adherence"] = float(v.tail(30).mean())
    return out


def offset_features(sbp_hist, age, is_male, cohort, clin=None):
    """~17 statistics of a patient's own history. Named, ordered, individually explainable."""
    h = pd.Series(sbp_hist).dropna().astype(float)
    t20, t10 = h.tail(20), h.tail(10)
    slope = float(np.polyfit(np.arange(len(t20)), t20.values, 1)[0]) if len(t20) >= 5 else 0.0
    q25, q50, q75, q90 = (float(h.quantile(x)) for x in (.25, .50, .75, .90))
    return dict(hist_n=float(len(h)), hist_mean=float(h.mean()),
                hist_sd=float(h.std(ddof=1) or 0.0),
                hist_q25=q25, hist_q50=q50, hist_q75=q75, hist_q90=q90, hist_iqr=q75 - q25,
                hist_min=float(h.min()), hist_max=float(h.max()),
                recent_mean10=float(t10.mean()), recent_sd10=float(t10.std(ddof=1) or 0.0),
                recent_slope20=slope, level_vs_cohort=float(q90 - cohort),
                age=float(age), is_male=float(is_male), cohort_prior=float(cohort),
                **(clin if clin is not None else clinical_context()))


OFFSET_FEATURES = list(offset_features([120, 130, 125], 65, 1, 140.0).keys())


def build_offset_frame(F, model):
    rows = []
    for sid, g in F.sort_values(["series_id", "step"]).groupby("series_id", sort=False):
        s = g.sbp.reset_index(drop=True)
        age = float(g.age.iloc[0]) if pd.notna(g.age.iloc[0]) else 65.0
        male = int(g.is_male.iloc[0])
        coh = float(model.cohort_prior.get(cohort_key(age, male), model.global_prior))
        for c in OFFSET_CUTS:
            if len(s) < c + 10:
                continue
            fut = s.iloc[c:c + OFFSET_FUTURE]
            if fut.notna().sum() < 8 or s.iloc[:c].notna().sum() < 5:
                continue
            rows.append(dict(series_id=sid, patient_split=g.patient_split.iloc[0], cut=c,
                             y=float(fut.quantile(OFFSET_TARGET_Q)),
                             # clinical context read only from history BEFORE the cut
                             **offset_features(s.iloc[:c], age, male, coh,
                                               clin=clinical_context(g.iloc[:c]))))
    return pd.DataFrame(rows)


OFR = build_offset_frame(F, BLEND)

# patient-disjoint everywhere: train / conformal-calibration / model-selection all come out
# of the `fit` patients; `holdout` patients are never touched until the report.
_h = OFR.series_id.map(lambda s: hash_bucket(s, salt="offset-split"))
OFR["offset_split"] = np.where(OFR.patient_split == "holdout", "holdout",
                               np.where(_h < 20, "calib",
                                        np.where(_h < 40, "select", "train")))

print(f"offset frame: {len(OFR):,} rows | {OFR.series_id.nunique()} patients | "
      f"{len(OFFSET_FEATURES)} features")
display(OFR.offset_split.value_counts().rename("rows").to_frame()
        .assign(patients=OFR.groupby("offset_split").series_id.nunique()))

## Candidates, conformal calibration, and selection

In [ ]:
def pinball_rows(y, p, q=OFFSET_TARGET_Q):
    d = np.asarray(y, float) - np.asarray(p, float)
    return np.maximum(q * d, (q - 1) * d)


def pinball(y, p, q=OFFSET_TARGET_Q):
    return float(np.mean(pinball_rows(y, p, q)))


class QuantileForest(BaseEstimator, RegressorMixin):
    """Empirical quantile across the trees, rather than their mean.

    A forest already carries a predictive distribution; averaging it away and then calling
    the result a threshold is the mistake this wrapper avoids.
    """

    def __init__(self, base=None, q=OFFSET_TARGET_Q):
        self.base, self.q = base, q

    def fit(self, X, y):
        self.base_ = clone(self.base).fit(X, y)
        return self

    def predict(self, X):
        P = np.column_stack([t.predict(X) for t in self.base_.estimators_])
        return np.quantile(P, self.q, axis=1)


class KNNQuantile(BaseEstimator, RegressorMixin):
    """Quantile of the k nearest training patients' outcomes - no functional form at all."""

    def __init__(self, k=25, q=OFFSET_TARGET_Q):
        self.k, self.q = k, q

    def fit(self, X, y):
        X = np.asarray(X, float)
        self.mu_, self.sd_ = X.mean(0), X.std(0) + 1e-9
        self.X_ = (X - self.mu_) / self.sd_
        self.y_ = np.asarray(y, float)
        self.nn_ = NearestNeighbors(n_neighbors=min(self.k, len(self.y_))).fit(self.X_)
        return self

    def predict(self, X):
        Z = (np.asarray(X, float) - self.mu_) / self.sd_
        idx = self.nn_.kneighbors(Z, return_distance=False)
        return np.array([np.quantile(self.y_[i], self.q) for i in idx])


class EmpiricalBayesBlend(BaseEstimator, RegressorMixin):
    """The shrinkage blend with the pooling strength LEARNED instead of hand-set.

    k = sigma^2 / tau^2 comes from the decomposition of between- and within-patient
    variance: the standard empirical-Bayes / random-intercept result. This is the
    principled version of Model 2's blend, and partial pooling is a story clinicians
    already accept.
    """

    def __init__(self, n_col=0, personal_col=6, cohort_col=16):
        self.n_col, self.personal_col, self.cohort_col = n_col, personal_col, cohort_col

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        personal, cohort = X[:, self.personal_col], X[:, self.cohort_col]
        tau2 = max(float(np.var(personal - cohort, ddof=1)), 1e-6)
        sigma2 = max(float(np.var(y - personal, ddof=1)), 1e-6)
        self.k_ = float(np.clip(sigma2 / tau2, 1.0, 400.0))
        self.bias_ = float(np.median(y - personal))
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        n, personal, cohort = X[:, self.n_col], X[:, self.personal_col], X[:, self.cohort_col]
        w = n / (n + self.k_)
        return w * personal + (1 - w) * cohort + self.bias_


def lin(m):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("model", m)])


_ix = {c: i for i, c in enumerate(OFFSET_FEATURES)}

OFFSET_CANDIDATES = {
    # name: (estimator, family, native target, serving cost)
    "linear regression": (lin(LinearRegression()), "linear", "mean", "low"),
    "ridge": (lin(Ridge(alpha=10.0)), "linear", "mean", "low"),
    "elastic net": (lin(ElasticNet(alpha=.05, l1_ratio=.5, random_state=SEED)),
                    "linear", "mean", "low"),
    "huber (robust)": (lin(HuberRegressor(epsilon=1.35, max_iter=500)), "linear", "mean", "low"),
    "kNN (mean)": (lin(KNeighborsRegressor(n_neighbors=25, weights="distance")),
                   "kernel", "mean", "low"),
    "MLP (2x32)": (lin(MLPRegressor(hidden_layer_sizes=(32, 32), max_iter=800,
                                    early_stopping=True, random_state=SEED)),
                   "neural", "mean", "medium"),
    "random forest (mean)": (RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                   n_jobs=-1, random_state=SEED),
                             "trees", "mean", "medium"),
    "extra trees (mean)": (ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5,
                                               n_jobs=-1, random_state=SEED),
                           "trees", "mean", "medium"),
    "GBM (pinball q90)": (GradientBoostingRegressor(loss="quantile", alpha=OFFSET_TARGET_Q,
                                                    n_estimators=300, learning_rate=.05,
                                                    random_state=SEED),
                          "trees", "quantile", "medium"),
    "quantile forest": (QuantileForest(RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                             n_jobs=-1, random_state=SEED)),
                        "trees", "quantile", "medium"),
    "kNN quantile": (KNNQuantile(k=25), "kernel", "quantile", "low"),
    "linear quantile": (lin(QuantileRegressor(quantile=OFFSET_TARGET_Q, alpha=1e-3,
                                              solver="highs")),
                        "linear", "quantile", "low"),
    "empirical-Bayes blend": (EmpiricalBayesBlend(n_col=_ix["hist_n"],
                                                  personal_col=_ix["hist_q90"],
                                                  cohort_col=_ix["cohort_prior"]),
                              "shrinkage", "quantile", "low"),
}

tr_o = OFR[OFR.offset_split == "train"]
ca_o = OFR[OFR.offset_split == "calib"]
se_o = OFR[OFR.offset_split == "select"]
ho_o = OFR[OFR.offset_split == "holdout"]
Xtr, ytr = tr_o[OFFSET_FEATURES].values, tr_o.y.values

rows, fitted_offsets = [], {}
for name, (est, family, native, cost) in OFFSET_CANDIDATES.items():
    try:
        m = clone(est).fit(Xtr, ytr)
    except Exception as e:
        rows.append(dict(model=name, family=family, note=f"{type(e).__name__}"))
        continue
    # conformal shift: the additive correction achieving q90 coverage on unseen calibration rows
    resid = ca_o.y.values - m.predict(ca_o[OFFSET_FEATURES].values)
    shift = float(np.quantile(resid, OFFSET_TARGET_Q)) if native == "mean" else \
        float(np.quantile(resid, OFFSET_TARGET_Q)) * 0.0
    fitted_offsets[name] = (m, shift)

    def score(part):
        p = apply_caps(m.predict(part[OFFSET_FEATURES].values) + shift)
        return dict(pinball=pinball(part.y.values, p),
                    MAE=mean_absolute_error(part.y.values, p),
                    coverage=float(np.mean(part.y.values <= p)),
                    within_10=float(np.mean(np.abs(part.y.values - p) <= 10)))

    rows.append(dict(model=name, family=family, native=native, cost=cost,
                     conformal_shift=round(shift, 2),
                     **{f"select_{k}": round(v, 3) for k, v in score(se_o).items()},
                     **{f"holdout_{k}": round(v, 3) for k, v in score(ho_o).items()}))

# the reference candidates the learned models must beat: the blend and its own ingredients
ref_rows = []
ho_ref = OFF[OFF.patient_split == "holdout"].merge(observed_band(F, BLEND.warm), on="series_id")
ho_ref = ho_ref.dropna(subset=["actual"])
for label, pred in [("capped blend (Model 2 baseline)", ho_ref.threshold.values),
                    ("personal q90 only", apply_caps(ho_ref.personal.values)),
                    ("cohort prior only", apply_caps(ho_ref.cohort.values)),
                    ("population constant 140", np.full(len(ho_ref), POPULATION_THRESHOLD_MMHG))]:
    ref_rows.append(dict(model=label, family="reference", native="quantile", cost="low",
                         holdout_pinball=round(pinball(ho_ref.actual.values, pred), 3),
                         holdout_MAE=round(mean_absolute_error(ho_ref.actual.values, pred), 3),
                         holdout_coverage=round(float(np.mean(ho_ref.actual.values <= pred)), 3),
                         holdout_within_10=round(float(np.mean(
                             np.abs(ho_ref.actual.values - pred) <= 10)), 3)))

OFFSET_BOARD = pd.concat([pd.DataFrame(rows), pd.DataFrame(ref_rows)], ignore_index=True)
display(OFFSET_BOARD.sort_values("holdout_pinball")
        [["model", "family", "native", "cost", "conformal_shift", "select_pinball",
          "holdout_pinball", "holdout_MAE", "holdout_coverage", "holdout_within_10"]])

## The promotion decision

In [ ]:
learned = OFFSET_BOARD[(OFFSET_BOARD.family != "reference")
                       & OFFSET_BOARD.select_pinball.notna()]
best_learned = learned.sort_values("select_pinball").iloc[0]        # chosen on `select`, not holdout
best_ref = OFFSET_BOARD[OFFSET_BOARD.family == "reference"].sort_values("holdout_pinball").iloc[0]

m_best, shift_best = fitted_offsets[best_learned.model]
p_learned = apply_caps(m_best.predict(ho_o[OFFSET_FEATURES].values) + shift_best)
err_learned = pinball_rows(ho_o.y.values, p_learned)

# score the winning reference on the same rows so the comparison is paired
coh_ho = ho_o.cohort_prior.values
ref_pred = {"capped blend (Model 2 baseline)": ho_o.hist_n.values / (ho_o.hist_n.values + BLEND.k)
            * ho_o.hist_q90.values + (1 - ho_o.hist_n.values / (ho_o.hist_n.values + BLEND.k))
            * coh_ho,
            "personal q90 only": ho_o.hist_q90.values,
            "cohort prior only": coh_ho,
            "population constant 140": np.full(len(ho_o), POPULATION_THRESHOLD_MMHG)}
err_ref = pinball_rows(ho_o.y.values, apply_caps(ref_pred[best_ref.model]))

delta, (lo, hi), _ = paired_delta(err_learned, err_ref)
print(f"best learned offset : {best_learned.model} (selected on `select`, "
      f"holdout pinball {best_learned.holdout_pinball})")
print(f"best reference      : {best_ref.model} (holdout pinball {best_ref.holdout_pinball})")
print(f"paired delta (learned - reference): {delta:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

if hi < 0:
    OFFSET_CHOICE, OFFSET_KIND = best_learned.model, "learned"
    print(f"-> PROMOTE the learned offset: {OFFSET_CHOICE}")
else:
    OFFSET_CHOICE, OFFSET_KIND = best_ref.model, "reference"
    print(f"-> KEEP the reference: {OFFSET_CHOICE} (the learned model does not clear the interval)")

## Offset diagnostics and safety invariant

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
b = OFFSET_BOARD.dropna(subset=["holdout_pinball"]).sort_values("holdout_pinball").head(12)
colors = ["#c44e52" if f == "reference" else "#4c72b0" for f in b.family]
ax[0].barh(b.model[::-1], b.holdout_pinball[::-1], color=colors[::-1])
ax[0].set(title="Offset candidates - holdout pinball loss (lower is better)", xlabel="pinball")

ax[1].scatter(ho_o.y.values, p_learned, s=18, alpha=.5)
lims = [ho_o.y.min() - 5, ho_o.y.max() + 5]
ax[1].plot(lims, lims, "k--")
ax[1].axhline(POPULATION_THRESHOLD_MMHG, c="crimson", ls=":", label="population 140")
ax[1].axhline(EMERGENCY_FLOOR_MMHG, c="black", label="emergency floor 180")
ax[1].set(title="Predicted threshold vs observed band", xlabel="actual q90", ylabel="predicted")
ax[1].legend(fontsize=8)

sns.histplot(OFF.offset, bins=30, ax=ax[2], color="seagreen")
ax[2].axvline(OFFSET_CAP_LOOSEN, c="crimson", ls="--", label="loosen cap +15")
ax[2].axvline(-OFFSET_CAP_TIGHTEN, c="darkorange", ls="--", label="tighten cap -25")
ax[2].set(title="Learned offsets, with the caps binding", xlabel="offset from 140 mmHg")
ax[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

assert float(np.max(p_learned)) < EMERGENCY_FLOOR_MMHG
assert float(OFF.threshold.max()) < EMERGENCY_FLOOR_MMHG
print(f"SAFETY INVARIANT: max personalised threshold over every candidate = "
      f"{max(float(np.max(p_learned)), float(OFF.threshold.max())):.1f} mmHg "
      f"< emergency floor {EMERGENCY_FLOOR_MMHG:.0f} mmHg")
print(f"caps bind for {OFF.capped.mean():.1%} of patients")

# 12. Model 03  - The early-warning detector

In [ ]:
SESSIONS_PER_WEEK = float(7.0 / panel.days_since_last.median())


def build_event_frame(F):
    """Future-window event label + the causal detector score matrix."""
    thr = (F[F.split == "train"].groupby("series_id").sbp
           .quantile(EVENT_QUANTILE).rename("event_threshold"))
    D = F.merge(thr, on="series_id", how="left").sort_values(["series_id", "step"])
    D = D.reset_index(drop=True)

    fut_cols = []
    for k in range(1, WARN_WINDOW + 1):
        c = f"_fut{k}"
        D[c] = D.groupby("series_id").sbp.shift(-k)
        fut_cols.append(c)

    fut = D[fut_cols].values.astype(float)
    thr_v = D.event_threshold.values.astype(float)[:, None]
    breach = np.where(np.isfinite(fut) & np.isfinite(thr_v), fut >= thr_v, False)
    all_nan = ~np.isfinite(fut).any(axis=1) | ~np.isfinite(thr_v.ravel())

    D["event_next"] = np.where(all_nan, np.nan, breach.any(axis=1).astype(float))
    first_k = np.where(breach.any(axis=1), breach.argmax(axis=1) + 1, np.nan)
    D["event_step"] = D.step.values + first_k
    D = D.drop(columns=fut_cols)

    # simple causal detectors, computed straight from the feature matrix
    D["d_personal_z"] = D["sbp_z"].abs()
    D["d_ewma_gap"] = (D["sbp_lag1"] - D["sbp_ewm0.1"]).abs() / D["sbp_base_std"].replace(0, np.nan)
    D["d_slope7"] = D["sbp_slope7"]
    D["d_mean7_excess"] = D["sbp_mean7"] - D["sbp_base_mean"]
    D["d_fixed_threshold"] = D["sbp_lag1"]       # the rule-engine analogue: today's level
    return D


D = build_event_frame(F)
base_rate = float(D[D.split == "test"].event_next.mean())
print(f"event: SBP >= personal q{EVENT_QUANTILE:.2f} within {WARN_WINDOW} sessions "
      f"(~{WARN_WINDOW / SESSIONS_PER_WEEK * 7:.0f} days)")
print(f"test-split base rate: {base_rate:.2%} | sessions per week: {SESSIONS_PER_WEEK:.2f}")
display(D.groupby("split").event_next.agg(["mean", "count"]).round(4))

## nspervised Detector Families

In [ ]:
# Diagnosed conditions, medication classes and demographics never vary inside a patient,
# so they say nothing about a CHANGE of state - but a covariance or distance detector
# will treat a rare condition as an outlier and flag that patient permanently. Against a
# 5% alert budget that is a pure false-positive generator. The forecaster keeps them; the
# detector matrix does not.
_nu = D.groupby("series_id")[FEATURES].nunique(dropna=True).max()
STATIC_IN_PATIENT = sorted(_nu[_nu <= 1].index)
dense_cols = [c for c in FEATURES
              if D[c].notna().mean() > .5 and c not in STATIC_IN_PATIENT]
print(f"detector matrix: {len(dense_cols)} columns "
      f"({len(STATIC_IN_PATIENT)} within-patient constants excluded)")
if STATIC_IN_PATIENT:
    print("  excluded:", STATIC_IN_PATIENT[:10],
          "..." if len(STATIC_IN_PATIENT) > 10 else "")
IMPUTER = SimpleImputer(strategy="median").fit(D[D.split == "train"][dense_cols])
X_train = IMPUTER.transform(D[D.split == "train"][dense_cols])
X_all = IMPUTER.transform(D[dense_cols])
X_sub = X_train[np.random.default_rng(SEED).permutation(len(X_train))[:KERNEL_FIT_ROWS]]
print(f"detector feature matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} dense features")


def per_patient_running(df, col, fn):
    """Apply a causal running statistic within each patient, in step order."""
    out = np.full(len(df), np.nan)
    pos = {sid: idx for sid, idx in df.groupby("series_id").indices.items()}
    vals = df[col].values
    steps = df.step.values
    for sid, idx in pos.items():
        order = idx[np.argsort(steps[idx])]
        out[order] = fn(vals[order])
    return out


def cusum(x, k=0.5):
    """One-sided CUSUM on standardised values: accumulates evidence of an upward shift."""
    x = np.asarray(x, float)
    head = x[:20]
    m = np.nanmean(head) if np.isfinite(head).any() else np.nanmean(x)
    s = np.nanstd(head) or (np.nanstd(x) or 1.0)
    z = (x - m) / (s if s > 0 else 1.0)
    out, acc = np.zeros(len(x)), 0.0
    for i, v in enumerate(z):
        acc = 0.0 if not np.isfinite(v) else max(0.0, acc + v - k)
        out[i] = acc
    return out


def page_hinkley(x, delta=0.05):
    """Cumulative departure from the running mean, minus a tolerance."""
    x = np.asarray(x, float)
    out, mu, n, mT, MT = np.zeros(len(x)), 0.0, 0, 0.0, 0.0
    for i, v in enumerate(x):
        if not np.isfinite(v):
            out[i] = MT - mT if n else 0.0
            continue
        n += 1
        mu += (v - mu) / n
        mT += v - mu - delta
        MT = min(MT, mT) if n > 1 else mT
        out[i] = mT - MT
    return out


D["d_cusum"] = per_patient_running(D, "sbp_lag1", cusum)
D["d_page_hinkley"] = per_patient_running(D, "sbp_lag1", page_hinkley)

DETECTOR_FITTED = {}

DETECTOR_FITTED["isoforest"] = IsolationForest(n_estimators=200, random_state=SEED,
                                               n_jobs=-1).fit(X_train)
D["d_isoforest"] = -DETECTOR_FITTED["isoforest"].score_samples(X_all)

DETECTOR_FITTED["lof"] = LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1).fit(X_sub)
D["d_lof"] = -DETECTOR_FITTED["lof"].score_samples(X_all)

DETECTOR_FITTED["elliptic"] = EllipticEnvelope(support_fraction=0.9, contamination=0.05,
                                               random_state=SEED).fit(X_sub)
D["d_elliptic"] = -DETECTOR_FITTED["elliptic"].score_samples(X_all)

DETECTOR_FITTED["ocsvm"] = OneClassSVM(nu=.05, gamma="scale").fit(X_sub[:8000])
D["d_ocsvm"] = -DETECTOR_FITTED["ocsvm"].decision_function(X_all)

DETECTOR_FITTED["pca"] = PCA(n_components=min(10, X_train.shape[1]), random_state=SEED).fit(X_train)
_p = DETECTOR_FITTED["pca"]
D["d_pca_recon"] = np.sqrt(((X_all - _p.inverse_transform(_p.transform(X_all))) ** 2).sum(1))

DETECTOR_FITTED["gmm"] = GaussianMixture(n_components=4, covariance_type="diag",
                                         random_state=SEED, reg_covar=1e-4).fit(X_train)
D["d_gmm_nll"] = -DETECTOR_FITTED["gmm"].score_samples(X_all)

# forecast-based: reuse the model selected in §7, standardised against each patient's own
# train distribution so the score means "unusually high FOR THIS PATIENT"
fc = FORECASTERS[("sbp", HORIZONS[0])]
pred_all = fc.predict(D, lag1=D["sbp_lag1"].values, series_id=D.series_id.values)
mu = D[D.split == "train"].groupby("series_id").sbp.mean()
sd = D[D.split == "train"].groupby("series_id").sbp.std().replace(0, np.nan)
D["d_forecast_level"] = (pred_all - D.series_id.map(mu)) / D.series_id.map(sd)
D["d_forecast_breach"] = pred_all - D.event_threshold      # margin over the event threshold

DETECTORS = ["d_personal_z", "d_ewma_gap", "d_slope7", "d_mean7_excess", "d_fixed_threshold",
             "d_isoforest", "d_lof", "d_elliptic", "d_ocsvm", "d_pca_recon", "d_gmm_nll",
             "d_cusum", "d_page_hinkley", "d_forecast_level", "d_forecast_breach"]

DETECTOR_FAMILY = {
    "d_personal_z": "control chart", "d_ewma_gap": "control chart",
    "d_slope7": "trend", "d_mean7_excess": "trend",
    "d_fixed_threshold": "reference (rule engine)",
    "d_isoforest": "isolation", "d_lof": "neighbourhood", "d_elliptic": "covariance",
    "d_ocsvm": "boundary", "d_pca_recon": "subspace", "d_gmm_nll": "density",
    "d_cusum": "change detection", "d_page_hinkley": "change detection",
    "d_forecast_level": "forecast", "d_forecast_breach": "forecast",
}
print(f"{len(DETECTORS)} detectors across "
      f"{len(set(DETECTOR_FAMILY[d] for d in DETECTORS))} families")

## Detector Tning

In [ ]:
D_val = D[(D.split == "val") & D.event_next.notna()]


def ap_lift(df, score):
    s = pd.DataFrame({"y": df.event_next.values, "s": np.asarray(score, float)}).dropna()
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 50 or s.y.nunique() < 2:
        return np.nan
    return average_precision_score(s.y, s.s) / max(float(s.y.mean()), 1e-9)


DETECTOR_GRID = {
    "d_isoforest": [dict(n_estimators=n, max_samples=ms)
                    for n in (100, 300) for ms in (256, 1024)],
    "d_lof": [dict(n_neighbors=k) for k in (10, 20, 50)],
    "d_ocsvm": [dict(nu=nu, gamma=g) for nu in (.02, .05, .1) for g in ("scale", "auto")],
    "d_gmm_nll": [dict(n_components=k) for k in (2, 4, 8)],
    "d_pca_recon": [dict(n_components=k) for k in (5, 10, 20)],
    "d_cusum": [dict(k=k) for k in (0.25, 0.5, 1.0)],
    "d_page_hinkley": [dict(delta=d) for d in (0.01, 0.05, 0.2)],
}


def refit_detector(col, params):
    """Refit one detector family with candidate parameters; return scores over all rows."""
    if col == "d_isoforest":
        m = IsolationForest(random_state=SEED, n_jobs=-1, **params).fit(X_train)
        return -m.score_samples(X_all), m
    if col == "d_lof":
        m = LocalOutlierFactor(novelty=True, n_jobs=-1, **params).fit(X_sub)
        return -m.score_samples(X_all), m
    if col == "d_ocsvm":
        m = OneClassSVM(**params).fit(X_sub[:8000])
        return -m.decision_function(X_all), m
    if col == "d_gmm_nll":
        m = GaussianMixture(covariance_type="diag", random_state=SEED, reg_covar=1e-4,
                            **params).fit(X_train)
        return -m.score_samples(X_all), m
    if col == "d_pca_recon":
        k = min(params["n_components"], X_train.shape[1])
        m = PCA(n_components=k, random_state=SEED).fit(X_train)
        return np.sqrt(((X_all - m.inverse_transform(m.transform(X_all))) ** 2).sum(1)), m
    if col == "d_cusum":
        return per_patient_running(D, "sbp_lag1", lambda x: cusum(x, **params)), None
    if col == "d_page_hinkley":
        return per_patient_running(D, "sbp_lag1", lambda x: page_hinkley(x, **params)), None
    raise ValueError(col)


tune_rows = []
BEST_DETECTOR_PARAMS = {}
for col, grid in DETECTOR_GRID.items():
    base = ap_lift(D_val, D.loc[D_val.index, col].values)
    best, best_p, best_scores, best_obj = base, None, None, None
    for params in grid:
        try:
            scores, obj = refit_detector(col, params)
        except Exception:
            continue
        lift = ap_lift(D_val, scores[D_val.index])
        if np.isfinite(lift) and lift > (best if np.isfinite(best) else -np.inf):
            best, best_p, best_scores, best_obj = lift, params, scores, obj
    if best_p is not None:
        D[col] = best_scores
        BEST_DETECTOR_PARAMS[col] = best_p
        if best_obj is not None:
            DETECTOR_FITTED[col.replace("d_", "").replace("_nll", "").replace("_recon", "")] = best_obj
    tune_rows.append(dict(detector=col, val_lift_default=round(base, 3),
                          val_lift_tuned=round(best, 3), params=best_p or "default"))

display(pd.DataFrame(tune_rows))

## Detector Fsion - The cheapest ensemble availabe

In [ ]:
FUSION_LEGS = [d for d in DETECTORS if d != "d_fixed_threshold"]
FUSE_STATS = {}
for c in FUSION_LEGS:
    v = D.loc[D.split == "train", c].replace([np.inf, -np.inf], np.nan).dropna()
    if len(v) >= 50:
        FUSE_STATS[c] = (float(v.mean()), float(v.std() or 1.0))

Z = pd.DataFrame({c: (D[c] - m) / (s if s > 0 else 1.0) for c, (m, s) in FUSE_STATS.items()},
                 index=D.index).clip(-8, 8)      # one exploding leg must not dominate

D["d_fuse_mean"] = Z.mean(axis=1)
D["d_fuse_median"] = Z.median(axis=1)
D["d_fuse_max"] = Z.max(axis=1)                   # "any family is alarmed" - high recall
trimmed = np.sort(Z.values, axis=1)
D["d_fuse_trimmed"] = np.nanmean(trimmed[:, 1:-1], axis=1)

# weighted fusion: weights from validation lift over the base rate, never from test
FUSE_WEIGHTS = {}
for c in FUSE_STATS:
    s = D_val[[c, "event_next"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) > 50 and s.event_next.nunique() > 1:
        FUSE_WEIGHTS[c] = max(average_precision_score(s.event_next, s[c])
                              - float(s.event_next.mean()), 0.0)

if sum(FUSE_WEIGHTS.values()) > 0:
    wv = pd.Series(FUSE_WEIGHTS)
    wv = wv / wv.sum()
    D["d_fuse_weighted"] = (Z[list(wv.index)] * wv.values).sum(axis=1)
    print("fusion weights (validation lift over base rate, normalised):")
    print(wv.sort_values(ascending=False).round(3).head(8).to_string())
else:
    D["d_fuse_weighted"] = D["d_fuse_mean"]
    wv = pd.Series(dtype=float)
    print("validation split too thin for weighted fusion - using the unweighted mean")

FUSION_DETECTORS = ["d_fuse_mean", "d_fuse_median", "d_fuse_max", "d_fuse_trimmed",
                    "d_fuse_weighted"]
DETECTORS += FUSION_DETECTORS
DETECTOR_FAMILY.update({c: "fusion (ensemble)" for c in FUSION_DETECTORS})
print(f"\n{len(FUSION_LEGS)} legs fused -> {len(DETECTORS)} detectors total")

## Scorecard and selection

In [ ]:
def detector_scorecard(df, detectors, budgets=(1, 2, 5, 10)):
    rows = []
    for col in detectors:
        s = df[[col, "event_next", "event_step", "step", "days_since_last"]].copy()
        s[col] = s[col].replace([np.inf, -np.inf], np.nan)
        s = s.dropna(subset=[col, "event_next"])
        if len(s) < 50 or s.event_next.nunique() < 2:
            continue
        base = float(s.event_next.mean())
        auc = roc_auc_score(s.event_next, s[col])
        ap = average_precision_score(s.event_next, s[col])
        for b in budgets:
            cut = np.percentile(s[col], 100 - b)
            flag = (s[col] >= cut).astype(int)
            tp = int(((flag == 1) & (s.event_next == 1)).sum())
            fp = int(((flag == 1) & (s.event_next == 0)).sum())
            fn = int(((flag == 0) & (s.event_next == 1)).sum())
            prec = tp / (tp + fp) if tp + fp else np.nan
            hit = s[(flag == 1) & (s.event_next == 1)]
            lead_sessions = hit.event_step - hit.step
            lead_days = lead_sessions * hit.days_since_last.median() if len(hit) else pd.Series(dtype=float)
            rows.append(dict(
                detector=col, family=DETECTOR_FAMILY.get(col, "?"), budget_pct=b,
                base_rate=round(base, 4),
                precision=round(prec, 3) if np.isfinite(prec) else np.nan,
                lift=round(prec / base, 2) if base and np.isfinite(prec) else np.nan,
                recall=round(tp / (tp + fn), 3) if tp + fn else np.nan,
                auc_roc=round(auc, 3), auc_pr=round(ap, 3), auc_pr_lift=round(ap / base, 2),
                lead_days_median=round(float(lead_days.median()), 2) if len(hit) else np.nan,
                lead_days_p90=round(float(lead_days.quantile(.90)), 2) if len(hit) else np.nan,
                alerts_per_patient_week=round(b / 100 * SESSIONS_PER_WEEK, 2)))
    return pd.DataFrame(rows)


D_test = D[(D.split == "test") & D.event_next.notna()]
D_val_ev = D[(D.split == "val") & D.event_next.notna()]

SCORECARD_VAL = detector_scorecard(D_val_ev, DETECTORS, budgets=(ALERT_BUDGET_PCT,))
SCORECARD = detector_scorecard(D_test, DETECTORS)

at_budget = SCORECARD[SCORECARD.budget_pct == ALERT_BUDGET_PCT].sort_values("auc_pr",
                                                                            ascending=False)
display(at_budget[["detector", "family", "precision", "lift", "recall", "auc_roc", "auc_pr",
                   "auc_pr_lift", "lead_days_median", "alerts_per_patient_week"]])

# selection on VALIDATION, reported on test
sel = SCORECARD_VAL[SCORECARD_VAL.detector != "d_fixed_threshold"].sort_values(
    "auc_pr", ascending=False)
DETECTOR_CHOICE = sel.iloc[0].detector
ref_row = SCORECARD[(SCORECARD.detector == "d_fixed_threshold")
                    & (SCORECARD.budget_pct == ALERT_BUDGET_PCT)]
chosen_row = at_budget[at_budget.detector == DETECTOR_CHOICE].iloc[0]

print(f"\nselected on validation -> {DETECTOR_CHOICE} ({DETECTOR_FAMILY[DETECTOR_CHOICE]})")
print(f"test performance at the {ALERT_BUDGET_PCT:.0f}% budget: "
      f"precision {chosen_row.precision:.3f} (lift {chosen_row.lift:.2f}x), "
      f"recall {chosen_row.recall:.3f}, median lead {chosen_row.lead_days_median:.1f} days")
if len(ref_row):
    r = ref_row.iloc[0]
    print(f"rule-engine reference (today's reading): precision {r.precision:.3f} "
          f"(lift {r.lift:.2f}x), median lead {r.lead_days_median:.1f} days")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
top_det = at_budget.head(8).detector.tolist()
for col in top_det + ["d_fixed_threshold"]:
    s = D_test[[col, "event_next"]].replace([np.inf, -np.inf], np.nan).dropna()
    if s.event_next.nunique() < 2:
        continue
    style = dict(lw=2.5, ls="--", color="k") if col == "d_fixed_threshold" else dict(lw=1.6)
    fpr, tpr, _ = roc_curve(s.event_next, s[col])
    ax[0, 0].plot(fpr, tpr, label=col, **style)
    pr, rc, _ = precision_recall_curve(s.event_next, s[col])
    ax[0, 1].plot(rc, pr, label=col, **style)
ax[0, 0].plot([0, 1], [0, 1], ":", c="grey")
ax[0, 0].set(title="ROC (test)", xlabel="FPR", ylabel="TPR")
ax[0, 0].legend(fontsize=6)
ax[0, 1].axhline(base_rate, ls=":", c="grey", label="base rate")
ax[0, 1].set(title="Precision-recall (test)", xlabel="recall", ylabel="precision")
ax[0, 1].legend(fontsize=6)

b = at_budget.head(12)
sns.barplot(data=b, x="auc_pr_lift", y="detector", hue="family", ax=ax[1, 0], dodge=False)
ax[1, 0].set(title=f"AUC-PR lift over base rate ({ALERT_BUDGET_PCT:.0f}% budget)")
ax[1, 0].legend(fontsize=6)
sns.scatterplot(data=at_budget, x="lead_days_median", y="lift", hue="family", s=70, ax=ax[1, 1])
ax[1, 1].set(title="The trade-off that matters: lift vs lead time",
             xlabel="median lead (days)", ylabel="precision lift")
ax[1, 1].legend(fontsize=6)
plt.tight_layout()
plt.show()

# operating threshold at the alert budget, fitted on validation scores only
DETECTOR_CUT = float(np.percentile(
    D_val_ev[DETECTOR_CHOICE].replace([np.inf, -np.inf], np.nan).dropna(),
    100 - ALERT_BUDGET_PCT))
print(f"operating cut for {DETECTOR_CHOICE}: {DETECTOR_CUT:.4f} "
      f"(the {100 - ALERT_BUDGET_PCT:.0f}th percentile of validation scores)")

# 13. The final combined Model

In [ ]:
def inference_features(history):
    """One feature row describing 'now' for a single patient, from their raw history."""
    g = history.sort_values("ts").copy()
    med_gap = float(g.ts.diff().dt.days.median() or 2)
    ph = g.iloc[[-1]].copy()
    ph["ts"] = g.ts.max() + pd.Timedelta(days=med_gap)
    for c in ("sbp", "dbp", "idwg", "weight", "sbp_drop", "uf_total"):
        if c in ph:
            ph[c] = np.nan
    gg = pd.concat([g, ph], ignore_index=True)
    gg["series_id"] = str(g.patient_id.iloc[0])
    gg["days_since_last"] = gg.ts.diff().dt.days.fillna(2).clip(1, 30)
    gg["step"] = np.arange(len(gg))
    gg["is_weekend"] = (gg.ts.dt.dayofweek >= 5).astype(int)
    for c, default in (("DM", 0), ("is_dm", 0), ("is_male", 0), ("age", 65.0)):
        if c not in gg:
            gg[c] = default
    return build_features_one(gg).iloc[[-1]]


class BPModel:
    """Frozen serving artifact holding all three models. No notebook globals are read."""

    def __init__(self, bundle):
        self.b = bundle

    # ---------- persistence ----------
    def save(self, path):
        import joblib
        joblib.dump(self.b, path)
        return path

    @classmethod
    def load(cls, path):
        import joblib
        return cls(joblib.load(path))

    # ---------- tiering ----------
    def tier(self, n):
        if n < self.b["cold_start_min_readings"]:
            return "cold_start"
        return "bootstrapping" if n < self.b["steady_state_readings"] else "steady"

    # ---------- Model 2 ----------
    def personal_threshold(self, sbp_hist, age, is_male, clin=None):
        b = self.b
        ck = cohort_key(age, is_male)
        cohort = float(b["cohort_prior"].get(ck, b["global_prior"]))
        h = pd.Series(sbp_hist).dropna()
        if len(h) < 5:
            return dict(threshold=round(float(apply_caps([cohort])[0]), 1), source="cohort prior",
                        cohort_key=ck, n=len(h))
        if b["offset_kind"] == "learned":
            _f = offset_features(h, age, is_male, cohort,
                                 clin=clin if clin is not None else clinical_context())
            X = np.array([[_f[k] for k in b["offset_feature_names"]]])
            raw = float(b["offset_model"].predict(X)[0]) + b["offset_shift"]
            source = b["offset_name"]
        else:
            w = len(h) / (len(h) + b["blend_k"])
            raw = w * float(h.head(b["blend_warm"]).quantile(b["blend_q"])) + (1 - w) * cohort
            source = "capped shrinkage blend"
        thr = float(apply_caps([raw])[0])
        assert thr < b["emergency_floor_mmHg"], "offset breached the emergency floor"
        return dict(threshold=round(thr, 1), offset=round(thr - b["population_threshold"], 1),
                    source=source, cohort_key=ck, n=len(h))

    # ---------- Model 4 ----------
    def symptom_risk(self, feat_row):
        """Calibrated probability per symptom per horizon, plus the alert decision.

        `fires` uses the threshold frozen on the validation split during training, not a
        threshold recomputed here - otherwise the alert rate at serving time would drift
        away from the staffing budget the system was sized for.
        """
        b = self.b
        models, cuts = b.get("symptom_models") or {}, b.get("symptom_cuts") or {}
        if not models:
            return None
        X = feat_row[b["features"]]
        out = {}
        for (s, h), est in models.items():
            try:
                p = float(est.predict_proba(X)[:, 1][0])
            except Exception:
                continue
            cut = cuts.get((s, h))
            out.setdefault(f"h{h}", {})[s] = dict(
                p=round(p, 4),
                fires=bool(cut is not None and p >= cut),
                mechanism=b["symptom_mechanism"].get(s, "group"),
                red_flag=s in b["symptom_red_flags"])
        for hk, d in out.items():
            rf = [s for s, v in d.items() if v["red_flag"] and v["fires"]]
            d["_summary"] = dict(any_fires=sorted(s for s, v in d.items()
                                                  if isinstance(v, dict) and v.get("fires")),
                                 red_flag_fires=sorted(rf),
                                 labels_are_synthetic=True)
        return out

    # ---------- Model 3 ----------
    def detector_scores(self, feat_row, sbp_hist):
        """Every causal detector leg, computed for a single row."""
        b = self.b
        s = {}
        s["d_personal_z"] = abs(float(feat_row["sbp_z"].iloc[0]))
        s["d_ewma_gap"] = abs(float(feat_row["sbp_lag1"].iloc[0])
                              - float(feat_row["sbp_ewm0.1"].iloc[0])) / \
            max(float(feat_row["sbp_base_std"].iloc[0]), 1e-6)
        s["d_slope7"] = float(feat_row["sbp_slope7"].iloc[0])
        s["d_mean7_excess"] = float(feat_row["sbp_mean7"].iloc[0]) - \
            float(feat_row["sbp_base_mean"].iloc[0])
        s["d_fixed_threshold"] = float(feat_row["sbp_lag1"].iloc[0])

        X = b["imputer"].transform(feat_row[b["dense_cols"]])
        fits = b["detector_fitted"]
        if "isoforest" in fits:
            s["d_isoforest"] = float(-fits["isoforest"].score_samples(X)[0])
        if "lof" in fits:
            s["d_lof"] = float(-fits["lof"].score_samples(X)[0])
        if "elliptic" in fits:
            s["d_elliptic"] = float(-fits["elliptic"].score_samples(X)[0])
        if "ocsvm" in fits:
            s["d_ocsvm"] = float(-fits["ocsvm"].decision_function(X)[0])
        if "pca" in fits:
            p = fits["pca"]
            s["d_pca_recon"] = float(np.sqrt(((X - p.inverse_transform(p.transform(X))) ** 2).sum()))
        if "gmm" in fits:
            s["d_gmm_nll"] = float(-fits["gmm"].score_samples(X)[0])

        y = np.asarray(pd.Series(sbp_hist).values, float)
        s["d_cusum"] = float(cusum(y, **b["cusum_params"])[-1])
        s["d_page_hinkley"] = float(page_hinkley(y, **b["ph_params"])[-1])
        return s

    def fuse(self, legs, pred_sbp, sbp_hist):
        """Reproduce the fusion transform: train-fitted z-scores, clipped, then combined."""
        b = self.b
        h = pd.Series(sbp_hist).dropna()
        mu, sd = float(h.mean()), float(h.std() or 1.0)
        thr_event = float(h.quantile(b["event_quantile"])) if len(h) >= 10 else np.nan
        legs = dict(legs)
        legs["d_forecast_level"] = (pred_sbp - mu) / (sd if sd > 0 else 1.0)
        legs["d_forecast_breach"] = pred_sbp - thr_event

        choice = b["detector_choice"]
        if not choice.startswith("d_fuse"):
            return float(legs.get(choice, np.nan)), legs

        z = {}
        for c, (m, sdv) in b["fuse_stats"].items():
            if c in legs and np.isfinite(legs[c]):
                z[c] = float(np.clip((legs[c] - m) / (sdv if sdv > 0 else 1.0), -8, 8))
        if not z:
            return np.nan, legs
        v = np.array(list(z.values()))
        if choice == "d_fuse_mean":
            score = float(v.mean())
        elif choice == "d_fuse_median":
            score = float(np.median(v))
        elif choice == "d_fuse_max":
            score = float(v.max())
        elif choice == "d_fuse_trimmed":
            score = float(np.sort(v)[1:-1].mean()) if len(v) >= 4 else float(v.mean())
        else:                                   # d_fuse_weighted
            w = b["fuse_weights"]
            keys = [k for k in z if k in w]
            if not keys or sum(w[k] for k in keys) <= 0:
                score = float(v.mean())
            else:
                ws = np.array([w[k] for k in keys], float)
                ws = ws / ws.sum()
                score = float((np.array([z[k] for k in keys]) * ws).sum())
        return score, legs

    # ---------- the advisory ----------
    def predict(self, history):
        """One patient's raw session history -> a serialisable advisory."""
        b = self.b
        t0 = time.perf_counter()
        g = history.sort_values("ts")
        n_obs = int(g.sbp.notna().sum())
        age = float(g.age.iloc[0]) if "age" in g and pd.notna(g.age.iloc[0]) else 65.0
        is_male = int(g.is_male.iloc[0]) if "is_male" in g else 0
        tier = self.tier(n_obs)

        out = dict(patient_id=str(g.patient_id.iloc[0]),
                   as_of=str(g.ts.max().date()),
                   model_version=b["model_version"],
                   n_observations=n_obs, confidence_tier=tier,
                   provider_visible_only=True,
                   emergency_floor_mmHg=b["emergency_floor_mmHg"])

        out["personalisation"] = self.personal_threshold(
            g.sbp, age, is_male, clin=clinical_context(g))

        if tier == "cold_start":
            out["note"] = (f"fewer than {b['cold_start_min_readings']} readings; "
                           f"cohort threshold only, no forecast issued")
            out["forecast"], out["early_warning"] = {}, None
            out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 1)
            return out

        row = inference_features(g)
        fc = {}
        for (sig, h), model in b["forecasters"].items():
            try:
                p = model.predict(row, lag1=row[f"{sig}_lag1"].values,
                                  hist=g[sig].values, series_id=row.series_id.values)
                fc.setdefault(sig, {})[f"h{h}"] = round(float(np.ravel(p)[0]), 1)
            except Exception:
                fc.setdefault(sig, {})[f"h{h}"] = None
        out["forecast"] = fc
        out["symptom_risk"] = self.symptom_risk(row)

        pred_sbp = fc.get("sbp", {}).get(f"h{b['horizons'][0]}")
        legs = self.detector_scores(row, g.sbp)
        score, all_legs = self.fuse(legs, pred_sbp if pred_sbp is not None else np.nan, g.sbp)
        out["early_warning"] = dict(
            detector=b["detector_choice"], score=None if not np.isfinite(score) else round(score, 4),
            cut=round(b["detector_cut"], 4),
            flag=bool(np.isfinite(score) and score >= b["detector_cut"]),
            warn_window_sessions=b["warn_window"],
            alert_budget_pct=b["alert_budget_pct"])

        if tier == "bootstrapping":
            out["note"] = (f"{n_obs} readings, below the {b['steady_state_readings']}-reading "
                           f"steady state; render a low-confidence badge")
        out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 1)
        return out


# ---- assemble the bundle ----
import time
import joblib

_off_model, _off_shift = (fitted_offsets[OFFSET_CHOICE] if OFFSET_KIND == "learned"
                          else (None, 0.0))

BUNDLE = dict(
    model_version=f"hemobp-bp-{pd.Timestamp.utcnow():%Y%m%dT%H%M%S}",
    horizons=list(HORIZONS), signals=list(SIGNALS),
    # governance
    population_threshold=POPULATION_THRESHOLD_MMHG,
    emergency_floor_mmHg=EMERGENCY_FLOOR_MMHG,
    alert_budget_pct=ALERT_BUDGET_PCT, warn_window=WARN_WINDOW,
    event_quantile=EVENT_QUANTILE,
    cold_start_min_readings=COLD_START_MIN_READINGS,
    steady_state_readings=STEADY_STATE_READINGS,
    # Model 1
    forecaster_name=SERVABLE, forecasters=FORECASTERS,
    # Model 2
    offset_kind=OFFSET_KIND, offset_name=OFFSET_CHOICE,
    offset_model=_off_model, offset_shift=_off_shift,
    offset_feature_names=OFFSET_FEATURES,
    cohort_prior=BLEND.cohort_prior, global_prior=BLEND.global_prior,
    blend_warm=BLEND.warm, blend_k=BLEND.k, blend_q=BLEND.q,
    # Model 3
    detector_choice=DETECTOR_CHOICE, detector_cut=DETECTOR_CUT,
    detector_fitted=DETECTOR_FITTED, dense_cols=dense_cols, imputer=IMPUTER,
    # Model 4 - the symptom head. Without these five entries the frozen artifact
    # forecasts blood pressure only, however well the notebook scored symptoms.
    features=list(FEATURES), symptom_models=SYMPTOM_MODELS, symptom_cuts=SYMPTOM_CUTS,
    symptom_mechanism=dict(SYMPTOM_MECHANISM), symptom_red_flags=list(SYMPTOM_RED_FLAG),
    symptom_labels_synthetic=True,
    fuse_stats=FUSE_STATS, fuse_weights=dict(wv) if len(wv) else {},
    cusum_params=BEST_DETECTOR_PARAMS.get("d_cusum", {"k": 0.5}),
    ph_params=BEST_DETECTOR_PARAMS.get("d_page_hinkley", {"delta": 0.05}),
)

MODEL = BPModel(BUNDLE)
BUNDLE_PATH = os.path.join(ARTIFACT_DIR, "bp_model.joblib")
MODEL.save(BUNDLE_PATH)
print(f"saved {BUNDLE_PATH} ({os.path.getsize(BUNDLE_PATH) / 1e6:.1f} MB)")
print(f"  Model 1 forecaster : {SERVABLE}")
print(f"  Model 4 symptoms   : {len(SYMPTOM_MODELS)} calibrated models "
      f"({len({s for s, _ in SYMPTOM_MODELS})} targets x {len(HORIZONS)} horizons)")
print("  NOTE: symptom heads were trained on SYNTHETIC labels - do not deploy them as "
      "clinical predictions")
print(f"  Model 2 offset     : {OFFSET_CHOICE} ({OFFSET_KIND})")
print(f"  Model 3 detector   : {DETECTOR_CHOICE}")

## ROnd trip, cold trip and train/serve parity

In [ ]:
RELOADED = BPModel.load(BUNDLE_PATH)

probe_id = (F[F.patient_split == "holdout"].groupby("series_id").size()
            .sort_values().index[-1])
hist = panel[panel.patient_id == probe_id].copy()
print(f"probe patient {probe_id}: {len(hist)} sessions, "
      f"{hist.ts.min().date()} to {hist.ts.max().date()}")

a_mem = MODEL.predict(hist)
a_disk = RELOADED.predict(hist)
same = json.dumps({k: v for k, v in a_mem.items() if k != "latency_ms"}, sort_keys=True, default=str) == \
    json.dumps({k: v for k, v in a_disk.items() if k != "latency_ms"}, sort_keys=True, default=str)
print(f"round trip reproduces the advisory exactly: {same}")
assert same, "the reloaded bundle disagrees with the in-memory model"
print(json.dumps(a_disk, indent=2, default=str))

rows = []
for n in [3, 10, 25, 60, len(hist)]:
    if n > len(hist):
        continue
    a = RELOADED.predict(hist.head(n))
    rows.append(dict(n_readings=n, tier=a["confidence_tier"],
                     threshold=a["personalisation"]["threshold"],
                     source=a["personalisation"]["source"],
                     forecast_sbp_h1=a["forecast"].get("sbp", {}).get(f"h{HORIZONS[0]}"),
                     ew_flag=(a["early_warning"] or {}).get("flag"),
                     latency_ms=a["latency_ms"]))
display(pd.DataFrame(rows))

## Batch scoring and train/serve parity

In [ ]:
batch_ids = F[F.patient_split == "holdout"].series_id.unique()[:40]
log_rows = []
t0 = time.perf_counter()
for pid in batch_ids:
    h = panel[panel.patient_id == pid]
    if len(h) < 10:
        continue
    a = RELOADED.predict(h)
    log_rows.append(dict(patient_id=pid, tier=a["confidence_tier"],
                         threshold=a["personalisation"]["threshold"],
                         sbp_h1=a["forecast"].get("sbp", {}).get(f"h{HORIZONS[0]}"),
                         ew_score=(a["early_warning"] or {}).get("score"),
                         ew_flag=(a["early_warning"] or {}).get("flag"),
                         latency_ms=a["latency_ms"]))
PRED_LOG = pd.DataFrame(log_rows)
elapsed = time.perf_counter() - t0

display(PRED_LOG.head(8))
print(f"{len(PRED_LOG)} advisories in {elapsed:.1f}s | "
      f"median latency {PRED_LOG.latency_ms.median():.0f} ms | "
      f"p95 {PRED_LOG.latency_ms.quantile(.95):.0f} ms")
print(f"alert rate in this batch: {PRED_LOG.ew_flag.mean():.1%} "
      f"(budget {ALERT_BUDGET_PCT:.0f}%)")

# walk-forward parity: serve at a cut point, compare with the actual reading h steps later
h_chk = HORIZONS[0]
served = []
for pid in batch_ids[:12]:
    g = panel[panel.patient_id == pid].sort_values("ts").reset_index(drop=True)
    if len(g) < 60:
        continue
    for cut in range(int(len(g) * 0.8), len(g) - h_chk, 3):
        a = RELOADED.predict(g.iloc[:cut])
        p = a["forecast"].get("sbp", {}).get(f"h{h_chk}")
        if p is not None:
            served.append(dict(pred=p, actual=float(g.sbp.iloc[cut + h_chk - 1])))
SERVED = pd.DataFrame(served).dropna()
if len(SERVED) > 20:
    serve_mae = mean_absolute_error(SERVED.actual, SERVED.pred)
    offline_mae = float(BOARD[(BOARD.split == "test") & (BOARD.model == SERVABLE)
                              & (BOARD.horizon == h_chk)].MAE.iloc[0]) \
        if len(BOARD[(BOARD.split == "test") & (BOARD.model == SERVABLE)]) else np.nan
    print(f"\ntrain/serve parity on {len(SERVED)} walk-forward points:")
    print(f"  serving-path MAE {serve_mae:.2f} mmHg vs offline test MAE {offline_mae:.2f} mmHg")
    print(f"  gap {abs(serve_mae - offline_mae):.2f} mmHg "
          f"{'- consistent' if abs(serve_mae - offline_mae) < 2 else '- INVESTIGATE'}")

# 14. Smmary

In [ ]:
SUMMARY = pd.DataFrame([
    dict(model="1. Forecaster", regime="self-supervised",
         chosen=SERVABLE,
         candidates=len(FORECAST_MODELS),
         metric="MAE (mmHg)",
         value=round(float(BOARD[(BOARD.split == "test") & (BOARD.model == SERVABLE)].MAE.mean()), 3),
         reference=f"{best_base} = "
                   f"{float(BOARD[(BOARD.split == 'test') & (BOARD.model == best_base)].MAE.mean()):.3f}",
         decision=DECISION),
    dict(model="2. Personalisation offset", regime="supervised quantile",
         chosen=OFFSET_CHOICE, candidates=len(OFFSET_CANDIDATES) + 4,
         metric=f"pinball q{OFFSET_TARGET_Q}",
         value=float(best_learned.holdout_pinball) if OFFSET_KIND == "learned"
         else float(best_ref.holdout_pinball),
         reference=f"{best_ref.model} = {best_ref.holdout_pinball}",
         decision="promoted" if OFFSET_KIND == "learned" else "reference kept"),
    dict(model="3. Early-warning detector", regime="unsupervised",
         chosen=DETECTOR_CHOICE, candidates=len(DETECTORS),
         metric=f"precision lift @ {ALERT_BUDGET_PCT:.0f}% budget",
         value=float(chosen_row.lift),
         reference=f"rule-engine analogue = "
                   f"{float(ref_row.iloc[0].lift) if len(ref_row) else float('nan')}",
         decision=f"median lead {chosen_row.lead_days_median:.1f} days"),
])
display(SUMMARY)

print(f"""
Cohort        : {panel.series_id.nunique()} patients, {len(panel):,} sessions
Features      : {len(FEATURES)} causal features, {len(HORIZONS)} horizons, {len(SIGNALS)} signals
Artifact      : {BUNDLE_PATH}
Safety        : emergency floor {EMERGENCY_FLOOR_MMHG:.0f} mmHg asserted at every layer;
                offset capped to [-{OFFSET_CAP_TIGHTEN:.0f}, +{OFFSET_CAP_LOOSEN:.0f}] mmHg;
                governance parameters excluded from every search space.
""")